In [3]:
# %%

# ===========================================================
# CELL 1: Install all dependencies
# ===========================================================

# Run this ONCE in your terminal or here.
# If already installed, this cell will skip quickly.

# librosa: Industry-standard audio analysis library
# soundfile: Efficient audio file I/O
# pydub: Audio format conversion (mp3→wav)
# tensorflow: Deep learning framework
# scikit-learn: Classical ML metrics and utilities
# seaborn: Statistical data visualization
# ===========================================================
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

packages = ["librosa", "soundfile", "pandas", "scikit-learn",
            "seaborn", "pydub", "tqdm", "matplotlib"]

for pkg in packages:
    try:
        __import__(pkg)  # Changed from import(pkg) to __import__(pkg)
        print(f"✓ {pkg} already installed")
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

print("✓ All dependencies available")

✓ librosa already installed
✓ soundfile already installed
✓ pandas already installed
Installing scikit-learn...
✓ seaborn already installed
✓ pydub already installed
✓ tqdm already installed
✓ matplotlib already installed
✓ All dependencies available


C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [ ]:
# %%

# ===========================================================

# CELL 2: All imports — organized by category

# ===========================================================

# --- Standard Library ---

import os

import sys

import gc

import json

import time

import random

import hashlib

import warnings

import logging

from pathlib import Path

from datetime import datetime

from collections import defaultdict, OrderedDict

from typing import Dict, List, Tuple, Optional, Any

# --- Numerical / Data ---

import numpy as np

import pandas as pd

# --- Audio Processing ---

import librosa

import librosa.display

import soundfile as sf

from scipy.signal import butter, lfilter

from pydub import AudioSegment

# --- Visualization ---

import matplotlib

import matplotlib.pyplot as plt

import matplotlib.gridspec as gridspec

from matplotlib.patches import FancyBboxPatch

import seaborn as sns

# --- Progress Bars ---

from tqdm import tqdm

# --- Deep Learning ---

import tensorflow as tf

from tensorflow.keras.layers import (

    Input, Conv2D, MaxPooling2D, BatchNormalization,

    Dropout, Dense, Reshape, Bidirectional, GRU,

    GlobalAveragePooling1D, GlobalAveragePooling2D,

    Multiply, Add, Concatenate, LayerNormalization,

    SeparableConv2D, Activation, Permute, Flatten

)

from tensorflow.keras.models import Model

from tensorflow.keras.callbacks import (

    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint,

    CSVLogger

)

from tensorflow.keras.regularizers import l2

# --- Evaluation Metrics ---

from sklearn.metrics import (

    f1_score, precision_score, recall_score,

    classification_report, confusion_matrix,

    average_precision_score, hamming_loss,

    roc_curve, auc, precision_recall_curve,

    multilabel_confusion_matrix

)

# --- Suppress noisy warnings ---

warnings.filterwarnings("ignore")

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# --- Logging setup ---

logging.basicConfig(

    level=logging.INFO,

    format='%(asctime)s | %(levelname)s | %(message)s',

    datefmt='%H:%M:%S'

)

logger = logging.getLogger("DefenceAudio")

# --- Print system info ---

print("=" * 60)

print("DEFENCE SOUND CLASSIFICATION SYSTEM")

print("=" * 60)

print(f"Python:     {sys.version.split()[0]}")

print(f"TensorFlow: {tf.__version__}")

print(f"NumPy:      {np.__version__}")

print(f"Pandas:     {pd.__version__}")

print(f"Librosa:    {librosa.__version__}")

print(f"GPU:        {tf.config.list_physical_devices('GPU')}")

print(f"Timestamp:  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("=" * 60)

In [ ]:
# %%

# ===========================================================

# CELL 3: Detect project root automatically

# ===========================================================

# This cell finds the project root directory automatically.

#

# It works whether you run the notebook from:

#   - defence_sound_classification/notebooks/

#   - defence_sound_classification/

#   - Any parent directory

#

# It looks for the "data/raw" folder to confirm the root.

# ===========================================================

def find_project_root(marker_folder="data"):

    """

    Walk upward from notebook location to find project root.

    

    The project root is identified by having a 'data' folder.

    This makes the code portable — works on any machine

    without hardcoded paths.

    """

    # Start from notebook's directory

    current = Path(os.getcwd()).resolve()

    

    # Check current and parent directories

    for _ in range(5):  # max 5 levels up

        if (current / marker_folder).exists():

            return current

        # Also check if we ARE inside notebooks/

        if current.name == "notebooks" and (current.parent / marker_folder).exists():

            return current.parent

        current = current.parent

    

    # Fallback: use current working directory

    logger.warning("Could not auto-detect project root. Using current directory.")

    return Path(os.getcwd()).resolve()

PROJECT_ROOT = find_project_root()

print(f"Project root detected: {PROJECT_ROOT}")

# Verify raw data exists

raw_data_path = PROJECT_ROOT / "data" / "raw"

if raw_data_path.exists():

    print(f"✓ Raw data found at: {raw_data_path}")

    for item in sorted(raw_data_path.iterdir()):

        if item.is_dir():

            count = len(list(item.rglob("*.*")))

            print(f"  {item.name}: {count} files")

else:

    print(f"⚠ Raw data NOT found at: {raw_data_path}")

    print(f"  Please place your dataset folders (drone, gunshot, jetplane, vehicle)")

    print(f"  inside: {raw_data_path}")

In [ ]:


# %%

# ===========================================================

# CELL 4: Master configuration

# ===========================================================

# ALL settings in ONE place. Paths are relative to PROJECT_ROOT.

# This makes the code portable across different machines.

#

# DESIGN PHILOSOPHY:

# - Every magic number has a name and comment

# - Paths are relative and auto-detected

# - Difficulty parameters are tunable

# - Everything is reproducible via SEED

# ===========================================================

class Config:

    """

    Master configuration for the entire pipeline.

    

    All paths are relative to PROJECT_ROOT.

    Change hyperparameters here — all other code reads from this class.

    """

    

    # ─── Reproducibility ───

    SEED = 42

    

    # ─── Sound Classes ───

    # These are the 4 threat categories our system detects

    CLASSES = ["drone", "gunshot", "jetplane", "vehicle"]

    NUM_CLASSES = len(CLASSES)

    

    # Class descriptions for reports

    CLASS_DESCRIPTIONS = {

        "drone":    "Unmanned Aerial Vehicle (UAV) — rotor/buzzing sounds",

        "gunshot":  "Firearm discharge — impulsive broadband sound",

        "jetplane": "Jet aircraft engine — sustained high-energy sound",

        "vehicle":  "Ground vehicle engine — cars, trucks, military vehicles"

    }

    

    # Consistent colors for all visualizations

    CLASS_COLORS = {

        "drone":    "#2196F3",   # Blue

        "gunshot":  "#F44336",   # Red

        "jetplane": "#4CAF50",   # Green

        "vehicle":  "#FF9800"    # Orange

    }

    

    # ─── Audio Parameters ───

    SR = 22050              # Sample rate (Hz). 22050 captures up to ~11kHz

    DURATION = 6            # Clip length in seconds

    SAMPLES = SR * DURATION # Total samples per clip = 132300

    

    # ─── Feature Extraction ───

    N_MELS = 128            # Mel frequency bands

    N_FFT = 2048            # FFT window size (~93ms at 22050Hz)

    HOP_LENGTH = 512        # Hop between windows (~23ms)

    # Result: ~259 time frames per 6-second clip

    

    # ─── Training ───

    BATCH_SIZE = 8         # Reduce to 8 if you get OOM errors

    EPOCHS = 25             # Max epochs (early stopping will cut shorter)

    LEARNING_RATE = 5e-4    # Initial LR for Adam optimizer

    THRESHOLD = 0.5         # Default sigmoid threshold

    

    # ─── Regularization (prevents overfitting) ───

    LABEL_SMOOTHING = 0.1   # Softens labels: 1→0.95, 0→0.05

    MIXUP_ALPHA = 0.3       # Blends training samples

    L2_REG = 1e-4           # Weight decay in conv/dense layers

    

    # ─── Difficulty (makes training realistic) ───

    MIN_GAIN = 0.08         # Very quiet signal (distant source)

    MAX_GAIN = 0.7          # Not full volume

    MIN_SNR = -5            # Noise louder than signal!

    MAX_SNR = 10            # Moderate noise

    MIN_EVENT_SEC = 0.3     # Shortest event (300ms)

    MAX_EVENT_SEC = 3.0     # Longest event (3s)

    MIN_CLASSES_MIX = 1     # Single class + noise

    MAX_CLASSES_MIX = 4     # All 4 classes overlapping

    

    # ─── Dataset Generation ───

    # Reduce these if generation takes too long on your machine

    NUM_MIXES_TRAIN = 3000

    NUM_MIXES_VAL = 600

    NUM_MIXES_TEST = 600

    HARD_RATIO_TRAIN = 0.4  # 40% hard samples in training

    HARD_RATIO_TEST = 0.5   # 50% hard samples in testing

    NUM_STRESS_TEST = 300

    

    # ─── Paths (ALL RELATIVE TO PROJECT_ROOT) ───

    # These are set in init after PROJECT_ROOT is known

    RAW_DATASET = ""

    PROCESSED_DATASET = ""

    NOISE_DIR = ""

    GENERATED_ROOT = ""

    STRESS_TEST_DIR = ""

    SPLITS_DIR = ""

    SPLIT_CSV = ""

    MODELS_DIR = ""

    BEST_MODEL_DIR = ""

    REPORTS_DIR = ""

    PLOTS_DIR = ""

    

    # ─── Validation Thresholds ───

    # Used to detect suspicious/vague metric values

    VALID_LOSS_RANGE = (0.001, 5.0)

    VALID_ACC_RANGE = (0.1, 1.0)

    VALID_F1_RANGE = (0.05, 1.0)

    VALID_AUC_RANGE = (0.4, 1.0)

    SUSPICIOUS_ACC_THRESHOLD = 0.98  # If above this → flag as suspicious

    

    # ─── Confidence Levels for Production Inference ───

    CONFIDENCE_LEVELS = {

        "CRITICAL":   0.90,  # Very high confidence → immediate alert

        "HIGH":       0.75,  # High confidence → alert + verify

        "MEDIUM":     0.50,  # Moderate → flag for review

        "LOW":        0.30,  # Low → background monitoring

        "NEGLIGIBLE": 0.0    # Below threshold → ignore

    }

    

    @classmethod

    def initialize(cls, project_root: Path):

        """

        Set all paths relative to the detected project root.

        This makes the code portable across machines.

        """

        root = project_root

        

        cls.RAW_DATASET       = str(root / "data" / "raw")

        cls.PROCESSED_DATASET = str(root / "data" / "processed")

        cls.NOISE_DIR         = str(root / "data" / "noise")

        cls.GENERATED_ROOT    = str(root / "data" / "generated")

        cls.STRESS_TEST_DIR   = str(root / "data" / "stress_test")

        cls.SPLITS_DIR        = str(root / "data" / "splits")

        cls.SPLIT_CSV         = str(root / "data" / "splits" / "split.csv")

        cls.MODELS_DIR        = str(root / "models")

        cls.BEST_MODEL_DIR    = str(root / "models" / "best_model")

        cls.REPORTS_DIR       = str(root / "reports")

        cls.PLOTS_DIR         = str(root / "reports" / "plots")

        

        logger.info(f"Paths initialized relative to: {root}")

    

    @classmethod

    def create_directories(cls):

        """Create all project directories if they don't exist."""

        dirs = [

            cls.PROCESSED_DATASET,

            cls.NOISE_DIR,

            cls.GENERATED_ROOT,

            cls.STRESS_TEST_DIR,

            cls.SPLITS_DIR,

            cls.MODELS_DIR,

            cls.BEST_MODEL_DIR,

            cls.REPORTS_DIR,

            f"{cls.PLOTS_DIR}/data_analysis",

            f"{cls.PLOTS_DIR}/training_curves",

            f"{cls.PLOTS_DIR}/evaluation",

            f"{cls.PLOTS_DIR}/model_comparison",

            f"{cls.PLOTS_DIR}/production_dashboard",

        ]

        

        for cls_name in cls.CLASSES:

            dirs.append(f"{cls.PROCESSED_DATASET}/{cls_name}")

        

        for split in ["train", "val", "test"]:

            dirs.append(f"{cls.GENERATED_ROOT}/{split}/audio")

        

        for d in dirs:

            os.makedirs(d, exist_ok=True)

        

        logger.info(f"Created {len(dirs)} directories")

    

    @classmethod

    def save_config(cls, project_root: Path):

        """Save configuration to JSON for reproducibility."""

        config_dict = {}

        for k, v in vars(cls).items():

            if not k.startswith('_') and not callable(v):

                try:

                    json.dumps(v)

                    config_dict[k] = v

                except (TypeError, ValueError):

                    config_dict[k] = str(v)

        

        config_path = project_root / "config.json"

        with open(config_path, 'w') as f:

            json.dump(config_dict, f, indent=2)

        logger.info(f"Config saved to {config_path}")

    

    @classmethod

    def set_seeds(cls):

        """Set all random seeds for full reproducibility."""

        random.seed(cls.SEED)

        np.random.seed(cls.SEED)

        tf.random.set_seed(cls.SEED)

        os.environ['PYTHONHASHSEED'] = str(cls.SEED)

        logger.info(f"All seeds set to {cls.SEED}")

# ─── Initialize everything ───

cfg = Config()

cfg.initialize(PROJECT_ROOT)

cfg.set_seeds()

cfg.create_directories()

cfg.save_config(PROJECT_ROOT)

print("\n✓ Configuration loaded")

print(f"  Project root:  {PROJECT_ROOT}")

print(f"  Raw data:      {cfg.RAW_DATASET}")

print(f"  Processed:     {cfg.PROCESSED_DATASET}")

print(f"  Models:        {cfg.MODELS_DIR}")

print(f"  Reports:       {cfg.REPORTS_DIR}")

print(f"  Classes:       {cfg.CLASSES}")

print(f"  Audio:         {cfg.SR}Hz, {cfg.DURATION}s")

print(f"  Difficulty:    gain=[{cfg.MIN_GAIN},{cfg.MAX_GAIN}], SNR=[{cfg.MIN_SNR},{cfg.MAX_SNR}]dB")

In [ ]:
# %%

# ===========================================================

# CELL 5: Metric Validator

# ===========================================================

# PRODUCTION REQUIREMENT:

# In defence systems, we CANNOT trust metrics blindly.

# - If accuracy is 0.99 → might be data leakage

# - If loss is NaN → training diverged

# - If precision=1.0 and recall=0.01 → model predicts nothing

#

# This validator checks EVERY metric for sanity and flags

# any suspicious values before we deploy anything.

# ===========================================================
class MetricValidator:

    """

    Validates training and evaluation metrics for sanity.

    

    WHY THIS EXISTS:

    In defence applications, deploying a model with misleading

    metrics could have catastrophic consequences. A model that

    appears 99% accurate but has data leakage provides FALSE

    confidence to operators.

    

    WHAT IT CHECKS:

    1. Metrics within valid numerical ranges

    2. Suspicious patterns (too good to be true)

    3. Train/val gap for overfitting detection

    4. Degenerate predictions (all same class)

    5. Precision-recall balance

    6. Generates human-readable validation reports

    """

    

    def init(self, config):

        self.cfg = config

        self.warnings = []

        self.errors = []

    

    def reset(self):

        """Clear all accumulated messages for fresh validation."""

        self.warnings = []

        self.errors = []

    

    def validate_single_metric(self, name: str, value: float,

                                valid_range: Tuple[float, float]) -> bool:

        """

        Check if a single metric value is within expected range.

        

        WHY: NaN, Inf, or out-of-range values indicate

        something went wrong during training/evaluation.

        

        Returns True if valid, False if problematic.

        """

        if value is None or np.isnan(value) or np.isinf(value):

            self.errors.append(f"INVALID: {name} = {value} (NaN or Inf)")

            return False

        

        low, high = valid_range

        if value < low or value > high:

            self.warnings.append(

                f"OUT OF RANGE: {name} = {value:.4f} "

                f"(expected [{low}, {high}])"

            )

            return False

        return True

    

    def validate_training_metrics(self, history: dict,

                                   model_name: str) -> dict:

        """

        Validate all metrics from a completed training run.

        

        CHECKS:

        1. Loss is actually decreasing (model is learning)

        2. No NaN/Inf values anywhere

        3. Val accuracy not suspiciously high (data leakage?)

        4. Train-val gap not too large (overfitting)

        5. Precision and recall are balanced (not degenerate)

        6. AUC is reasonable

        

        Returns a report dict with status and all issues found.

        """

        self.reset()

        report = {"model": model_name, "status": "PASS", "issues": []}

        

        # ── Check for NaN/Inf in loss ──

        train_loss = history.get('loss', [])

        if any(np.isnan(train_loss)) or any(np.isinf(train_loss)):

            self.errors.append("Training loss contains NaN/Inf — training DIVERGED")

            report["status"] = "FAIL"

        

        # ── Check loss is actually decreasing ──

        if len(train_loss) > 5:

            first_half = np.mean(train_loss[:len(train_loss)//2])

            second_half = np.mean(train_loss[len(train_loss)//2:])

            if second_half >= first_half * 0.95:

                self.warnings.append(

                    f"Loss NOT decreasing meaningfully: "

                    f"first_half={first_half:.4f}, second_half={second_half:.4f}"

                )

        

        # ── Check val accuracy not suspiciously high ──

        val_acc = history.get('val_bin_acc', [])

        if val_acc and max(val_acc) > self.cfg.SUSPICIOUS_ACC_THRESHOLD:

            self.warnings.append(

                f"SUSPICIOUS: Val accuracy = {max(val_acc):.4f} "

                f"(>{self.cfg.SUSPICIOUS_ACC_THRESHOLD}). "

                f"Possible data leakage or too-easy test set!"

            )

        

        # ── Check train-val gap (overfitting detector) ──

        val_loss = history.get('val_loss', [])

        if train_loss and val_loss:

            final_gap = abs(train_loss[-1] - val_loss[-1])

            if final_gap > 0.5:

                self.warnings.append(

                    f"OVERFITTING: Train-val loss gap = {final_gap:.4f} "

                    f"(train={train_loss[-1]:.4f}, val={val_loss[-1]:.4f})"

                )

        

        # ── Check precision-recall balance ──

        val_prec = history.get('val_precision', [])

        val_rec = history.get('val_recall', [])

        if val_prec and val_rec:

            fp = val_prec[-1]

            fr = val_rec[-1]

            

            # High precision + low recall = predicting mostly "absent"

            if fp > 0.95 and fr < 0.3:

                self.warnings.append(

                    f"DEGENERATE: precision={fp:.3f} but recall={fr:.3f}. "

                    f"Model predicts 'absent' for almost everything."

                )

            

            # High recall + low precision = predicting mostly "present"

            if fr > 0.95 and fp < 0.3:

                self.warnings.append(

                    f"DEGENERATE: recall={fr:.3f} but precision={fp:.3f}. "

                    f"Model predicts 'present' for everything."

                )

        

        # ── Check AUC ──

        val_auc = history.get('val_auc', [])

        if val_auc:

            self.validate_single_metric('val_auc', max(val_auc),

                                         self.cfg.VALID_AUC_RANGE)

        

        # ── Compile final report ──

        report["issues"] = self.warnings + self.errors

        if self.errors:

            report["status"] = "FAIL"

        elif self.warnings:

            report["status"] = "WARNING"

        

        report["num_epochs"] = len(train_loss)

        report["final_train_loss"] = train_loss[-1] if train_loss else None

        report["final_val_loss"] = val_loss[-1] if val_loss else None

        report["best_val_auc"] = max(val_auc) if val_auc else None

        

        return report

    

    def validate_test_metrics(self, y_true: np.ndarray,

                               y_pred: np.ndarray,

                               y_probs: np.ndarray,

                               model_name: str) -> dict:

        """

        Validate test set evaluation metrics.

        

        CHECKS:

        1. Predictions aren't all zeros or all ones (degenerate)

        2. Probability distribution has spread (model is discriminating)

        3. No class is completely ignored

        4. No class is always predicted

        5. Overall F1 is reasonable

        6. Suspiciously high metrics flagged

        """

        self.reset()

        report = {"model": model_name, "status": "PASS", "issues": []}

        

        # ── Check for degenerate all-same predictions ──

        pred_mean = y_pred.mean()

        if pred_mean < 0.01:

            self.errors.append(

                f"DEGENERATE: Model predicts almost all ABSENT "

                f"(mean prediction rate = {pred_mean:.4f})"

            )

        if pred_mean > 0.99:

            self.errors.append(

                f"DEGENERATE: Model predicts almost all PRESENT "

                f"(mean prediction rate = {pred_mean:.4f})"

            )

        

        # ── Check probability distribution spread ──

        prob_std = y_probs.std()

        if prob_std < 0.05:

            self.warnings.append(

                f"LOW SPREAD: Probabilities have std={prob_std:.4f}. "

                f"Model may not be discriminating between classes."

            )

        

        # ── Per-class checks ──

        for i, cls in enumerate(self.cfg.CLASSES):

            cls_pred_rate = y_pred[:, i].mean()

            cls_true_rate = y_true[:, i].mean()

            

            # Class being completely ignored

            if cls_pred_rate < 0.01 and cls_true_rate > 0.1:

                self.warnings.append(

                    f"CLASS IGNORED: {cls} — pred_rate={cls_pred_rate:.3f} "

                    f"but true_rate={cls_true_rate:.3f}"

                )

            

            # Class being over-predicted

            if cls_pred_rate > 0.99:

                self.warnings.append(

                    f"CLASS OVER-PREDICTED: {cls} — pred_rate={cls_pred_rate:.3f}"

                )

        

        # ── Overall F1 check ──

        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

        self.validate_single_metric('macro_f1', macro_f1, self.cfg.VALID_F1_RANGE)

        

        if macro_f1 > 0.95:

            self.warnings.append(

                f"SUSPICIOUS F1: macro_f1 = {macro_f1:.4f}. "

                f"Check for data leakage or test set overlap."

            )

        

        # ── Compile report ──

        report["issues"] = self.warnings + self.errors

        if self.errors:

            report["status"] = "FAIL"

        elif self.warnings:

            report["status"] = "WARNING"

        

        report["macro_f1"] = macro_f1

        report["pred_mean"] = float(pred_mean)

        report["prob_std"] = float(prob_std)

        

        return report

    

    def print_report(self, report: dict):

        """Pretty-print a validation report with status emoji."""

        emoji = {"PASS": "✅", "WARNING": "⚠️", "FAIL": "❌"}.get(report["status"], "❓")

        

        print(f"\n{emoji} Validation: {report['model']} → {report['status']}")

        if report["issues"]:

            for issue in report["issues"]:

                print(f"   ⚡ {issue}")

        else:

            print(f"   All checks passed.")

        

        if "final_val_loss" in report and report["final_val_loss"] is not None:

            print(f"   Final val_loss: {report['final_val_loss']:.4f}")

        if "best_val_auc" in report and report["best_val_auc"] is not None:

            print(f"   Best val_auc: {report['best_val_auc']:.4f}")

# Initialize validator

validator = MetricValidator(cfg)

print("✓ Metric validator initialized")

In [ ]:
# %%

# ===========================================================

# CELL 6: Production Confidence Level System

# ===========================================================

# In a real defence system, a binary 0/1 prediction is NOT enough.

# Operators need to know HOW CONFIDENT the system is.

#

# CRITICAL (>0.90): Deploy countermeasures immediately

# HIGH (0.75-0.90): Alert + verify with second sensor

# MEDIUM (0.50-0.75): Flag for human review

# LOW (0.30-0.50): Background monitoring only

# NEGLIGIBLE (<0.30): Likely environmental noise

#

# This maps directly to military alert levels.

# ===========================================================


class ConfidenceAnalyzer:

    """

    Assigns military-grade confidence levels to predictions.

    

    WHY: In defence, operators need actionable intelligence,

    not just probabilities. A drone at 95% confidence triggers

    different response than drone at 35% confidence.

    

    Also performs CALIBRATION ANALYSIS:

    When model says 80% confident, is it actually correct

    80% of the time? Poor calibration = unreliable system.

    """

    

    def init(self, config):

        self.cfg = config

        self.levels = config.CONFIDENCE_LEVELS

    

    def get_confidence_level(self, probability: float) -> str:

        """Map probability → confidence level string."""

        for level_name, threshold in sorted(

            self.levels.items(), key=lambda x: x[1], reverse=True

        ):

            if probability >= threshold:

                return level_name

        return "NEGLIGIBLE"

    

    def get_confidence_color(self, level: str) -> str:

        """Color coding for dashboard display."""

        colors = {

            "CRITICAL":   "#D32F2F",  # Dark red

            "HIGH":       "#F44336",  # Red

            "MEDIUM":     "#FF9800",  # Orange

            "LOW":        "#FFC107",  # Yellow

            "NEGLIGIBLE": "#4CAF50"   # Green (safe)

        }

        return colors.get(level, "#9E9E9E")

    

    def analyze_prediction(self, probs: np.ndarray,

                           classes: List[str],

                           per_class_thresholds: dict = None) -> dict:

        """

        Full analysis of a single prediction.

        

        Returns dict with:

        - Per-class probability, threshold, detection status

        - Confidence level for each class

        - Recommended action

        - Overall threat level (highest among detected classes)

        - Timestamp for logging

        """

        analysis = {

            "timestamp": datetime.now().isoformat(),

            "overall_threat_level": "NEGLIGIBLE",

            "detections": [],

            "summary": ""

        }

        

        # Priority order for overall threat level

        level_priority = {

            "NEGLIGIBLE": 0, "LOW": 1, "MEDIUM": 2,

            "HIGH": 3, "CRITICAL": 4

        }

        

        # Recommended actions per level

        actions = {

            "CRITICAL":   "⚠ IMMEDIATE: Deploy countermeasures NOW",

            "HIGH":       "🔴 ALERT: Verify with secondary sensor",

            "MEDIUM":     "🟠 FLAG: Assign operator for review",

            "LOW":        "🟡 WATCH: Continue background monitoring",

            "NEGLIGIBLE": "🟢 CLEAR: No action required"

        }

        

        max_priority = 0

        

        for i, cls in enumerate(classes):

            prob = float(probs[i])

            threshold = (per_class_thresholds[cls]["threshold"]

                        if per_class_thresholds else self.cfg.THRESHOLD)

            

            level = self.get_confidence_level(prob)

            detected = prob >= threshold

            

            detection = {

                "class": cls,

                "probability": round(prob, 4),

                "threshold": threshold,

                "detected": detected,

                "confidence_level": level,

                "color": self.get_confidence_color(level),

                "action": actions[level]

            }

            analysis["detections"].append(detection)

            

            if detected:

                priority = level_priority.get(level, 0)

                if priority > max_priority:

                    max_priority = priority

                    analysis["overall_threat_level"] = level

        

        # Summary

        detected_classes = [d["class"] for d in analysis["detections"] if d["detected"]]

        if detected_classes:

            analysis["summary"] = (

                f"THREATS DETECTED: {', '.join(detected_classes)} | "

                f"Level: {analysis['overall_threat_level']}"

            )

        else:

            analysis["summary"] = "ALL CLEAR — No threats detected"

        

        return analysis

    

    def calibration_analysis(self, y_true: np.ndarray,

                              y_probs: np.ndarray,

                              n_bins: int = 10) -> dict:

        """

        Check if model confidence is well-calibrated.

        

        WHAT: When model says 80% confident, is it correct 80% of time?

        

        WHY: A miscalibrated model that says "90% confident" but is

        only right 50% of the time gives false sense of security.

        

        METRIC: Expected Calibration Error (ECE)

        - ECE < 0.05: Well calibrated

        - ECE 0.05-0.15: Moderate calibration

        - ECE > 0.15: Poorly calibrated → needs recalibration

        """

        calibration = {}

        

        for i, cls in enumerate(self.cfg.CLASSES):

            bin_edges = np.linspace(0, 1, n_bins + 1)

            bin_accs = []

            bin_confs = []

            bin_counts = []

            

            for j in range(n_bins):

                mask = ((y_probs[:, i] >= bin_edges[j]) &

                        (y_probs[:, i] < bin_edges[j+1]))

                if mask.sum() > 0:

                    bin_accs.append(y_true[mask, i].mean())

                    bin_confs.append(y_probs[mask, i].mean())

                    bin_counts.append(int(mask.sum()))

            

            # Expected Calibration Error

            ece = 0.0

            total = sum(bin_counts) if bin_counts else 1

            for acc, conf, count in zip(bin_accs, bin_confs, bin_counts):

                ece += (count / total) * abs(acc - conf)

            

            calibration[cls] = {

                "bin_accuracies": bin_accs,

                "bin_confidences": bin_confs,

                "bin_counts": bin_counts,

                "ece": round(ece, 4)

            }

        

        return calibration

# Initialize

confidence_analyzer = ConfidenceAnalyzer(cfg)

print("✓ Confidence analyzer initialized")

print(f"  Levels: {list(cfg.CONFIDENCE_LEVELS.keys())}")

In [ ]:
# %%

# ===========================================================

# CELL 7: Audio processing toolkit

# ===========================================================

# Complete audio processing for:

# - Loading, normalizing, padding, cropping

# - 14+ augmentation/manipulation functions

# - Mixed audio generation

# - Mel spectrogram feature extraction

#

# Every function has a docstring explaining WHY it exists

# specifically for defence sound classification.

# ===========================================================

class AudioProcessor:

    """

    Production audio processing toolkit.

    

    Encapsulates all audio manipulation functions for:

    1. Data preprocessing (convert, normalize, pad/crop)

    2. Data augmentation (noise, shift, stretch, filter)

    3. Mixed audio generation (overlay multiple classes)

    4. Feature extraction (mel spectrograms)

    

    All functions handle errors gracefully — critical for

    production where corrupt files must not crash the system.

    """

    

    def init(self, config):

        self.cfg = config

    

    # ─── Basic I/O ───

    

    def load_audio(self, path: str) -> Optional[np.ndarray]:

        """

        Safely load audio file as mono float32 waveform.

        Returns None on error (production-safe: never crashes).

        """

        try:

            y, _ = librosa.load(path, sr=self.cfg.SR, mono=True)

            return y.astype(np.float32)

        except Exception as e:

            logger.warning(f"Failed to load {path}: {e}")

            return None

    

    def normalize(self, y: np.ndarray) -> np.ndarray:

        """

        Peak normalize to 0.95.

        WHY: Prevents clipping after mixing multiple signals.

        """

        peak = np.max(np.abs(y))

        if peak > 1e-6:

            y = y / peak * 0.95

        return y

    

    def pad_or_crop(self, y: np.ndarray,

                     target_len: int = None) -> np.ndarray:

        """

        Make audio exactly target_len samples.

        WHY: Neural networks require fixed-size inputs.

        """

        if target_len is None:

            target_len = self.cfg.SAMPLES

        if len(y) == target_len:

            return y

        if len(y) > target_len:

            start = random.randint(0, len(y) - target_len)

            return y[start:start + target_len]

        pad_total = target_len - len(y)

        pad_left = random.randint(0, pad_total)

        return np.pad(y, (pad_left, pad_total - pad_left))

    

    def place_randomly(self, y: np.ndarray,

                        total_len: int = None) -> Tuple[np.ndarray, int]:

        """

        Place audio at random position in zero buffer.

        WHY: Events don't always start at time=0 in real recordings.

        """

        if total_len is None:

            total_len = self.cfg.SAMPLES

        if len(y) > total_len:

            y = self.pad_or_crop(y, total_len)

        out = np.zeros(total_len, dtype=np.float32)

        max_offset = total_len - len(y)

        offset = random.randint(0, max(0, max_offset))

        out[offset:offset + len(y)] += y

        return out, offset

    

    # ─── Signal Manipulation ───

    

    def random_gain(self, y, min_g=None, max_g=None):

        """Random volume. WHY: Simulates distance variation."""

        if min_g is None: min_g = self.cfg.MIN_GAIN

        if max_g is None: max_g = self.cfg.MAX_GAIN

        gain = random.uniform(min_g, max_g)

        return y * gain, gain

    

    def add_gaussian_noise(self, y, min_std=0.002, max_std=0.03):

        """WHY: All microphones produce electronic noise."""

        std = random.uniform(min_std, max_std)

        return y + np.random.randn(len(y)).astype(np.float32) * std

    

    def time_shift(self, y, max_shift=4000):

        """WHY: Prevents memorizing temporal positions."""

        return np.roll(y, random.randint(-max_shift, max_shift))

    

    def pitch_shift(self, y, max_steps=3.0):

        """WHY: Same source type at different RPM/speed."""

        steps = random.uniform(-max_steps, max_steps)

        return librosa.effects.pitch_shift(y, sr=self.cfg.SR, n_steps=steps)

    

    def time_stretch(self, y, min_rate=0.8, max_rate=1.25):

        """WHY: Simulates Doppler and playback speed variation."""

        rate = random.uniform(min_rate, max_rate)

        stretched = librosa.effects.time_stretch(y, rate=rate)

        return self.pad_or_crop(stretched)

    

    # ─── Channel Effects ───

    

    def lowpass_filter(self, y, cutoff=4000):

        """WHY: Simulates sound through walls/barriers."""

        nyq = 0.5 * self.cfg.SR

        b, a = butter(5, min(cutoff / nyq, 0.99), btype='low')

        return lfilter(b, a, y).astype(np.float32)

    

    def highpass_filter(self, y, cutoff=300):

        """WHY: Simulates wind noise removal or small speaker."""

        nyq = 0.5 * self.cfg.SR

        b, a = butter(3, max(cutoff / nyq, 0.01), btype='high')

        return lfilter(b, a, y).astype(np.float32)

    

    def bandpass_filter(self, y, low=300, high=4000):

        """WHY: Military radio has limited bandwidth."""

        nyq = 0.5 * self.cfg.SR

        b, a = butter(3, [max(low/nyq, 0.01), min(high/nyq, 0.99)], btype='band')

        return lfilter(b, a, y).astype(np.float32)

    

    def add_reverb(self, y):

        """WHY: Urban/canyon environments create echoes."""

        decay = np.exp(-np.linspace(0, 5, int(self.cfg.SR * random.uniform(0.1, 0.5))))

        impulse = np.random.randn(len(decay)).astype(np.float32) * decay

        impulse /= (np.max(np.abs(impulse)) + 1e-8)

        wet = np.convolve(y, impulse, mode='same').astype(np.float32)

        ratio = random.uniform(0.1, 0.5)

        return ((1 - ratio)  y + ratio  wet).astype(np.float32)

    

    def add_clipping(self, y, clip_val=None):

        """WHY: Loud events overdrive field microphones."""

        if clip_val is None:

            clip_val = random.uniform(0.2, 0.6)

        return np.clip(y, -clip_val, clip_val)

    

    def reduce_bitdepth(self, y, bits=8):

        """WHY: Field equipment may use low-resolution ADC."""

        levels = 2 ** bits

        return (np.round(y * levels) / levels).astype(np.float32)

    

    # ─── Noise Mixing ───

    

    def add_noise_at_snr(self, signal, noise, snr_db):

        """

        Mix noise at specific SNR.

        WHY: SNR is the main difficulty parameter.

        Negative SNR = noise louder than signal (realistic for distant threats).

        """

        sig_power = np.mean(signal ** 2) + 1e-10

        noise_power = np.mean(noise ** 2) + 1e-10

        scale = np.sqrt(sig_power / (10 ** (snr_db / 10) * noise_power))

        return signal + noise * scale

    

    def fit_noise_to_length(self, noise, target_len):

        """Tile or crop noise to match target length."""

        if len(noise) >= target_len:

            start = random.randint(0, len(noise) - target_len)

            return noise[start:start + target_len]

        repeats = int(np.ceil(target_len / len(noise)))

        return np.tile(noise, repeats)[:target_len]

    

    # ─── Augmentation Pipeline ───

    

    def augment_aggressive(self, y):

        """

        Apply AGGRESSIVE augmentation chain.

        

        WHY THE MODEL WAS GETTING 0.97 ACCURACY:

        Training data was too clean. This augmentation:

        - Forces robust feature learning

        - Prevents memorization

        - Simulates real-world degradation

        """

        y = self.add_gaussian_noise(y, 0.001, 0.015)

        y,  = self.randomgain(y, 0.4, 1.3)

        

        if random.random() < 0.6: y = self.time_shift(y, int(self.cfg.SR * 1.5))

        if random.random() < 0.4: y = self.pitch_shift(y, 2.5)

        if random.random() < 0.4: y = self.time_stretch(y, 0.85, 1.2)

        if random.random() < 0.35: y = self.lowpass_filter(y, random.randint(2500, 6000))

        if random.random() < 0.25: y = self.highpass_filter(y, random.randint(100, 600))

        if random.random() < 0.2: y = self.bandpass_filter(y, random.randint(200,500), random.randint(3000,6000))

        if random.random() < 0.2: y = self.add_reverb(y)

        if random.random() < 0.15: y = self.add_clipping(y)

        if random.random() < 0.1: y = self.reduce_bitdepth(y, random.choice([6, 8, 10]))

        

        return self.normalize(y)

    

    # ─── Feature Extraction ───

    

    def extract_mel_spectrogram(self, file_path: str,

                                 augment: bool = False) -> np.ndarray:

        """

        Extract log-mel spectrogram from audio file.

        

        WHY log-mel:

        - Mel scale matches human auditory perception

        - Log compression handles huge dynamic range

        - 2D representation works with CNNs

        - Standard in audio ML research

        

        Returns: (N_MELS, time_frames) float32 array

        """

        y = self.load_audio(file_path)

        if y is None:

            return np.zeros((self.cfg.N_MELS, 259), dtype=np.float32)

        

        y = self.pad_or_crop(y)

        if augment:

            y = self.augment_aggressive(y)

        

        mel = librosa.feature.melspectrogram(

            y=y, sr=self.cfg.SR, n_mels=self.cfg.N_MELS,

            n_fft=self.cfg.N_FFT, hop_length=self.cfg.HOP_LENGTH

        )

        return librosa.power_to_db(mel, ref=np.max).astype(np.float32)

# Initialize

audio_proc = AudioProcessor(cfg)

# Determine input shape from actual data

samplefiles = list(Path(cfg.PROCESSED_DATASET).rglob("*.wav"))

if not samplefiles:

    samplefiles = list(Path(cfg.RAW_DATASET).rglob("*.wav")) + \ 
                    list(Path(cfg.RAW_DATASET).rglob("*.mp3"))

if samplefiles:

    samplemel = audio_proc.extract_mel_spectrogram(str(_sample_files[0]))

    EXPECTED_TIME_FRAMES = samplemel.shape[1]

else:

    EXPECTED_TIME_FRAMES = 259

INPUT_SHAPE = (cfg.N_MELS, EXPECTED_TIME_FRAMES, 1)

print(f"✓ Audio processor initialized")

print(f"  Model input shape: {INPUT_SHAPE}")

In [ ]:
# %%

# ===========================================================

# CELL 8: Convert dataset to WAV (skip if already done)

# ===========================================================

# Converts all audio formats (.mp3, .wav, .flac, .ogg, .m4a)

# to standardized WAV format:

# - Mono channel

# - 22050 Hz sample rate

# - 16-bit depth

#

# Skips files that already exist in processed folder.

# ===========================================================

SUPPORTED_EXTS = [".wav", ".mp3", ".flac", ".ogg", ".m4a"]

def convert_dataset_if_needed(raw_root, processed_root):

    """Convert all audio to standardized WAV. Skips existing files."""

    raw_root = Path(raw_root)

    processed_root = Path(processed_root)

    

    if not raw_root.exists():

        logger.warning(f"Raw dataset not found at: {raw_root}")

        logger.info("Please place your audio folders (drone, gunshot, jetplane, vehicle)")

        logger.info(f"inside: {raw_root}")

        return

    

    converted, skipped, errors = 0, 0, 0

    

    for cls_dir in sorted(raw_root.iterdir()):

        if not cls_dir.is_dir():

            continue

        for f in tqdm(list(cls_dir.rglob("*")), desc=f"Converting {cls_dir.name}"):

            if f.suffix.lower() not in SUPPORTED_EXTS:

                continue

            

            out_path = processed_root / f.relative_to(raw_root).with_suffix(".wav")

            if out_path.exists():

                skipped += 1

                continue

            

            try:

                audio = AudioSegment.from_file(str(f))

                audio = audio.set_frame_rate(cfg.SR).set_channels(1)

                out_path.parent.mkdir(parents=True, exist_ok=True)

                audio.export(str(out_path), format="wav")

                converted += 1

            except Exception as e:

                errors += 1

                logger.warning(f"Error: {f.name}: {e}")

    

    logger.info(f"Conversion: {converted} new, {skipped} existing, {errors} errors")

convert_dataset_if_needed(cfg.RAW_DATASET, cfg.PROCESSED_DATASET)

# Verify

print("\nProcessed dataset:")

for cls in cfg.CLASSES:

    cls_path = Path(cfg.PROCESSED_DATASET) / cls

    count = len(list(cls_path.glob("*.wav"))) if cls_path.exists() else 0

    print(f"  {cls}: {count} files")

In [ ]:
# %%

# ===========================================================

# CELL 9: Deduplication and strict splitting

# ===========================================================

def compute_audio_hash(filepath, sr=22050, duration=3):

    """Content-based hash for duplicate detection."""

    try:

        y, _ = librosa.load(filepath, sr=sr, mono=True, duration=duration)

        y_q = np.round(y * 1000).astype(np.int16)

        return hashlib.md5(y_q.tobytes()).hexdigest()

    except:

        return None

def find_and_report_duplicates(dataset_root, classes):

    """Find duplicate audio files and save report CSV."""

    hash_to_files = defaultdict(list)

    all_files = []

    

    for cls in classes:

        cls_dir = Path(dataset_root) / cls

        if cls_dir.exists():

            for f in cls_dir.glob("*.wav"):

                all_files.append((str(f), cls))

    

    logger.info(f"Hashing {len(all_files)} files for duplicate detection...")

    for fp, cls in tqdm(all_files, desc="Deduplication"):

        h = compute_audio_hash(fp)

        if h:

            hash_to_files[h].append((fp, cls))

    

    duplicates = {k: v for k, v in hash_to_files.items() if len(v) > 1}

    total_dupes = sum(len(v) - 1 for v in duplicates.values())

    

    # Save report

    rows = []

    for h, files in duplicates.items():

        for fp, cls in files:

            rows.append({"hash": h, "filepath": fp, "class": cls, "group_size": len(files)})

    

    if rows:

        pd.DataFrame(rows).to_csv(f"{cfg.SPLITS_DIR}/duplicate_report.csv", index=False)

    

    logger.info(f"Found {len(duplicates)} duplicate groups ({total_dupes} extra files)")

    return duplicates

def create_strict_split(dataset_root, output_csv, duplicates):

    """

    Create train/val/test split with ZERO data leakage.

    All duplicates are forced into the same split.

    """

    dataset_root = Path(dataset_root)

    

    # Assign group IDs (all duplicates share same group)

    file_to_group = {}

    group_id = 0

    for h, files in duplicates.items():

        for fp, cls in files:

            file_to_group[fp] = group_id

        group_id += 1

    

    all_rows = []

    for cls in cfg.CLASSES:

        cls_dir = dataset_root / cls

        if not cls_dir.exists():

            continue

        for f in sorted(cls_dir.glob("*.wav")):

            fp = str(f)

            if fp not in file_to_group:

                file_to_group[fp] = group_id

                group_id += 1

            all_rows.append({"filepath": fp, "label": cls, "group": file_to_group[fp]})

    

    df = pd.DataFrame(all_rows)

    

    # Split by GROUP not by file

    groups = list(df["group"].unique())

    random.shuffle(groups)

    n = len(groups)

    n_train = int(n * 0.7)

    n_val = int(n * 0.15)

    

    train_groups = set(groups[:n_train])

    val_groups = set(groups[n_train:n_train + n_val])

    

    df["split"] = df["group"].apply(

        lambda g: "train" if g in train_groups else ("val" if g in val_groups else "test")

    )

    

    df.to_csv(output_csv, index=False)

    

    # VERIFY zero leakage

    for s1, s2 in [("train","val"), ("train","test"), ("val","test")]:

        f1 = set(df[df["split"] == s1]["filepath"])

        f2 = set(df[df["split"] == s2]["filepath"])

        overlap = len(f1 & f2)

        assert overlap == 0, f"DATA LEAKAGE between {s1} and {s2}: {overlap} files!"

    

    logger.info("✓ Zero data leakage confirmed")

    print(df.groupby(["split", "label"]).size())

    return df

# Run deduplication and splitting

duplicates = find_and_report_duplicates(cfg.PROCESSED_DATASET, cfg.CLASSES)

split_df = create_strict_split(cfg.PROCESSED_DATASET, cfg.SPLIT_CSV, duplicates)

In [ ]:
# %%

# ===========================================================

# CELL 10: Generate mixed multi-label dataset

# ===========================================================

def load_noise_files(noise_dir):

    """Load all noise file paths from noise directory."""

    noise_dir = Path(noise_dir)

    if not noise_dir.exists():

        logger.info("No noise directory found. Using synthetic Gaussian noise.")

        return []

    files = list(noise_dir.glob("*.wav"))

    logger.info(f"Found {len(files)} background noise files")

    return files

def build_split_class_dict(split_df, split_name):

    """Get file paths per class for a specific split."""

    sub = split_df[split_df["split"] == split_name]

    return {cls: sub[sub["label"] == cls]["filepath"].tolist() for cls in cfg.CLASSES}

def generate_one_mix(class_files, noise_files, hard_mode=False):

    """

    Generate one mixed multi-label audio clip.

    

    Hard mode increases difficulty via:

    - Weaker signals (lower gain)

    - More noise (lower SNR)

    - Shorter events

    - More channel degradation

    """

    # Choose number of classes (weighted random)

    if hard_mode:

        n_cls = random.choices([1,2,3,4], weights=[0.1,0.3,0.4,0.2])[0]

    else:

        n_cls = random.choices([1,2,3,4], weights=[0.15,0.35,0.35,0.15])[0]

    

    chosen = random.sample(cfg.CLASSES, min(n_cls, cfg.NUM_CLASSES))

    mix = np.zeros(cfg.SAMPLES, dtype=np.float32)

    labels = {cls: 0 for cls in cfg.CLASSES}

    

    for cls in chosen:

        if not class_files.get(cls):

            continue

        y = audio_proc.load_audio(random.choice(class_files[cls]))

        if y is None:

            continue

        

        # Event duration

        min_ev = int((cfg.MIN_EVENT_SEC if hard_mode else 0.5) * cfg.SR)

        max_ev = int(min((cfg.MAX_EVENT_SEC if hard_mode else 4.0) * cfg.SR,

                        len(y), cfg.SAMPLES))

        if max_ev <= min_ev:

            max_ev = min_ev + 1

        

        y = audio_proc.pad_or_crop(y, random.randint(min_ev, max_ev))

        

        # Signal gain

        if hard_mode:

            y,  = audioproc.random_gain(y, 0.05, 0.5)

        else:

            y,  = audioproc.random_gain(y, 0.15, 0.8)

        

        placed,  = audioproc.place_randomly(y)

        mix += placed

        labels[cls] = 1

    

    # ── Mandatory noise ──

    if noise_files:

        nf = audio_proc.load_audio(str(random.choice(noise_files)))

        if nf is not None:

            nf = audio_proc.fit_noise_to_length(nf, cfg.SAMPLES)

            snr = random.uniform(cfg.MIN_SNR, 5) if hard_mode else random.uniform(0, cfg.MAX_SNR)

            mix = audio_proc.add_noise_at_snr(mix, nf, snr)

        else:

            gauss = np.random.randn(cfg.SAMPLES).astype(np.float32)

            snr = random.uniform(-3, 5) if hard_mode else random.uniform(3, 15)

            mix = audio_proc.add_noise_at_snr(mix, gauss, snr)

    else:

        gauss = np.random.randn(cfg.SAMPLES).astype(np.float32)

        snr = random.uniform(-3, 5) if hard_mode else random.uniform(3, 15)

        mix = audio_proc.add_noise_at_snr(mix, gauss, snr)

    

    # ── Channel effects ──

    if hard_mode:

        if random.random() < 0.5: mix = audio_proc.lowpass_filter(mix, random.randint(1500, 4000))

        if random.random() < 0.4: mix = audio_proc.highpass_filter(mix, random.randint(200, 800))

        if random.random() < 0.3: mix = audio_proc.add_reverb(mix)

        if random.random() < 0.25: mix = audio_proc.add_clipping(mix)

        if random.random() < 0.15: mix = audio_proc.reduce_bitdepth(mix, random.choice([6, 8]))

    else:

        if random.random() < 0.3: mix = audio_proc.lowpass_filter(mix, random.randint(3000, 6000))

        if random.random() < 0.2: mix = audio_proc.add_reverb(mix)

    

    return audio_proc.normalize(mix), labels

def generate_dataset_for_split(split_name, class_files, num_mixes,

                                noise_files, hard_ratio):

    """Generate all mixed clips for one split."""

    audio_dir = Path(cfg.GENERATED_ROOT) / split_name / "audio"

    audio_dir.mkdir(parents=True, exist_ok=True)

    

    rows = []

    num_hard = int(num_mixes * hard_ratio)

    

    for i in tqdm(range(num_mixes), desc=f"Generating {split_name}"):

        hard = i >= (num_mixes - num_hard)

        mix, labels = generate_one_mix(class_files, noise_files, hard_mode=hard)

        

        prefix = "hard" if hard else "mix"

        fname = f"{prefix}_{split_name}_{i:06d}.wav"

        fpath = str(audio_dir / fname)

        sf.write(fpath, mix, cfg.SR)

        

        row = {"filepath": fpath, "hard_mode": int(hard)}

        row.update(labels)

        chosen = [c for c in cfg.CLASSES if labels[c] == 1]

        row["classes_present"] = ",".join(chosen)

        row["num_classes"] = len(chosen)

        rows.append(row)

    

    return pd.DataFrame(rows)

def generate_pure_multilabel(split_df, split_name):

    """Convert single-class source files to multi-label format."""

    sub = split_df[split_df["split"] == split_name]

    rows = []

    for _, r in sub.iterrows():

        row = {"filepath": r["filepath"], "hard_mode": 0}

        for cls in cfg.CLASSES:

            row[cls] = 1 if r["label"] == cls else 0

        row["classes_present"] = r["label"]

        row["num_classes"] = 1

        rows.append(row)

    return pd.DataFrame(rows)

# ─── Generate everything ───

noise_files = load_noise_files(cfg.NOISE_DIR)

all_dfs = {}

mix_counts = {

    "train": cfg.NUM_MIXES_TRAIN,

    "val": cfg.NUM_MIXES_VAL,

    "test": cfg.NUM_MIXES_TEST

}

hard_ratios = {

    "train": cfg.HARD_RATIO_TRAIN,

    "val": 0.3,

    "test": cfg.HARD_RATIO_TEST

}

for split_name in ["train", "val", "test"]:

    logger.info(f"Generating {split_name} set...")

    class_files = build_split_class_dict(split_df, split_name)

    

    for cls in cfg.CLASSES:

        print(f"  {cls}: {len(class_files[cls])} source files")

    

    pure_df = generate_pure_multilabel(split_df, split_name)

    mix_df = generate_dataset_for_split(

        split_name, class_files, mix_counts[split_name],

        noise_files, hard_ratios[split_name]

    )

    

    combined = pd.concat([pure_df, mix_df], ignore_index=True)

    combined = combined.sample(frac=1, random_state=cfg.SEED).reset_index(drop=True)

    

    csv_path = Path(cfg.GENERATED_ROOT) / split_name / "metadata.csv"

    combined.to_csv(csv_path, index=False)

    all_dfs[split_name] = combined

    

    print(f"  → {len(pure_df)} pure + {len(mix_df)} mixed = {len(combined)} total\n")

train_df = all_dfs["train"]

val_df = all_dfs["val"]

test_df = all_dfs["test"]

print(f"✓ Dataset generation complete")

print(f"  Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
# %%

# ===========================================================

# CELL 11: Data generator with label smoothing + mixup

# ===========================================================

class ProductionDataGenerator(tf.keras.utils.Sequence):

    """

    Production data generator for multi-label audio classification.

    

    Features:

    - On-the-fly mel spectrogram extraction (memory efficient)

    - Label smoothing (prevents overconfident predictions)

    - Mixup augmentation (creates virtual training examples)

    - Normalization with pre-computed statistics

    - Audio augmentation during training only

    """

    

    def init(self, df, batch_size=16, augment=False,

                 shuffle=True, mean=None, std=None,

                 label_smoothing=0.0, mixup_alpha=0.0):

        self.df = df.reset_index(drop=True)

        self.batch_size = batch_size

        self.augment = augment

        self.shuffle = shuffle

        self.mean = mean

        self.std = std

        self.label_smoothing = label_smoothing

        self.mixup_alpha = mixup_alpha

        self.indexes = np.arange(len(self.df))

        self.on_epoch_end()

    

    def len(self):

        return int(np.ceil(len(self.df) / self.batch_size))

    

    def getitem(self, idx):

        batch_idx = self.indexes[idx  self.batch_size:(idx + 1)  self.batch_size]

        batch_df = self.df.iloc[batch_idx]

        

        X_batch, y_batch = [], []

        

        for , row in batchdf.iterrows():

            mel = audio_proc.extract_mel_spectrogram(

                row["filepath"], augment=self.augment

            )

            

            # Normalize using training statistics

            if self.mean is not None and self.std is not None:

                mel = (mel - self.mean) / (self.std + 1e-8)

            

            mel = mel[..., np.newaxis]  # Add channel dim for CNN

            

            label = row[cfg.CLASSES].values.astype(np.float32)

            

            # Label smoothing: 1→0.95, 0→0.05

            # WHY: Prevents model from being 100% certain

            if self.label_smoothing > 0:

                label = label * (1 - self.label_smoothing) + self.label_smoothing / 2

            

            X_batch.append(mel)

            y_batch.append(label)

        

        X_batch = np.array(X_batch)

        y_batch = np.array(y_batch)

        

        # Mixup: blend two samples for regularization

        if self.mixup_alpha > 0 and self.augment and len(X_batch) > 1:

            X_batch, y_batch = self._mixup(X_batch, y_batch)

        

        return X_batch, y_batch

    

    def _mixup(self, X, y):

        """

        Mixup augmentation: X_new = λ*X1 + (1-λ)*X2

        Creates virtual training examples → reduces overfitting.

        """

        indices = np.random.permutation(len(X))

        lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)

        lam = max(lam, 1 - lam)

        return lam  X + (1-lam)  X[indices], lam  y + (1-lam)  y[indices]

    

    def on_epoch_end(self):

        if self.shuffle:

            np.random.shuffle(self.indexes)

print("✓ Data generator class defined")

In [ ]:
# %%

# ===========================================================

# CELL 12: Compute normalization statistics from training data

# ===========================================================

# WHY: Neural networks train better with zero-mean unit-variance input.

# We compute mean and std from TRAINING data only.

# Using test data for normalization = data leakage.

# ===========================================================

def compute_normalization_stats(df, num_samples=500):

    """Compute global mean and std from training subset."""

    subset = df.sample(n=min(num_samples, len(df)), random_state=cfg.SEED)

    mels = []

    for _, row in tqdm(subset.iterrows(), total=len(subset), desc="Computing normalization"):

        mel = audio_proc.extract_mel_spectrogram(row["filepath"])

        mels.append(mel)

    mels = np.array(mels)

    mean, std = float(np.mean(mels)), float(np.std(mels))

    

    # Save for production inference

    stats = {"mean": mean, "std": std}

    stats_path = Path(cfg.BEST_MODEL_DIR) / "normalization_stats.json"

    with open(stats_path, 'w') as f:

        json.dump(stats, f, indent=2)

    

    logger.info(f"Normalization: mean={mean:.4f}, std={std:.4f}")

    logger.info(f"Stats saved to: {stats_path}")

    return mean, std

MEAN, STD = compute_normalization_stats(train_df)

In [ ]:
# %%

# ===========================================================

# CELL 13: Create all data generators

# ===========================================================

train_gen = ProductionDataGenerator(

    train_df, batch_size=cfg.BATCH_SIZE, augment=True, shuffle=True,

    mean=MEAN, std=STD,

    label_smoothing=cfg.LABEL_SMOOTHING,

    mixup_alpha=cfg.MIXUP_ALPHA

)

val_gen = ProductionDataGenerator(

    val_df, batch_size=cfg.BATCH_SIZE, augment=False, shuffle=False,

    mean=MEAN, std=STD

    # No smoothing/mixup for validation — need true metric values

)

test_gen = ProductionDataGenerator(

    test_df, batch_size=cfg.BATCH_SIZE, augment=False, shuffle=False,

    mean=MEAN, std=STD

    # No smoothing/mixup for testing

)

# Verify

X_sample, y_sample = train_gen[0]

print(f"✓ Data generators created")

print(f"  Batch X shape: {X_sample.shape}")

print(f"  Batch y shape: {y_sample.shape}")

print(f"  Sample label (with smoothing): {y_sample[0]}")

print(f"  Train batches: {len(train_gen)}")

print(f"  Val batches: {len(val_gen)}")

print(f"  Test batches: {len(test_gen)}")

In [ ]:
# %%

# ===========================================================

# CELL 14: Focal Loss function

# ===========================================================

# WHY Focal Loss instead of standard Binary Cross-Entropy:

#

# Standard BCE treats all examples equally.

# But most examples become "easy" early in training.

# Focal loss DOWN-WEIGHTS easy examples and FOCUSES on hard ones.

#

# gamma: How much to focus on hard examples (2.0 is standard)

# alpha: Class balance weight (0.25 is standard)

#

# This is CRITICAL for defence: hard examples are the ones

# that determine if the system catches real threats.

# ===========================================================

def focal_loss(gamma=2.0, alpha=0.25):

    """

    Focal Loss: FL = -α  (1-p_t)^γ  log(p_t)

    

    - p_t high (easy example): (1-p_t)^γ small → low loss

    - p_t low (hard example): (1-p_t)^γ large → high loss

    """

    def loss_fn(y_true, y_pred):

        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)

        bce = -(y_true * tf.math.log(y_pred) +

                (1 - y_true) * tf.math.log(1 - y_pred))

        p_t = y_true  y_pred + (1 - y_true)  (1 - y_pred)

        focal_weight = alpha  (1 - p_t) * gamma

        return tf.reduce_mean(focal_weight * bce)

    return loss_fn

print("✓ Focal loss defined")

print("\n" + "=" * 60)

print("  PART 1 COMPLETE: Data Pipeline Ready")

print("=" * 60)

print(f"\n  Input shape:    {INPUT_SHAPE}")

print(f"  Num classes:    {cfg.NUM_CLASSES}")

print(f"  Train samples:  {len(train_df):,}")

print(f"  Val samples:    {len(val_df):,}")

print(f"  Test samples:   {len(test_df):,}")

print(f"  Normalization:  mean={MEAN:.4f}, std={STD:.4f}")

In [ ]:
# %%

# ===========================================================

# CELL 15: Attention mechanism building blocks

# ===========================================================

# Three types of attention used in our models:

#

# 1. CHANNEL ATTENTION (Squeeze-and-Excitation):

#    Learns WHICH frequency bands matter for each class.

#    Example: Emphasize low-freq bands for vehicle engine hum.

#

# 2. SPATIAL ATTENTION:

#    Learns WHERE in the spectrogram to focus.

#    Example: Focus on the time region where gunshot occurs.

#

# 3. MULTI-HEAD SELF-ATTENTION:

#    Learns relationships BETWEEN different time-frequency regions.

#    Example: Connect the onset and decay of a jet engine sound.

#

# These are the KEY components that differentiate our best model

# from simpler architectures. We test WITH and WITHOUT attention

# to prove their value.

# ===========================================================

def squeeze_excitation_block(x, ratio=8):

    """

    Squeeze-and-Excitation (SE) Channel Attention.

    

    HOW IT WORKS:

    1. SQUEEZE: Global average pool → one value per channel

       (Summarizes each frequency band's overall energy)

    2. EXCITATION: Two FC layers learn channel importance weights

       (Which frequency bands are useful for this input?)

    3. SCALE: Multiply original features by learned weights

       (Amplify useful bands, suppress irrelevant ones)

    

    WHY FOR DEFENCE AUDIO:

    - Drone sounds concentrate in specific frequency bands (rotor harmonics)

    - Gunshots have energy spread across all bands (broadband impulse)

    - SE block learns to emphasize the RIGHT bands for each sound type

    

    Args:

        x: Input tensor from CNN layer

        ratio: Reduction ratio for bottleneck (smaller = more parameters)

    

    Returns:

        Attention-weighted tensor (same shape as input)

    """

    channels = x.shape[-1]

    

    # Squeeze: spatial dims → single value per channel

    squeeze = GlobalAveragePooling2D()(x)

    squeeze = Reshape((1, 1, channels))(squeeze)

    

    # Excitation: learn channel-wise importance

    excitation = Dense(channels // ratio, activation='relu',

                       kernel_regularizer=l2(cfg.L2_REG))(squeeze)

    excitation = Dense(channels, activation='sigmoid',

                       kernel_regularizer=l2(cfg.L2_REG))(excitation)

    

    # Scale: apply learned weights

    return Multiply()([x, excitation])

def spatial_attention_block(x):

    """

    Spatial Attention Module.

    

    HOW IT WORKS:

    1. Compute average and max across channels at each spatial position

    2. Concatenate these two maps

    3. Learn a 2D attention map via convolution

    4. Multiply original features by spatial attention map

    

    WHY FOR DEFENCE AUDIO:

    - Not all time-frequency regions are equally important

    - A gunshot occupies a very brief time region but wide frequency range

    - A drone occupies specific frequency bands across all time

    - Spatial attention learns WHERE to look in the spectrogram

    

    Args:

        x: Input tensor (batch, freq, time, channels)

    

    Returns:

        Spatially-attended tensor (same shape as input)

    """

    # Average and max pooling across channel dimension

    avg_pool = tf.reduce_mean(x, axis=-1, keepdims=True)

    max_pool = tf.reduce_max(x, axis=-1, keepdims=True)

    

    # Concatenate spatial descriptors

    concat = Concatenate(axis=-1)([avg_pool, max_pool])

    

    # Learn spatial attention map

    attention = Conv2D(1, (7, 7), padding='same', activation='sigmoid',

                       kernel_regularizer=l2(cfg.L2_REG))(concat)

    

    return Multiply()([x, attention])

def cbam_block(x, ratio=8):

    """

    Convolutional Block Attention Module (CBAM).

    

    Combines Channel Attention + Spatial Attention sequentially.

    

    WHY BOTH:

    - Channel attention: WHAT frequency bands to focus on

    - Spatial attention: WHERE in time-frequency to focus

    - Together they provide complete attention guidance

    

    This is more powerful than either attention alone.

    

    Args:

        x: Input tensor

        ratio: SE reduction ratio

    

    Returns:

        Doubly-attended tensor

    """

    # First: channel attention

    x = squeeze_excitation_block(x, ratio)

    # Then: spatial attention

    x = spatial_attention_block(x)

    return x

def temporal_attention_layer(x, num_heads=4, key_dim=64):

    """

    Multi-Head Self-Attention for temporal sequences.

    

    HOW IT WORKS:

    1. Input sequence: each time step has a feature vector

    2. Multi-head attention computes relationships between ALL time steps

    3. Each "head" learns different relationship patterns

    4. Output: enriched features that capture long-range temporal dependencies

    

    WHY FOR DEFENCE AUDIO:

    - A jet engine sound has gradual onset → peak → decay

    - Self-attention connects distant time steps directly

    - GRU processes sequentially; attention sees everything at once

    - This catches patterns that RNNs might miss

    

    Args:

        x: Input tensor (batch, time_steps, features)

        num_heads: Number of attention heads (more = more patterns)

        key_dim: Dimension of query/key vectors

    

    Returns:

        Attention-enriched tensor (same shape as input)

    """

    # Multi-head self-attention

    attn_output = tf.keras.layers.MultiHeadAttention(

        num_heads=num_heads,

        key_dim=key_dim,

        dropout=0.1

    )(x, x)  # Self-attention: query = key = value = x

    

    # Residual connection + layer normalization

    # WHY: Prevents degradation in deep networks

    x = Add()([x, attn_output])

    x = LayerNormalization()(x)

    

    return x

print("✓ Attention mechanisms defined:")

print("  - Squeeze-and-Excitation (Channel Attention)")

print("  - Spatial Attention")

print("  - CBAM (Channel + Spatial)")

print("  - Multi-Head Self-Attention (Temporal)")

In [ ]:
# %%

# ===========================================================

# CELL 16: CNN building blocks

# ===========================================================

# Reusable CNN blocks with consistent structure.

# Each block: Conv → BN → Conv → BN → [Attention] → Pool → Dropout

#

# WHY double convolution per block:

# - First conv detects basic patterns

# - Second conv combines basic patterns into complex features

# - This is the VGG-style design, proven effective for audio

# ===========================================================

def cnn_block(x, filters, dropout_rate=0.25, use_attention=None):

    """

    Standard CNN block with optional attention.

    

    Args:

        x: Input tensor

        filters: Number of convolution filters

        dropout_rate: Dropout probability after pooling

        use_attention: None, 'se', 'spatial', 'cbam'

    

    Returns:

        Feature tensor after conv + optional attention + pool + dropout

    """

    # First convolution: detect basic patterns

    x = Conv2D(filters, (3, 3), padding='same', activation='relu',

               kernel_regularizer=l2(cfg.L2_REG))(x)

    x = BatchNormalization()(x)

    

    # Second convolution: combine basic patterns

    x = Conv2D(filters, (3, 3), padding='same', activation='relu',

               kernel_regularizer=l2(cfg.L2_REG))(x)

    x = BatchNormalization()(x)

    

    # Optional attention mechanism

    if use_attention == 'se':

        x = squeeze_excitation_block(x)

    elif use_attention == 'spatial':

        x = spatial_attention_block(x)

    elif use_attention == 'cbam':

        x = cbam_block(x)

    

    # Downsample spatial dimensions by 2x

    x = MaxPooling2D((2, 2))(x)

    

    # Regularization

    x = Dropout(dropout_rate)(x)

    

    return x

def classification_head(x, num_classes, dropout_rate=0.5):

    """

    Dense classification head for multi-label output.

    

    WHY two Dense layers:

    - First layer (256): learns high-level feature combinations

    - Second layer (128): further abstraction before output

    - High dropout (0.5): prevents memorization

    

    WHY sigmoid (not softmax):

    - Softmax: classes compete (probabilities sum to 1)

    - Sigmoid: each class is independent (can have multiple active)

    - We NEED sigmoid because one clip can contain drone + vehicle

    """

    x = Dense(256, activation='relu', kernel_regularizer=l2(cfg.L2_REG))(x)

    x = Dropout(dropout_rate)(x)

    x = Dense(128, activation='relu', kernel_regularizer=l2(cfg.L2_REG))(x)

    x = Dropout(dropout_rate)(x)

    

    # Sigmoid: independent probability per class

    out = Dense(num_classes, activation='sigmoid', name='output')(x)

    return out

print("✓ CNN building blocks defined")

In [ ]:
# %%

# ===========================================================

# CELL 17: All 8 model architectures

# ===========================================================

# We define 8 models to compare:

#

# WITHOUT ATTENTION:

#   1. Dense_Only      — Baseline MLP (no spatial/temporal awareness)

#   2. CNN_Only        — Pure CNN (no temporal modeling)

#   3. GRU_Only        — Pure BiGRU (no CNN feature extraction)

#   4. CNN_LSTM        — CNN + BiLSTM

#   5. CNN_GRU         — CNN + BiGRU (no attention)

#

# WITH ATTENTION:

#   6. CNN_GRU_SE      — CNN + SE Attention + BiGRU

#   7. CNN_GRU_CBAM    — CNN + CBAM (Channel+Spatial) + BiGRU

#   8. CNN_GRU_MultiHead — CNN + SE + BiGRU + Multi-Head Self-Attention

#

# This lets us prove:

# - CNN is needed (compare GRU_Only vs CNN_GRU)

# - Temporal modeling is needed (compare CNN_Only vs CNN_GRU)

# - Attention helps (compare CNN_GRU vs CNN_GRU_SE vs CNN_GRU_CBAM)

# - Multi-head attention helps (compare CNN_GRU_SE vs CNN_GRU_MultiHead)

# ===========================================================

def build_dense_only(input_shape, num_classes):

    """

    MODEL 1: Simple MLP (Dense layers only).

    

    PURPOSE: Absolute baseline.

    - Flattens spectrogram → treats as flat feature vector

    - NO spatial awareness (doesn't know frequency bands)

    - NO temporal awareness (doesn't know time progression)

    

    EXPECTED: Worst performance. Proves that structure matters.

    """

    inp = Input(shape=input_shape, name="mel_input")

    

    x = Flatten()(inp)

    x = Dense(512, activation='relu', kernel_regularizer=l2(cfg.L2_REG))(x)

    x = Dropout(0.5)(x)

    x = Dense(256, activation='relu', kernel_regularizer=l2(cfg.L2_REG))(x)

    x = Dropout(0.5)(x)

    x = Dense(128, activation='relu', kernel_regularizer=l2(cfg.L2_REG))(x)

    x = Dropout(0.4)(x)

    

    out = Dense(num_classes, activation='sigmoid', name='output')(x)

    return Model(inputs=inp, outputs=out, name="Dense_Only")

def build_cnn_only(input_shape, num_classes):

    """

    MODEL 2: Pure CNN (no RNN, no attention).

    

    PURPOSE: Tests if CNN alone is sufficient.

    - Extracts spatial features from spectrogram

    - Global Average Pooling collapses time dimension

    - NO temporal modeling

    

    EXPECTED: Decent for pure clips, worse for mixed/temporal events.

    """

    inp = Input(shape=input_shape, name="mel_input")

    

    x = cnn_block(inp, 32, 0.25, use_attention=None)

    x = cnn_block(x, 64, 0.3, use_attention=None)

    x = cnn_block(x, 128, 0.35, use_attention=None)

    x = cnn_block(x, 256, 0.4, use_attention=None)

    

    x = GlobalAveragePooling2D()(x)

    out = classification_head(x, num_classes)

    

    return Model(inputs=inp, outputs=out, name="CNN_Only")

def build_gru_only(input_shape, num_classes):

    """

    MODEL 3: Pure BiGRU (no CNN).

    

    PURPOSE: Tests if temporal modeling alone is enough.

    - Reshapes spectrogram: treat time frames as sequence

    - Each time frame's mel bands = feature vector

    - BiGRU processes the sequence

    - NO convolutional feature extraction

    

    EXPECTED: Moderate. Missing local pattern detection that CNN provides.

    """

    inp = Input(shape=input_shape, name="mel_input")

    

    # Reshape: (128, 259, 1) → (259, 128) — time steps × features

    x = Reshape((input_shape[1], input_shape[0]))(inp)

    

    x = Bidirectional(GRU(128, return_sequences=True, dropout=0.3,

                          recurrent_dropout=0.15))(x)

    x = Bidirectional(GRU(64, return_sequences=True, dropout=0.3,

                          recurrent_dropout=0.15))(x)

    x = GlobalAveragePooling1D()(x)

    

    out = classification_head(x, num_classes)

    return Model(inputs=inp, outputs=out, name="GRU_Only")

def build_cnn_lstm(input_shape, num_classes):

    """

    MODEL 4: CNN + BiLSTM (no attention).

    

    PURPOSE: Compare LSTM vs GRU as the recurrent component.

    - LSTM has explicit forget gate → may capture longer dependencies

    - But has 33% more parameters than GRU → slower, more overfitting risk

    

    EXPECTED: Similar or slightly better than CNN_GRU on long events,

    but potentially more overfitting due to more parameters.

    """

    inp = Input(shape=input_shape, name="mel_input")

    

    x = cnn_block(inp, 32, 0.25, use_attention=None)

    x = cnn_block(x, 64, 0.3, use_attention=None)

    x = cnn_block(x, 128, 0.35, use_attention=None)

    x = cnn_block(x, 256, 0.4, use_attention=None)

    

    # Reshape for RNN: (freq_reduced, time_reduced × channels)

    shape = x.shape

    x = Reshape((shape[1], shape[2] * shape[3]))(x)

    

    # BiLSTM instead of BiGRU

    x = Bidirectional(tf.keras.layers.LSTM(

        128, return_sequences=True, dropout=0.35, recurrent_dropout=0.2

    ))(x)

    x = Bidirectional(tf.keras.layers.LSTM(

        64, return_sequences=True, dropout=0.35, recurrent_dropout=0.2

    ))(x)

    x = GlobalAveragePooling1D()(x)

    

    out = classification_head(x, num_classes)

    return Model(inputs=inp, outputs=out, name="CNN_LSTM")

def build_cnn_gru(input_shape, num_classes):

    """

    MODEL 5: CNN + BiGRU (no attention).

    

    PURPOSE: The "standard" CRNN baseline.

    - CNN extracts local spectro-temporal features

    - BiGRU captures temporal dependencies

    - No attention mechanism

    

    This is the model to compare attention variants against.

    

    WHY GRU over LSTM:

    - 33% fewer parameters (faster training)

    - Similar or better performance on audio tasks

    - Less overfitting risk

    """

    inp = Input(shape=input_shape, name="mel_input")

    

    x = cnn_block(inp, 32, 0.25, use_attention=None)

    x = cnn_block(x, 64, 0.3, use_attention=None)

    x = cnn_block(x, 128, 0.35, use_attention=None)

    x = cnn_block(x, 256, 0.4, use_attention=None)

    

    shape = x.shape

    x = Reshape((shape[1], shape[2] * shape[3]))(x)

    

    x = Bidirectional(GRU(128, return_sequences=True,

                          dropout=0.35, recurrent_dropout=0.2))(x)

    x = Bidirectional(GRU(64, return_sequences=True,

                          dropout=0.35, recurrent_dropout=0.2))(x)

    x = GlobalAveragePooling1D()(x)

    

    out = classification_head(x, num_classes)

    return Model(inputs=inp, outputs=out, name="CNN_GRU")

def build_cnn_gru_se(input_shape, num_classes):

    """

    MODEL 6: CNN + SE Attention + BiGRU.

    

    PURPOSE: Test if channel attention improves CNN+GRU.

    - SE blocks after each CNN layer learn which mel bands matter

    - Same GRU layers for temporal modeling

    

    EXPECTED: Better than CNN_GRU because attention helps

    the CNN focus on class-relevant frequency bands.

    """

    inp = Input(shape=input_shape, name="mel_input")

    

    # SE attention after each CNN block

    x = cnn_block(inp, 32, 0.25, use_attention='se')

    x = cnn_block(x, 64, 0.3, use_attention='se')

    x = cnn_block(x, 128, 0.35, use_attention='se')

    x = cnn_block(x, 256, 0.4, use_attention='se')

    

    shape = x.shape

    x = Reshape((shape[1], shape[2] * shape[3]))(x)

    

    x = Bidirectional(GRU(128, return_sequences=True,

                          dropout=0.35, recurrent_dropout=0.2))(x)

    x = Bidirectional(GRU(64, return_sequences=True,

                          dropout=0.35, recurrent_dropout=0.2))(x)

    x = GlobalAveragePooling1D()(x)

    

    out = classification_head(x, num_classes)

    return Model(inputs=inp, outputs=out, name="CNN_GRU_SE")

def build_cnn_gru_cbam(input_shape, num_classes):

    """

    MODEL 7: CNN + CBAM (Channel+Spatial) Attention + BiGRU.

    

    PURPOSE: Test if adding spatial attention on top of channel

    attention provides further improvement.

    - CBAM = Channel Attention (what) + Spatial Attention (where)

    - More parameters than SE alone, but richer attention

    

    EXPECTED: Best or near-best CNN performance.

    May have diminishing returns over SE alone.

    """

    inp = Input(shape=input_shape, name="mel_input")

    

    # CBAM attention (channel + spatial) after each CNN block

    x = cnn_block(inp, 32, 0.25, use_attention='cbam')

    x = cnn_block(x, 64, 0.3, use_attention='cbam')

    x = cnn_block(x, 128, 0.35, use_attention='cbam')

    x = cnn_block(x, 256, 0.4, use_attention='cbam')

    

    shape = x.shape

    x = Reshape((shape[1], shape[2] * shape[3]))(x)

    

    x = Bidirectional(GRU(128, return_sequences=True,

                          dropout=0.35, recurrent_dropout=0.2))(x)

    x = Bidirectional(GRU(64, return_sequences=True,

                          dropout=0.35, recurrent_dropout=0.2))(x)

    x = GlobalAveragePooling1D()(x)

    

    out = classification_head(x, num_classes)

    return Model(inputs=inp, outputs=out, name="CNN_GRU_CBAM")

def build_cnn_gru_multihead(input_shape, num_classes):

    """

    MODEL 8: CNN + SE + BiGRU + Multi-Head Self-Attention.

    

    PURPOSE: Our MOST ADVANCED architecture.

    Combines ALL attention types:

    - SE attention in CNN (channel/frequency focus)

    - BiGRU for sequential temporal modeling

    - Multi-head self-attention for long-range temporal relationships

    

    WHY THIS IS THE STRONGEST:

    - CNN+SE: Learns which frequency bands are important

    - BiGRU: Captures local temporal patterns (onset, decay)

    - Multi-head attention: Captures GLOBAL temporal relationships

      that GRU might miss (e.g., connecting jet engine approach

      with its departure sound)

    

    TRADEOFF: Most parameters, slowest training.

    But for defence, accuracy is more important than speed.

    

    EXPECTED: Best overall performance, especially on

    complex multi-class overlapping scenarios.

    """

    inp = Input(shape=input_shape, name="mel_input")

    

    # CNN with SE attention

    x = cnn_block(inp, 32, 0.25, use_attention='se')

    x = cnn_block(x, 64, 0.3, use_attention='se')

    x = cnn_block(x, 128, 0.35, use_attention='se')

    x = cnn_block(x, 256, 0.4, use_attention='se')

    

    shape = x.shape

    x = Reshape((shape[1], shape[2] * shape[3]))(x)

    

    # BiGRU for local temporal modeling

    x = Bidirectional(GRU(128, return_sequences=True,

                          dropout=0.35, recurrent_dropout=0.2))(x)

    

    # Multi-head self-attention for global temporal relationships

    x = temporal_attention_layer(x, num_heads=4, key_dim=64)

    

    # Second BiGRU after attention enrichment

    x = Bidirectional(GRU(64, return_sequences=True,

                          dropout=0.35, recurrent_dropout=0.2))(x)

    

    # Second multi-head attention

    x = temporal_attention_layer(x, num_heads=2, key_dim=32)

    

    x = GlobalAveragePooling1D()(x)

    

    out = classification_head(x, num_classes)

    return Model(inputs=inp, outputs=out, name="CNN_GRU_MultiHead")

print("✓ All 8 model architectures defined")

In [ ]:
# %%

# ===========================================================

# CELL 18: Model Registry

# ===========================================================

# Central registry of all models with metadata.

# This pattern makes it easy to:

# - Add new models

# - Run selective training

# - Generate comparison reports

# ===========================================================

MODEL_REGISTRY = OrderedDict({

    # ── WITHOUT ATTENTION ──

    "Dense_Only": {

        "builder": build_dense_only,

        "description": "Baseline MLP — no spatial or temporal awareness",

        "has_attention": False,

        "has_temporal": False,

        "category": "baseline"

    },

    "CNN_Only": {

        "builder": build_cnn_only,

        "description": "Pure CNN — spatial features, no temporal modeling",

        "has_attention": False,

        "has_temporal": False,

        "category": "cnn"

    },

    "GRU_Only": {

        "builder": build_gru_only,

        "description": "Pure BiGRU — temporal modeling, no CNN features",

        "has_attention": False,

        "has_temporal": True,

        "category": "rnn"

    },

    "CNN_LSTM": {

        "builder": build_cnn_lstm,

        "description": "CNN + BiLSTM — LSTM variant for comparison",

        "has_attention": False,

        "has_temporal": True,

        "category": "crnn"

    },

    "CNN_GRU": {

        "builder": build_cnn_gru,

        "description": "CNN + BiGRU — standard CRNN baseline",

        "has_attention": False,

        "has_temporal": True,

        "category": "crnn"

    },

    # ── WITH ATTENTION ──

    "CNN_GRU_SE": {

        "builder": build_cnn_gru_se,

        "description": "CNN + SE channel attention + BiGRU",

        "has_attention": True,

        "has_temporal": True,

        "category": "attention"

    },

    "CNN_GRU_CBAM": {

        "builder": build_cnn_gru_cbam,

        "description": "CNN + CBAM (channel+spatial) attention + BiGRU",

        "has_attention": True,

        "has_temporal": True,

        "category": "attention"

    },

    "CNN_GRU_MultiHead": {

        "builder": build_cnn_gru_multihead,

        "description": "CNN + SE + BiGRU + Multi-Head Self-Attention (FULL)",

        "has_attention": True,

        "has_temporal": True,

        "category": "advanced"

    },

})

print("=" * 70)

print("MODEL REGISTRY")

print("=" * 70)

print(f"\n{'#':<3} {'Name':<22} {'Attention':<12} {'Temporal':<10} {'Category'}")

print("-" * 70)

for i, (name, info) in enumerate(MODEL_REGISTRY.items(), 1):

    att = "✓" if info["has_attention"] else "✗"

    temp = "✓" if info["has_temporal"] else "✗"

    print(f"{i:<3} {name:<22} {att:<12} {temp:<10} {info['category']}")

print(f"\nTotal models to train: {len(MODEL_REGISTRY)}")

In [ ]:
# %%

# ===========================================================

# CELL 19: Preview all model architectures

# ===========================================================

# Build each model once to verify shapes and count parameters.

# This catches shape mismatches BEFORE wasting time on training.

# ===========================================================

print("=" * 70)

print("MODEL PARAMETER COUNTS")

print("=" * 70)

model_param_counts = {}

for name, info in MODEL_REGISTRY.items():

    try:

        # Build model

        test_model = info["builder"](INPUT_SHAPE, cfg.NUM_CLASSES)

        total_params = test_model.count_params()

        trainable = sum(

            tf.keras.backend.count_params(w) for w in test_model.trainable_weights

        )

        non_trainable = total_params - trainable

        

        model_param_counts[name] = {

            "total": total_params,

            "trainable": trainable,

            "non_trainable": non_trainable

        }

        

        print(f"\n{name}:")

        print(f"  Total params:     {total_params:>12,}")

        print(f"  Trainable:        {trainable:>12,}")

        print(f"  Non-trainable:    {non_trainable:>12,}")

        print(f"  Status: ✓ Built successfully")

        

        # Clean up

        del test_model

        tf.keras.backend.clear_session()

        

    except Exception as e:

        print(f"\n{name}:")

        print(f"  ❌ BUILD FAILED: {e}")

        model_param_counts[name] = {"total": 0, "trainable": 0, "non_trainable": 0}

gc.collect()

In [ ]:
# %%

# ===========================================================

# CELL 20: VISUALIZATION — Model size comparison

# ===========================================================

# SIGNIFICANCE:

# Shows the parameter count for each architecture.

# More parameters = more capacity but also more overfitting risk.

# For deployment on edge devices, smaller models are preferred.

#

# INTERPRETATION:

# - Dense_Only has MOST params (flattened input is huge)

# - CNN_Only is moderate

# - Attention models add ~5-15% more params over base CNN+GRU

# - The added parameters of attention are small but impactful

# ===========================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

names = list(model_param_counts.keys())

totals = [model_param_counts[n]["total"] / 1e6 for n in names]

trainables = [model_param_counts[n]["trainable"] / 1e6 for n in names]

# Color by category

category_colors = {

    "baseline": "#9E9E9E",

    "cnn": "#FF9800",

    "rnn": "#2196F3",

    "crnn": "#9C27B0",

    "attention": "#4CAF50",

    "advanced": "#F44336"

}

colors = [category_colors[MODEL_REGISTRY[n]["category"]] for n in names]

# Bar chart

bars = axes[0].barh(names, totals, color=colors, edgecolor='black')

for bar, val in zip(bars, totals):

    axes[0].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,

                f'{val:.2f}M', va='center', fontweight='bold', fontsize=9)

axes[0].set_xlabel("Parameters (Millions)", fontsize=12)

axes[0].set_title("Total Parameters per Model", fontsize=13, fontweight='bold')

axes[0].invert_yaxis()

# Pie chart of categories

category_params = defaultdict(float)

for n in names:

    cat = MODEL_REGISTRY[n]["category"]

    category_params[cat] += model_param_counts[n]["total"]

cats = list(category_params.keys())

cat_vals = [category_params[c] / 1e6 for c in cats]

cat_colors = [category_colors[c] for c in cats]

axes[1].pie(cat_vals, labels=cats, autopct='%1.1f%%', colors=cat_colors,

           startangle=90, textprops={'fontsize': 11})

axes[1].set_title("Parameter Distribution by Category", fontsize=13, fontweight='bold')

plt.suptitle("Model Architecture Complexity Analysis",

             fontsize=15, fontweight='bold', y=1.02)

plt.tight_layout()

plt.savefig(f"{cfg.PLOTS_DIR}/model_comparison/model_sizes.png",

            dpi=150, bbox_inches='tight')

plt.show()

In [ ]:
# %%

# ===========================================================

# CELL 21: Training infrastructure

# ===========================================================

# Functions for compiling, training, and evaluating models.

# Separated from the loop for clarity and reusability.

# ===========================================================

def compile_model(model):

    """

    Compile model with focal loss and multi-label metrics.

    

    METRICS TRACKED:

    - bin_acc: Per-label binary accuracy

    - precision: True positives / (True positives + False positives)

    - recall: True positives / (True positives + False negatives)

    - auc: Area Under ROC Curve (overall discrimination quality)

    

    All metrics are computed per-batch during training for monitoring.

    Full evaluation is done separately on the complete test set.

    """

    model.compile(

        optimizer=tf.keras.optimizers.Adam(learning_rate=cfg.LEARNING_RATE),

        loss=focal_loss(gamma=2.0, alpha=0.25),

        metrics=[

            tf.keras.metrics.BinaryAccuracy(name='bin_acc'),

            tf.keras.metrics.Precision(name='precision'),

            tf.keras.metrics.Recall(name='recall'),

            tf.keras.metrics.AUC(name='auc', multi_label=True)

        ]

    )

    return model

def get_callbacks(model_name):

    """

    Create training callbacks for a specific model.

    

    CALLBACKS:

    1. EarlyStopping: Stop if val_loss doesn't improve for 7 epochs.

       WHY: Prevents wasting compute and overfitting.

    

    2. ReduceLROnPlateau: Halve learning rate if stuck for 4 epochs.

       WHY: Fine-tunes learning in later epochs.

    

    3. ModelCheckpoint: Save best model based on val_auc.

       WHY: We want the model that DISCRIMINATES best, not just lowest loss.

    

    4. CSVLogger: Save epoch-by-epoch metrics to CSV.

       WHY: For post-training analysis and debugging.

    """

    model_dir = f"{cfg.MODELS_DIR}/{model_name}"

    os.makedirs(model_dir, exist_ok=True)

    

    return [

        EarlyStopping(

            monitor='val_loss',

            patience=7,

            restore_best_weights=True,

            verbose=1

        ),

        ReduceLROnPlateau(

            monitor='val_loss',

            factor=0.5,

            patience=4,

            min_lr=1e-7,

            verbose=1

        ),

        ModelCheckpoint(

            filepath=f"{model_dir}/model.keras",

            monitor='val_auc',

            mode='max',

            save_best_only=True,

            verbose=0

        ),

        CSVLogger(

            f"{model_dir}/training_log.csv",

            append=False

        )

    ]

def evaluate_model_on_test(model, test_gen, test_df):

    """

    Evaluate a trained model on the test set.

    

    Returns:

        y_true: Ground truth binary labels

        y_pred: Thresholded predictions

        y_probs: Raw probability outputs

        metrics_dict: Dictionary of all computed metrics

    """

    # Collect predictions

    y_true_list = []

    y_pred_list = []

    

    for i in range(len(test_gen)):

        X_batch, y_batch = test_gen[i]

        preds = model.predict(X_batch, verbose=0)

        y_true_list.append(y_batch)

        y_pred_list.append(preds)

    

    y_true = (np.vstack(y_true_list) >= 0.5).astype(int)

    y_probs = np.vstack(y_pred_list)

    y_pred = (y_probs >= cfg.THRESHOLD).astype(int)

    

    # Compute metrics

    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)

    samples_f1 = f1_score(y_true, y_pred, average='samples', zero_division=0)

    macro_p = precision_score(y_true, y_pred, average='macro', zero_division=0)

    macro_r = recall_score(y_true, y_pred, average='macro', zero_division=0)

    h_loss = hamming_loss(y_true, y_pred)

    exact = np.all(y_pred == y_true, axis=1).mean()

    

    # Per-class AP and mAP

    aps = []

    per_class_f1 = []

    for j in range(cfg.NUM_CLASSES):

        try:

            ap = average_precision_score(y_true[:, j], y_probs[:, j])

        except:

            ap = 0.0

        aps.append(ap)

        per_class_f1.append(

            f1_score(y_true[:, j], y_pred[:, j], zero_division=0)

        )

    mAP = np.mean(aps)

    

    metrics = {

        "macro_f1": round(macro_f1, 4),

        "micro_f1": round(micro_f1, 4),

        "samples_f1": round(samples_f1, 4),

        "macro_precision": round(macro_p, 4),

        "macro_recall": round(macro_r, 4),

        "hamming_loss": round(h_loss, 4),

        "exact_match": round(exact, 4),

        "mAP": round(mAP, 4),

    }

    

    # Per-class metrics

    for j, cls in enumerate(cfg.CLASSES):

        metrics[f"AP_{cls}"] = round(aps[j], 4)

        metrics[f"F1_{cls}"] = round(per_class_f1[j], 4)

    

    return y_true, y_pred, y_probs, metrics

def measure_inference_speed(model, num_runs=50):

    """

    Measure average inference time per sample.

    

    WHY: For real-time defence systems, latency matters.

    A model that takes 500ms per clip may miss fast events.

    Target: <100ms per 6-second clip for real-time operation.

    """

    dummy = np.random.randn(1, *INPUT_SHAPE).astype(np.float32)

    

    # Warmup (first run is always slower due to graph compilation)

    model.predict(dummy, verbose=0)

    

    start = time.time()

    for  in range(numruns):

        model.predict(dummy, verbose=0)

    elapsed = (time.time() - start) / num_runs * 1000  # ms

    

    return round(elapsed, 2)

print("✓ Training infrastructure ready")

In [ ]:
# %%

# ===========================================================

# CELL 22: MAIN TRAINING LOOP — Train all 8 models

# ===========================================================

# This is the core training cell.

#

# For each model:

# 1. Build architecture

# 2. Compile with focal loss

# 3. Train with callbacks

# 4. Validate training metrics (catch suspicious values)

# 5. Evaluate on test set

# 6. Validate test metrics

# 7. Measure inference speed

# 8. Save everything (model, history, results)

#

# All results are collected for comparison in Part 3.

#

# ⏱ ESTIMATED TIME on Intel i5-1335U:

# - Dense_Only: ~5 min

# - CNN_Only: ~10 min

# - GRU_Only: ~15 min

# - CNN_LSTM: ~25 min

# - CNN_GRU: ~20 min

# - CNN_GRU_SE: ~22 min

# - CNN_GRU_CBAM: ~25 min

# - CNN_GRU_MultiHead: ~30 min

# Total: ~2.5-3 hours

#

# TIP: If too slow, reduce NUM_MIXES_TRAIN to 3000 in Config.

# ===========================================================

# Storage for all results

all_results = OrderedDict()

all_histories = OrderedDict()

all_predictions = OrderedDict()

all_validation_reports = []

print("=" * 70)

print("  STARTING MODEL TRAINING LOOP")

print(f"  Models to train: {len(MODEL_REGISTRY)}")

print(f"  Epochs (max): {cfg.EPOCHS}")

print(f"  Batch size: {cfg.BATCH_SIZE}")

print(f"  Training samples: {len(train_df):,}")

print("=" * 70)

training_start_time = time.time()

for model_idx, (model_name, model_info) in enumerate(MODEL_REGISTRY.items(), 1):

    

    print(f"\n{'═'*70}")

    print(f"  [{model_idx}/{len(MODEL_REGISTRY)}] Training: {model_name}")

    print(f"  {model_info['description']}")

    print(f"  Attention: {'✓' if model_info['has_attention'] else '✗'}")

    print(f"  Temporal:  {'✓' if model_info['has_temporal'] else '✗'}")

    print(f"{'═'*70}")

    

    model_start_time = time.time()

    

    # ── Step 1: Clear previous model from memory ──

    tf.keras.backend.clear_session()

    gc.collect()

    

    # ── Step 2: Build model ──

    try:

        current_model = model_info["builder"](INPUT_SHAPE, cfg.NUM_CLASSES)

        total_params = current_model.count_params()

        print(f"  Parameters: {total_params:,}")

    except Exception as e:

        print(f"  ❌ BUILD FAILED: {e}")

        all_results[model_name] = {"status": "BUILD_FAILED", "error": str(e)}

        continue

    

    # ── Step 3: Compile ──

    current_model = compile_model(current_model)

    

    # ── Step 4: Train ──

    try:

        callbacks = get_callbacks(model_name)

        

        history = current_model.fit(

            train_gen,

            validation_data=val_gen,

            epochs=cfg.EPOCHS,

            callbacks=callbacks,

            verbose=1

        )

        

        # Save history

        history_dict = {k: [float(v) for v in vals] for k, vals in history.history.items()}

        all_histories[model_name] = history_dict

        

        with open(f"{cfg.MODELS_DIR}/{model_name}/history.json", 'w') as f:

            json.dump(history_dict, f, indent=2)

        

    except Exception as e:

        print(f"  ❌ TRAINING FAILED: {e}")

        all_results[model_name] = {"status": "TRAIN_FAILED", "error": str(e)}

        continue

    

    # ── Step 5: Validate training metrics ──

    print(f"\n  Validating training metrics...")

    train_report = validator.validate_training_metrics(history_dict, model_name)

    validator.print_report(train_report)

    all_validation_reports.append(train_report)

    

    # ── Step 6: Evaluate on test set ──

    print(f"\n  Evaluating on test set...")

    y_true, y_pred, y_probs, test_metrics = evaluate_model_on_test(

        current_model, test_gen, test_df

    )

    

    # ── Step 7: Validate test metrics ──

    test_report = validator.validate_test_metrics(

        y_true, y_pred, y_probs, model_name

    )

    validator.print_report(test_report)

    all_validation_reports.append(test_report)

    

    # ── Step 8: Measure inference speed ──

    inference_ms = measure_inference_speed(current_model)

    print(f"\n  Inference speed: {inference_ms:.1f} ms/sample")

    

    # ── Step 9: Compile all results ──

    training_time = time.time() - model_start_time

    

    result = {

        "status": "OK",

        "params": total_params,

        "epochs_trained": len(history_dict['loss']),

        "training_time_sec": round(training_time, 1),

        "inference_ms": inference_ms,

        "has_attention": model_info["has_attention"],

        "has_temporal": model_info["has_temporal"],

        "category": model_info["category"],

        "train_validation": train_report["status"],

        "test_validation": test_report["status"],

    }

    result.update(test_metrics)

    

    all_results[model_name] = result

    all_predictions[model_name] = {

        "y_true": y_true,

        "y_pred": y_pred,

        "y_probs": y_probs

    }

    

    # ── Step 10: Save results ──

    with open(f"{cfg.MODELS_DIR}/{model_name}/results.json", 'w') as f:

        json.dump(result, f, indent=2, default=str)

    

    # ── Print summary ──

    print(f"\n  {'─'*50}")

    print(f"  RESULTS: {model_name}")

    print(f"  {'─'*50}")

    print(f"  Macro F1:       {test_metrics['macro_f1']:.4f}")

    print(f"  mAP:            {test_metrics['mAP']:.4f}")

    print(f"  Exact Match:    {test_metrics['exact_match']:.4f}")

    print(f"  Hamming Loss:   {test_metrics['hamming_loss']:.4f}")

    print(f"  Precision:      {test_metrics['macro_precision']:.4f}")

    print(f"  Recall:         {test_metrics['macro_recall']:.4f}")

    print(f"  Training:       {training_time/60:.1f} min")

    print(f"  Inference:      {inference_ms:.1f} ms/sample")

    print(f"  Validation:     Train={train_report['status']}, Test={test_report['status']}")

total_time = time.time() - training_start_time

print(f"\n{'═'*70}")

print(f"  ALL TRAINING COMPLETE")

print(f"  Total time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")

print(f"{'═'*70}")

In [ ]:
# %%

# ===========================================================

# CELL 23: Results table

# ===========================================================

# Create a clean DataFrame of all results for analysis.

# This is saved as CSV for the final report.

# ===========================================================

# Build comparison DataFrame

results_rows = []

for name, res in all_results.items():

    if res.get("status") not in ["OK"]:

        continue

    row = {"model": name}

    row.update(res)

    results_rows.append(row)

results_df = pd.DataFrame(results_rows)

# Sort by macro_f1 descending

if "macro_f1" in results_df.columns:

    results_df = results_df.sort_values("macro_f1", ascending=False)

# Save

results_df.to_csv(f"{cfg.REPORTS_DIR}/model_comparison_report.csv", index=False)

# Display

print("\n" + "=" * 90)

print("  MODEL COMPARISON TABLE (sorted by Macro F1)")

print("=" * 90)

display_cols = ["model", "macro_f1", "mAP", "exact_match", "hamming_loss",

                "macro_precision", "macro_recall", "params", "inference_ms",

                "has_attention", "train_validation", "test_validation"]

available_cols = [c for c in display_cols if c in results_df.columns]

print(results_df[available_cols].to_string(index=False))

In [ ]:


# %%

# ===========================================================

# CELL 24: Auto-select and save best model

# ===========================================================

# The model with the highest macro_f1 that PASSES validation

# is automatically selected as the production model.

#

# WHY macro_f1 as selection criterion:

# - F1 balances precision and recall

# - Macro averages across classes equally

# - Better than accuracy for multi-label tasks

# - Defence needs both: catch threats (recall) AND

#   avoid false alarms (precision)

# ===========================================================

def select_best_model():

    """

    Select the best model based on:

    1. Must pass validation (no FAIL status)

    2. Highest macro_f1 among passing models

    3. If tied, prefer model with lower hamming_loss

    """

    candidates = []

    

    for name, res in all_results.items():

        if res.get("status") != "OK":

            continue

        if res.get("test_validation") == "FAIL":

            logger.warning(f"Skipping {name}: failed test validation")

            continue

        candidates.append((name, res))

    

    if not candidates:

        logger.error("No models passed validation!")

        return None

    

    # Sort by macro_f1 descending, then hamming_loss ascending

    candidates.sort(key=lambda x: (-x[1]["macro_f1"], x[1]["hamming_loss"]))

    

    best_name, best_result = candidates[0]

    return best_name, best_result

selection = select_best_model()

if selection:

    best_model_name, best_result = selection

    

    print(f"\n{'═'*60}")

    print(f"  🏆 BEST MODEL: {best_model_name}")

    print(f"{'═'*60}")

    print(f"  Macro F1:      {best_result['macro_f1']:.4f}")

    print(f"  mAP:           {best_result['mAP']:.4f}")

    print(f"  Exact Match:   {best_result['exact_match']:.4f}")

    print(f"  Hamming Loss:  {best_result['hamming_loss']:.4f}")

    print(f"  Inference:     {best_result['inference_ms']:.1f} ms")

    print(f"  Attention:     {'Yes' if best_result['has_attention'] else 'No'}")

    

    # Copy best model files to best_model directory

    import shutil

    

    src_dir = Path(cfg.MODELS_DIR) / best_model_name

    dst_dir = Path(cfg.BEST_MODEL_DIR)

    

    # Copy model file

    src_model = src_dir / "model.keras"

    if src_model.exists():

        shutil.copy2(src_model, dst_dir / "model.keras")

        print(f"\n  Model saved to: {dst_dir / 'model.keras'}")

    

    # Save config for the best model

    best_config = {

        "model_name": best_model_name,

        "description": MODEL_REGISTRY[best_model_name]["description"],

        "has_attention": best_result["has_attention"],

        "has_temporal": best_result["has_temporal"],

        "input_shape": list(INPUT_SHAPE),

        "classes": cfg.CLASSES,

        "num_classes": cfg.NUM_CLASSES,

        "threshold": cfg.THRESHOLD,

        "normalization": {"mean": MEAN, "std": STD},

        "confidence_levels": cfg.CONFIDENCE_LEVELS,

        "metrics": {k: v for k, v in best_result.items()

                    if isinstance(v, (int, float, str, bool))},

        "selected_at": datetime.now().isoformat()

    }

    

    with open(dst_dir / "config.json", 'w') as f:

        json.dump(best_config, f, indent=2)

    

    # Save class labels

    with open(dst_dir / "class_labels.json", 'w') as f:

        json.dump({"classes": cfg.CLASSES, "descriptions": cfg.CLASS_DESCRIPTIONS}, f, indent=2)

    

    print(f"  Config saved to: {dst_dir / 'config.json'}")

    print(f"  Labels saved to: {dst_dir / 'class_labels.json'}")

    

else:

    best_model_name = None

    print("⚠ No model passed validation. Manual review required.")

In [ ]:


# %%

# ===========================================================

# CELL 25: Find optimal thresholds for best model

# ===========================================================

# The default threshold of 0.5 may not be optimal for each class.

# We search for the threshold that maximizes F1 per class.

#

# WHY PER-CLASS:

# - Gunshot (distinctive) might work best at t=0.40

# - Vehicle (common, confusable) might need t=0.60

# - Using per-class thresholds improves overall performance

# ===========================================================

def find_optimal_thresholds(y_true, y_probs, classes):

    """

    Find threshold maximizing F1 for each class.

    

    Tests thresholds from 0.10 to 0.85 in steps of 0.05.

    Returns dict mapping class → {threshold, f1}.

    """

    thresholds = np.arange(0.10, 0.90, 0.05)

    optimal = {}

    

    for i, cls in enumerate(classes):

        best_t, best_f1 = 0.5, 0.0

        all_f1s = []

        

        for t in thresholds:

            preds = (y_probs[:, i] >= t).astype(int)

            f1 = f1_score(y_true[:, i], preds, zero_division=0)

            all_f1s.append(f1)

            if f1 > best_f1:

                best_f1 = f1

                best_t = float(t)

        

        optimal[cls] = {

            "threshold": round(best_t, 2),

            "f1": round(best_f1, 4),

            "all_f1s": [round(f, 4) for f in all_f1s]

        }

        print(f"  {cls:<12s}: t={best_t:.2f}, F1={best_f1:.4f}")

    

    return optimal

if best_model_name and best_model_name in all_predictions:

    preds = all_predictions[best_model_name]

    

    print(f"\nOptimal thresholds for {best_model_name}:")

    optimal_thresholds = find_optimal_thresholds(

        preds["y_true"], preds["y_probs"], cfg.CLASSES

    )

    

    # Save thresholds

    thresh_save = {cls: {"threshold": v["threshold"], "f1": v["f1"]}

                   for cls, v in optimal_thresholds.items()}

    with open(f"{cfg.BEST_MODEL_DIR}/optimal_thresholds.json", 'w') as f:

        json.dump(thresh_save, f, indent=2)

    

    # Re-evaluate with optimal thresholds

    y_pred_opt = np.zeros_like(preds["y_pred"])

    for i, cls in enumerate(cfg.CLASSES):

        t = optimal_thresholds[cls]["threshold"]

        y_pred_opt[:, i] = (preds["y_probs"][:, i] >= t).astype(int)

    

    opt_f1 = f1_score(preds["y_true"], y_pred_opt, average='macro', zero_division=0)

    fixed_f1 = all_results[best_model_name]["macro_f1"]

    

    print(f"\n  Fixed threshold (0.5) F1: {fixed_f1:.4f}")

    print(f"  Optimal thresholds F1:    {opt_f1:.4f}")

    print(f"  Improvement:              +{(opt_f1 - fixed_f1):.4f}")

    

    # Store optimal predictions

    all_predictions[best_model_name]["y_pred_optimal"] = y_pred_opt

    all_results[best_model_name]["macro_f1_optimal"] = round(opt_f1, 4)

else:

    optimal_thresholds = None

    print("⚠ No best model available for threshold optimization")

In [ ]:


# %%

# ===========================================================

# CELL 26: Save all validation reports

# ===========================================================

# Compile all validation reports into a single file.

# This is the audit trail showing every check we performed.

# ===========================================================

validation_file = Path(cfg.REPORTS_DIR) / "validation_report.txt"

with open(validation_file, 'w') as f:

    f.write("=" * 70 + "\n")

    f.write("DEFENCE SOUND CLASSIFICATION — VALIDATION REPORT\n")

    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    f.write("=" * 70 + "\n\n")

    

    pass_count = sum(1 for r in all_validation_reports if r["status"] == "PASS")

    warn_count = sum(1 for r in all_validation_reports if r["status"] == "WARNING")

    fail_count = sum(1 for r in all_validation_reports if r["status"] == "FAIL")

    

    f.write(f"Summary: {pass_count} PASS, {warn_count} WARNING, {fail_count} FAIL\n")

    f.write(f"Total checks: {len(all_validation_reports)}\n\n")

    

    for report in all_validation_reports:

        emoji = {"PASS": "✅", "WARNING": "⚠️", "FAIL": "❌"}.get(report["status"], "❓")

        f.write(f"{emoji} {report['model']} — {report['status']}\n")

        if report["issues"]:

            for issue in report["issues"]:

                f.write(f"   ⚡ {issue}\n")

        f.write("\n")

print(f"✓ Validation report saved: {validation_file}")

print(f"  Results: {pass_count} PASS, {warn_count} WARNING, {fail_count} FAIL")

In [ ]:


# %%

# ===========================================================

# CELL 27: With vs Without Attention comparison

# ===========================================================

# This is a KEY analysis for your project.

# We directly compare models WITH and WITHOUT attention

# to prove attention mechanisms add value.

# ===========================================================

print("\n" + "=" * 70)

print("  ATTENTION MECHANISM ANALYSIS")

print("=" * 70)

attention_comparison = {

    "Without Attention": [],

    "With Attention": []

}

for name, res in all_results.items():

    if res.get("status") != "OK":

        continue

    

    category = "With Attention" if res.get("has_attention") else "Without Attention"

    attention_comparison[category].append({

        "model": name,

        "macro_f1": res["macro_f1"],

        "mAP": res["mAP"],

        "exact_match": res["exact_match"],

        "hamming_loss": res["hamming_loss"],

        "params": res["params"],

        "inference_ms": res["inference_ms"]

    })

for category, models in attention_comparison.items():

    print(f"\n{category}:")

    for m in models:

        print(f"  {m['model']:<22s} F1={m['macro_f1']:.4f}  mAP={m['mAP']:.4f}  "

              f"Exact={m['exact_match']:.4f}  Params={m['params']:,}")

# Direct comparison: CNN_GRU vs CNN_GRU_SE vs CNN_GRU_CBAM vs CNN_GRU_MultiHead

compare_models = ["CNN_GRU", "CNN_GRU_SE", "CNN_GRU_CBAM", "CNN_GRU_MultiHead"]

available_models = [m for m in compare_models if m in all_results and all_results[m].get("status") == "OK"]

if len(available_models) >= 2:

    print(f"\n{'─'*70}")

    print("  Direct comparison: Same CNN+GRU base, different attention")

    print(f"{'─'*70}")

    

    base_model = "CNN_GRU"

    if base_model in all_results and all_results[base_model].get("status") == "OK":

        base_f1 = all_results[base_model]["macro_f1"]

        

        print(f"\n  {'Model':<22s} {'F1':>8} {'vs base':>10} {'mAP':>8} {'Speed':>8}")

        print(f"  {'─'*58}")

        

        for name in available_models:

            res = all_results[name]

            diff = res["macro_f1"] - base_f1

            sign = "+" if diff >= 0 else ""

            print(f"  {name:<22s} {res['macro_f1']:>8.4f} {sign}{diff:>9.4f} "

                  f"{res['mAP']:>8.4f} {res['inference_ms']:>7.1f}ms")

print(f"\n{'═'*70}")

print("  PART 2 COMPLETE: All Models Trained & Validated")

print(f"{'═'*70}")

if best_model_name:

    print(f"\n  🏆 Best model: {best_model_name}")

    print(f"     Macro F1: {all_results[best_model_name]['macro_f1']:.4f}")

    if optimal_thresholds:

        print(f"     Optimized F1: {all_results[best_model_name].get('macro_f1_optimal', 'N/A')}")


In [ ]:
# %%

# ===========================================================

# CELL 28: Visualization configuration

# ===========================================================

# Global plot styling for consistent, professional appearance.

# These settings apply to ALL plots in this notebook.

# ===========================================================

# Professional plot style

plt.style.use('seaborn-v0_8-whitegrid')

sns.set_palette("husl")

# Global matplotlib settings

plt.rcParams.update({

    'figure.figsize': (14, 6),

    'figure.dpi': 120,

    'axes.titlesize': 14,

    'axes.titleweight': 'bold',

    'axes.labelsize': 12,

    'xtick.labelsize': 10,

    'ytick.labelsize': 10,

    'legend.fontsize': 10,

    'font.family': 'sans-serif',

    'savefig.bbox': 'tight',

    'savefig.dpi': 150

})

# Model colors — consistent across all comparison plots

MODEL_COLORS = {

    "Dense_Only":        "#9E9E9E",  # Gray — baseline

    "CNN_Only":          "#FF9800",  # Orange — CNN

    "GRU_Only":          "#2196F3",  # Blue — RNN

    "CNN_LSTM":          "#9C27B0",  # Purple — CRNN

    "CNN_GRU":           "#00BCD4",  # Cyan — CRNN

    "CNN_GRU_SE":        "#4CAF50",  # Green — Attention

    "CNN_GRU_CBAM":      "#8BC34A",  # Light green — Attention

    "CNN_GRU_MultiHead": "#F44336",  # Red — Advanced

}

# Helper to save plots consistently

def save_plot(fig, subfolder, filename):

    """Save plot to the correct reports/plots subfolder."""

    path = Path(cfg.PLOTS_DIR) / subfolder / filename

    path.parent.mkdir(parents=True, exist_ok=True)

    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')

    logger.info(f"Plot saved: {path}")

# Filter to only successfully trained models

trained_models = [name for name, res in all_results.items() if res.get("status") == "OK"]

print(f"✓ Visualization config loaded. {len(trained_models)} models to visualize.")

In [ ]:


# %%

# ===========================================================

# CELL 29: VIS 1 — Dataset class distribution

# ===========================================================

# SIGNIFICANCE:

# Shows the composition of our training, validation, and test sets.

# Balanced classes are important for fair evaluation.

# The ratio of pure vs mixed clips determines task difficulty.

#

# INTERPRETATION:

# - All classes should have similar representation

# - Higher proportion of mixed clips = harder task

# - Hard mode clips should appear mainly in test set

# ===========================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, (split_name, df) in enumerate([

    ("Train", train_df), ("Val", val_df), ("Test", test_df)

]):

    # Count per class (multi-label: count each class independently)

    class_counts = {cls: int(df[cls].sum()) for cls in cfg.CLASSES}

    

    colors = [cfg.CLASS_COLORS[cls] for cls in cfg.CLASSES]

    bars = axes[idx].bar(cfg.CLASSES, [class_counts[c] for c in cfg.CLASSES],

                         color=colors, edgecolor='black', linewidth=0.8)

    

    # Value labels on bars

    for bar, cls in zip(bars, cfg.CLASSES):

        val = class_counts[cls]

        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,

                      f'{val}', ha='center', fontweight='bold', fontsize=10)

    

    # Add pie inset for pure vs mixed ratio

    num_pure = int((df["num_classes"] == 1).sum())

    num_mixed = int((df["num_classes"] > 1).sum())

    

    inset = axes[idx].inset_axes([0.65, 0.55, 0.32, 0.40])

    inset.pie([num_pure, num_mixed],

              labels=[f'Pure\n{num_pure}', f'Mixed\n{num_mixed}'],

              colors=['#E3F2FD', '#FFECB3'],

              autopct='%1.0f%%', textprops={'fontsize': 7},

              wedgeprops={'edgecolor': 'black', 'linewidth': 0.5})

    inset.set_title('Composition', fontsize=8, fontweight='bold')

    

    axes[idx].set_title(f"{split_name} Set ({len(df):,} samples)",

                        fontsize=13, fontweight='bold')

    axes[idx].set_ylabel("Positive Label Count")

    axes[idx].set_xlabel("Sound Class")

plt.suptitle("Dataset Distribution — Class Balance & Composition",

             fontsize=16, fontweight='bold', y=1.03)

plt.tight_layout()

save_plot(fig, "data_analysis", "dataset_distribution.png")

plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 30: VIS 2 — Class co-occurrence heatmap

# ===========================================================

# SIGNIFICANCE:

# Shows how often pairs of classes appear TOGETHER in the same clip.

# This reveals what multi-label combinations the model must learn.

#

# INTERPRETATION:

# - Diagonal = how often each class appears overall

# - Off-diagonal = co-occurrence (e.g., drone+vehicle often together)

# - Higher off-diagonal = more multi-label complexity

# - Symmetric matrix (drone+vehicle = vehicle+drone)

# ===========================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for idx, (split_name, df) in enumerate([

    ("Train", train_df), ("Val", val_df), ("Test", test_df)

]):

    co_matrix = np.zeros((cfg.NUM_CLASSES, cfg.NUM_CLASSES), dtype=int)

    for _, row in df.iterrows():

        present = [i for i, cls in enumerate(cfg.CLASSES) if row[cls] == 1]

        for i in present:

            for j in present:

                co_matrix[i][j] += 1

    

    sns.heatmap(co_matrix, annot=True, fmt='d', cmap='YlOrRd',

                xticklabels=cfg.CLASSES, yticklabels=cfg.CLASSES,

                ax=axes[idx], linewidths=1, linecolor='white')

    axes[idx].set_title(f"{split_name}", fontsize=13, fontweight='bold')

plt.suptitle("Class Co-occurrence Matrix — Which Classes Appear Together",

             fontsize=15, fontweight='bold', y=1.03)

plt.tight_layout()

save_plot(fig, "data_analysis", "co_occurrence.png")

plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 31: VIS 3 — Mel spectrogram examples per class

# ===========================================================

# SIGNIFICANCE:

# Shows what the CNN actually "sees" for each sound class.

# Each class has distinct spectral patterns that the model learns.

#

# INTERPRETATION:

# - Drone: Horizontal bands (constant frequency harmonics)

# - Gunshot: Vertical bright line (broadband impulse)

# - Jetplane: Wide frequency energy, gradual change

# - Vehicle: Low-frequency rumble, periodic engine patterns

# ===========================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes = axes.flatten()

for idx, cls in enumerate(cfg.CLASSES):

    cls_dir = Path(cfg.PROCESSED_DATASET) / cls

    files = list(cls_dir.glob("*.wav")) if cls_dir.exists() else []

    

    if files:

        y = audio_proc.load_audio(str(files[0]))

        if y is not None:

            y = audio_proc.pad_or_crop(y)

            mel = librosa.feature.melspectrogram(

                y=y, sr=cfg.SR, n_mels=cfg.N_MELS,

                n_fft=cfg.N_FFT, hop_length=cfg.HOP_LENGTH

            )

            mel_db = librosa.power_to_db(mel, ref=np.max)

            

            img = librosa.display.specshow(

                mel_db, sr=cfg.SR, x_axis='time', y_axis='mel',

                hop_length=cfg.HOP_LENGTH, ax=axes[idx]

            )

            fig.colorbar(img, ax=axes[idx], format='%+2.0f dB')

    

    axes[idx].set_title(f"{cls.upper()} — {cfg.CLASS_DESCRIPTIONS[cls]}",

                        fontsize=11, fontweight='bold', color=cfg.CLASS_COLORS[cls])

plt.suptitle("Mel Spectrogram Signatures — What the Model Sees",

             fontsize=15, fontweight='bold', y=1.02)

plt.tight_layout()

save_plot(fig, "data_analysis", "mel_spectrogram_examples.png")

plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 32: VIS 4 — Training loss curves — all models

# ===========================================================

# SIGNIFICANCE:

# Shows how each model's loss decreases during training.

# Faster convergence = model learns features more easily.

# Final loss level = how well model fits the data.

#

# INTERPRETATION:

# - Steep initial drop = good (learning useful features quickly)

# - Flat curve = bad (not learning or already converged)

# - Lower final loss = better fit (but watch for overfitting)

# - Models with attention should converge smoother

# ===========================================================

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

for name in trained_models:

    if name not in all_histories:

        continue

    hist = all_histories[name]

    color = MODEL_COLORS.get(name, '#333333')

    

    # Training loss

    axes[0].plot(hist['loss'], label=name, color=color, linewidth=2, alpha=0.85)

    # Validation loss

    axes[1].plot(hist['val_loss'], label=name, color=color, linewidth=2, alpha=0.85)

axes[0].set_title("Training Loss", fontsize=14, fontweight='bold')

axes[0].set_xlabel("Epoch")

axes[0].set_ylabel("Focal Loss")

axes[0].legend(fontsize=9, loc='upper right')

axes[0].grid(True, alpha=0.3)

axes[1].set_title("Validation Loss", fontsize=14, fontweight='bold')

axes[1].set_xlabel("Epoch")

axes[1].set_ylabel("Focal Loss")

axes[1].legend(fontsize=9, loc='upper right')

axes[1].grid(True, alpha=0.3)

plt.suptitle("Training Convergence — All Models",

             fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()

save_plot(fig, "training_curves", "loss_curves_all.png")

plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 33: VIS 5 — Training AUC curves — all models

# ===========================================================

# SIGNIFICANCE:

# AUC (Area Under ROC Curve) measures discrimination ability.

# Higher AUC = model better separates present from absent classes.

# Comparing train vs val AUC shows overfitting.

#

# INTERPRETATION:

# - AUC approaching 1.0 = excellent discrimination

# - AUC near 0.5 = random guessing (model not learning)

# - Gap between train and val = overfitting amount

# - Attention models should achieve higher val AUC

# ===========================================================

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

for name in trained_models:

    if name not in all_histories:

        continue

    hist = all_histories[name]

    color = MODEL_COLORS.get(name, '#333333')

    

    if 'auc' in hist:

        axes[0].plot(hist['auc'], label=name, color=color, linewidth=2, alpha=0.85)

    if 'val_auc' in hist:

        axes[1].plot(hist['val_auc'], label=name, color=color, linewidth=2, alpha=0.85)

axes[0].set_title("Training AUC", fontsize=14, fontweight='bold')

axes[0].set_xlabel("Epoch")

axes[0].set_ylabel("AUC")

axes[0].legend(fontsize=9)

axes[0].grid(True, alpha=0.3)

axes[0].set_ylim(0.5, 1.0)

axes[1].set_title("Validation AUC", fontsize=14, fontweight='bold')

axes[1].set_xlabel("Epoch")

axes[1].set_ylabel("AUC")

axes[1].legend(fontsize=9)

axes[1].grid(True, alpha=0.3)

axes[1].set_ylim(0.5, 1.0)

plt.suptitle("AUC Convergence — Model Discrimination Quality",

             fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()

save_plot(fig, "training_curves", "auc_curves_all.png")

plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 34: VIS 6 — Precision and Recall curves

# ===========================================================

# SIGNIFICANCE:

# Precision = "Of predictions marked PRESENT, how many are correct?"

# Recall = "Of truly PRESENT sounds, how many did we catch?"

#

# For DEFENCE:

# - HIGH RECALL is critical (don't miss threats → people die)

# - But low precision means constant false alarms (boy who cried wolf)

# - We need BALANCE, which is what F1 score measures

#

# INTERPRETATION:

# - Ideal: both precision and recall are high (>0.7)

# - High precision + low recall = conservative (misses threats)

# - Low precision + high recall = aggressive (too many false alarms)

# ===========================================================

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

for name in trained_models:

    if name not in all_histories:

        continue

    hist = all_histories[name]

    color = MODEL_COLORS.get(name, '#333333')

    

    if 'val_precision' in hist:

        axes[0].plot(hist['val_precision'], label=name, color=color, linewidth=2, alpha=0.85)

    if 'val_recall' in hist:

        axes[1].plot(hist['val_recall'], label=name, color=color, linewidth=2, alpha=0.85)

axes[0].set_title("Validation Precision", fontsize=14, fontweight='bold')

axes[0].set_xlabel("Epoch")

axes[0].set_ylabel("Precision")

axes[0].legend(fontsize=9)

axes[0].grid(True, alpha=0.3)

axes[1].set_title("Validation Recall", fontsize=14, fontweight='bold')

axes[1].set_xlabel("Epoch")

axes[1].set_ylabel("Recall")

axes[1].legend(fontsize=9)

axes[1].grid(True, alpha=0.3)

plt.suptitle("Precision vs Recall — Defence System Balance",

             fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()

save_plot(fig, "training_curves", "precision_recall_curves.png")

plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 35: VIS 7 — Generalization gap analysis

# ===========================================================

# SIGNIFICANCE:

# The GAP between training and validation metrics indicates

# overfitting severity. In defence, overfit models are DANGEROUS

# because they appear accurate but fail on new environments.

#

# INTERPRETATION:

# - Red shading = overfitting (train better than val)

# - Green shading = good generalization

# - Large gap = model memorized training data

# - Models with attention typically have smaller gaps

# ===========================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

axes = axes.flatten()

gap_metrics = [

    ('loss', 'val_loss', 'Loss Gap (lower=better)', False),

    ('bin_acc', 'val_bin_acc', 'Binary Accuracy Gap', True),

    ('precision', 'val_precision', 'Precision Gap', True),

    ('recall', 'val_recall', 'Recall Gap', True)

]

for idx, (train_key, val_key, title, higher_is_better) in enumerate(gap_metrics):

    for name in trained_models:

        if name not in all_histories:

            continue

        hist = all_histories[name]

        if train_key not in hist or val_key not in hist:

            continue

        

        train_vals = np.array(hist[train_key])

        val_vals = np.array(hist[val_key])

        

        # For loss: positive gap = overfitting

        # For acc/prec/rec: positive gap = overfitting

        gap = train_vals - val_vals

        

        color = MODEL_COLORS.get(name, '#333333')

        axes[idx].plot(gap, label=name, color=color, linewidth=1.5, alpha=0.8)

    

    axes[idx].axhline(y=0, color='black', linestyle='--', alpha=0.5)

    axes[idx].set_title(title, fontsize=12, fontweight='bold')

    axes[idx].set_xlabel("Epoch")

    axes[idx].set_ylabel("Train - Val")

    axes[idx].legend(fontsize=7, loc='upper right')

    axes[idx].grid(True, alpha=0.3)

plt.suptitle("Generalization Gap Analysis — Overfitting Detection\n"

             "(Closer to 0 = better generalization)",

             fontsize=15, fontweight='bold', y=1.03)

plt.tight_layout()

save_plot(fig, "training_curves", "generalization_gap.png")

plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 36: VIS 8 — Main model comparison bar chart

# ===========================================================

# SIGNIFICANCE:

# THE key comparison chart for your presentation.

# Shows all 8 models side by side on the most important metrics.

# Red border highlights the best model per metric.

#

# INTERPRETATION:

# - Taller bar = better (except hamming_loss where lower = better)

# - CNN_GRU_MultiHead or CNN_GRU_SE should dominate

# - Dense_Only should clearly be worst (proves structure matters)

# - Attention models should beat non-attention variants

# ===========================================================

metrics_to_compare = ["macro_f1", "mAP", "exact_match", "macro_precision", "macro_recall"]

n_metrics = len(metrics_to_compare)

fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 7))

for idx, metric in enumerate(metrics_to_compare):

    vals = []

    names_plot = []

    colors_plot = []

    

    for name in trained_models:

        res = all_results[name]

        if metric in res:

            vals.append(res[metric])

            names_plot.append(name)

            colors_plot.append(MODEL_COLORS.get(name, '#333333'))

    

    if not vals:

        continue

    

    bars = axes[idx].bar(range(len(vals)), vals, color=colors_plot, edgecolor='black', linewidth=0.8)

    

    # Highlight best

    best_idx = np.argmax(vals)

    bars[best_idx].set_edgecolor('#D32F2F')

    bars[best_idx].set_linewidth(3)

    

    # Value labels

    for bar, v in zip(bars, vals):

        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,

                      f'{v:.3f}', ha='center', fontweight='bold', fontsize=8, rotation=0)

    

    axes[idx].set_xticks(range(len(names_plot)))

    axes[idx].set_xticklabels(names_plot, rotation=45, ha='right', fontsize=8)

    axes[idx].set_title(metric.replace('_', ' ').title(), fontsize=12, fontweight='bold')

    axes[idx].set_ylabel("Score")

    axes[idx].set_ylim(0, min(1.0, max(vals) * 1.15))

plt.suptitle("Model Architecture Comparison — Key Metrics\n"

             "(Red border = best model for that metric)",

             fontsize=16, fontweight='bold', y=1.05)

plt.tight_layout()

save_plot(fig, "model_comparison", "main_comparison_bars.png")

plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 37: VIS 9 — Hamming loss comparison

# ===========================================================

# SIGNIFICANCE:

# Hamming loss = fraction of labels that are WRONG.

# This is the most practical error metric for multi-label.

# LOWER is BETTER (unlike F1/accuracy where higher is better).

#

# INTERPRETATION:

# - Hamming loss 0.10 = 10% of all label predictions are wrong

# - Hamming loss 0.05 = only 5% wrong (very good)

# - For 4 classes, hamming loss 0.25 = average 1 wrong per sample

# ===========================================================

fig, ax = plt.subplots(figsize=(14, 6))

h_losses = []

names_plot = []

colors_plot = []

for name in trained_models:

    res = all_results[name]

    if "hamming_loss" in res:

        h_losses.append(res["hamming_loss"])

        names_plot.append(name)

        colors_plot.append(MODEL_COLORS.get(name, '#333333'))

if h_losses:

    bars = ax.bar(names_plot, h_losses, color=colors_plot, edgecolor='black')

    

    # Highlight best (lowest)

    best_idx = np.argmin(h_losses)

    bars[best_idx].set_edgecolor('#4CAF50')

    bars[best_idx].set_linewidth(3)

    

    for bar, v in zip(bars, h_losses):

        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,

               f'{v:.4f}', ha='center', fontweight='bold', fontsize=10)

    

    ax.set_title("Hamming Loss — Fraction of Wrong Labels (LOWER = BETTER)\n"

                 "(Green border = best model)",

                 fontsize=14, fontweight='bold')

    ax.set_ylabel("Hamming Loss")

    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()

save_plot(fig, "model_comparison", "hamming_loss_comparison.png")

plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 38: VIS 10 — With vs Without Attention comparison

# ===========================================================

# SIGNIFICANCE:

# DIRECTLY compares attention vs no-attention architectures.

# This is the KEY EVIDENCE for why attention was chosen.

#

# INTERPRETATION:

# - Green bars (with attention) should be taller for F1/mAP

# - If no difference → attention adds complexity without benefit

# - If clear improvement → attention is justified for defence use

# ===========================================================

fig, axes = plt.subplots(1, 4, figsize=(20, 6))

metrics_attn = ["macro_f1", "mAP", "exact_match", "hamming_loss"]

for idx, metric in enumerate(metrics_attn):

    without = []

    with_attn = []

    

    for name in trained_models:

        res = all_results[name]

        if metric not in res:

            continue

        # Only compare CRNN variants (not Dense_Only or pure GRU/CNN)

        if res.get("has_temporal") and res.get("category") in ["crnn", "attention", "advanced"]:

            if res.get("has_attention"):

                with_attn.append(res[metric])

            else:

                without.append(res[metric])

    

    if not without or not with_attn:

        axes[idx].text(0.5, 0.5, "Insufficient data", ha='center', va='center',

                      transform=axes[idx].transAxes)

        continue

    

    avg_without = np.mean(without)

    avg_with = np.mean(with_attn)

    

    bars = axes[idx].bar(

        ["Without\nAttention", "With\nAttention"],

        [avg_without, avg_with],

        color=["#BBDEFB", "#4CAF50"],

        edgecolor='black',

        width=0.5

    )

    

    for bar, v in zip(bars, [avg_without, avg_with]):

        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,

                      f'{v:.4f}', ha='center', fontweight='bold', fontsize=11)

    

    # Show improvement

    if metric == "hamming_loss":

        diff = avg_without - avg_with

        better = "↓" if diff > 0 else "↑"

    else:

        diff = avg_with - avg_without

        better = "↑" if diff > 0 else "↓"

    

    axes[idx].set_title(f"{metric.replace('_',' ').title()}\n"

                        f"({better} {abs(diff):.4f})",

                        fontsize=12, fontweight='bold')

plt.suptitle("With vs Without Attention — CRNN Models Only\n"

             "(Direct evidence that attention improves performance)",

             fontsize=15, fontweight='bold', y=1.05)

plt.tight_layout()

save_plot(fig, "model_comparison", "attention_comparison.png")

plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 39: VIS 11 — ROC curves for best model

# ===========================================================

# SIGNIFICANCE:

# ROC curve shows the tradeoff between True Positive Rate

# (detecting real threats) and False Positive Rate (false alarms)

# at ALL possible thresholds.

#

# INTERPRETATION:

# - Diagonal = random guessing (AUC=0.5)

# - Curve hugging top-left = excellent (AUC near 1.0)

# - Each class has its own curve (some classes easier than others)

# - For defence: gunshot AUC should be highest (most distinctive)

# ===========================================================

if best_model_name and best_model_name in all_predictions:

    preds = all_predictions[best_model_name]

    

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    

    # Per-class ROC

    for i, cls in enumerate(cfg.CLASSES):

        fpr, tpr,  = roccurve(preds["y_true"][:, i], preds["y_probs"][:, i])

        roc_auc = auc(fpr, tpr)

        axes[0].plot(fpr, tpr, color=cfg.CLASS_COLORS[cls], linewidth=2.5,

                    label=f'{cls} (AUC={roc_auc:.3f})')

    

    axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random')

    axes[0].set_xlabel("False Positive Rate (False Alarms)", fontsize=12)

    axes[0].set_ylabel("True Positive Rate (Detections)", fontsize=12)

    axes[0].set_title(f"ROC Curves — {best_model_name}", fontsize=13, fontweight='bold')

    axes[0].legend(fontsize=11)

    axes[0].grid(True, alpha=0.3)

    axes[0].set_xlim([0, 1])

    axes[0].set_ylim([0, 1.02])

    

    # Per-class Precision-Recall

    for i, cls in enumerate(cfg.CLASSES):

        prec_vals, rec_vals,  = precisionrecall_curve(

            preds["y_true"][:, i], preds["y_probs"][:, i]

        )

        ap = average_precision_score(preds["y_true"][:, i], preds["y_probs"][:, i])

        axes[1].plot(rec_vals, prec_vals, color=cfg.CLASS_COLORS[cls], linewidth=2.5,

                    label=f'{cls} (AP={ap:.3f})')

    

    axes[1].set_xlabel("Recall (Threat Detection Rate)", fontsize=12)

    axes[1].set_ylabel("Precision (Prediction Correctness)", fontsize=12)

    axes[1].set_title(f"Precision-Recall Curves — {best_model_name}",

                      fontsize=13, fontweight='bold')

    axes[1].legend(fontsize=11)

    axes[1].grid(True, alpha=0.3)

    

    plt.suptitle("Detection Performance Analysis — Best Model",

                 fontsize=16, fontweight='bold', y=1.02)

    plt.tight_layout()

    save_plot(fig, "evaluation", "roc_pr_curves.png")

    plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 40: VIS 12 — Per-class confusion matrices (best model)

# ===========================================================

# SIGNIFICANCE:

# One confusion matrix per class showing:

# - TN (top-left): Correctly identified as absent

# - FP (top-right): FALSE ALARM — predicted present when absent

# - FN (bottom-left): MISSED DETECTION — predicted absent when present

# - TP (bottom-right): Correctly detected as present

#

# For DEFENCE:

# - FN (missed detection) is MOST DANGEROUS → threats slip through

# - FP (false alarm) wastes resources but doesn't cause harm

# ===========================================================

if best_model_name and best_model_name in all_predictions:

    preds = all_predictions[best_model_name]

    y_pred_final = preds.get("y_pred_optimal", preds["y_pred"])

    

    fig, axes = plt.subplots(1, cfg.NUM_CLASSES, figsize=(5 * cfg.NUM_CLASSES, 5))

    

    for i, cls in enumerate(cfg.CLASSES):

        cm = confusion_matrix(preds["y_true"][:, i], y_pred_final[:, i])

        

        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',

                    xticklabels=['Absent', 'Present'],

                    yticklabels=['Absent', 'Present'],

                    ax=axes[i], annot_kws={"size": 14},

                    linewidths=1, linecolor='white')

        

        axes[i].set_title(f"{cls.upper()}", fontsize=14, fontweight='bold',

                         color=cfg.CLASS_COLORS[cls])

        axes[i].set_xlabel("Predicted")

        axes[i].set_ylabel("Actual")

    

    plt.suptitle(f"Per-Class Confusion Matrices — {best_model_name}\n"

                 f"(FN = Missed Threat ⚠️  |  FP = False Alarm)",

                 fontsize=15, fontweight='bold', y=1.08)

    plt.tight_layout()

    save_plot(fig, "evaluation", "confusion_matrices.png")

    plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 41: VIS 13 — Prediction confidence distributions

# ===========================================================

# SIGNIFICANCE:

# Shows how the model distributes confidence scores.

# Blue = actually ABSENT, Red = actually PRESENT.

# The threshold line separates predictions.

#

# INTERPRETATION:

# GOOD: Two well-separated peaks (blue near 0, red near 1)

# BAD: Overlapping peaks around 0.5 (model is uncertain)

# For defence: we want clear separation → confident decisions

# ===========================================================

if best_model_name and best_model_name in all_predictions:

    preds = all_predictions[best_model_name]

    

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    axes = axes.flatten()

    

    for i, cls in enumerate(cfg.CLASSES):

        pos = preds["y_probs"][preds["y_true"][:, i] == 1, i]

        neg = preds["y_probs"][preds["y_true"][:, i] == 0, i]

        

        axes[i].hist(neg, bins=50, alpha=0.6, label=f'Absent (n={len(neg)})',

                    color='#42A5F5', edgecolor='black', linewidth=0.5)

        axes[i].hist(pos, bins=50, alpha=0.6, label=f'Present (n={len(pos)})',

                    color='#EF5350', edgecolor='black', linewidth=0.5)

        

        # Threshold line

        t = optimal_thresholds[cls]["threshold"] if optimal_thresholds else cfg.THRESHOLD

        axes[i].axvline(t, color='black', linestyle='--', linewidth=2,

                       label=f'Threshold={t:.2f}')

        

        # Confidence level shading

        for level, thresh in sorted(cfg.CONFIDENCE_LEVELS.items(), key=lambda x: x[1]):

            color = confidence_analyzer.get_confidence_color(level)

            axes[i].axvspan(thresh, min(thresh + 0.15, 1.0), alpha=0.05, color=color)

        

        axes[i].set_title(f"{cls.upper()}", fontsize=13, fontweight='bold',

                         color=cfg.CLASS_COLORS[cls])

        axes[i].set_xlabel("Predicted Probability")

        axes[i].set_ylabel("Count")

        axes[i].legend(fontsize=9)

    

    plt.suptitle(f"Prediction Confidence Distribution — {best_model_name}\n"

                 f"(Good: blue peak left, red peak right, clear separation)",

                 fontsize=15, fontweight='bold', y=1.03)

    plt.tight_layout()

    save_plot(fig, "evaluation", "confidence_distributions.png")

    plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 42: VIS 14 — Threshold optimization curves

# ===========================================================

# SIGNIFICANCE:

# Shows F1 at every possible threshold for each class.

# The PEAK of each curve is the optimal threshold.

#

# INTERPRETATION:

# - Sharp peak = threshold-sensitive (must be precise)

# - Flat top = threshold-robust (any value in range works)

# - Different peak positions = classes need different thresholds

# ===========================================================

if best_model_name and best_model_name in all_predictions and optimal_thresholds:

    preds = all_predictions[best_model_name]

    thresholds_range = np.arange(0.05, 0.95, 0.02)

    

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    axes = axes.flatten()

    

    for i, cls in enumerate(cfg.CLASSES):

        f1_scores = []

        prec_scores = []

        rec_scores = []

        

        for t in thresholds_range:

            p = (preds["y_probs"][:, i] >= t).astype(int)

            f1_scores.append(f1_score(preds["y_true"][:, i], p, zero_division=0))

            prec_scores.append(precision_score(preds["y_true"][:, i], p, zero_division=0))

            rec_scores.append(recall_score(preds["y_true"][:, i], p, zero_division=0))

        

        axes[i].plot(thresholds_range, f1_scores, color=cfg.CLASS_COLORS[cls],

                    linewidth=2.5, label='F1 Score')

        axes[i].plot(thresholds_range, prec_scores, color='gray', linewidth=1.5,

                    linestyle='--', alpha=0.7, label='Precision')

        axes[i].plot(thresholds_range, rec_scores, color='gray', linewidth=1.5,

                    linestyle=':', alpha=0.7, label='Recall')

        

        opt_t = optimal_thresholds[cls]["threshold"]

        opt_f1 = optimal_thresholds[cls]["f1"]

        axes[i].axvline(opt_t, color='red', linewidth=2, linestyle='-.',

                       label=f'Optimal t={opt_t:.2f}')

        axes[i].scatter([opt_t], [opt_f1], color='red', s=120, zorder=5, edgecolor='black')

        

        axes[i].set_title(f"{cls.upper()}", fontsize=13, fontweight='bold',

                         color=cfg.CLASS_COLORS[cls])

        axes[i].set_xlabel("Threshold")

        axes[i].set_ylabel("Score")

        axes[i].legend(fontsize=9)

        axes[i].grid(True, alpha=0.3)

        axes[i].set_ylim(0, 1.05)

    

    plt.suptitle("Threshold Optimization — F1 vs Threshold per Class\n"

                 "(Red dot = selected optimal threshold)",

                 fontsize=15, fontweight='bold', y=1.03)

    plt.tight_layout()

    save_plot(fig, "evaluation", "threshold_optimization.png")

    plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 43: VIS 15 — Confidence calibration plot

# ===========================================================

# SIGNIFICANCE:

# Shows if model confidence matches actual accuracy.

# A CALIBRATED model: "80% confident" → correct 80% of the time.

#

# INTERPRETATION:

# - Points on diagonal = perfectly calibrated

# - Points above diagonal = model is UNDERconfident

# - Points below diagonal = model is OVERconfident (dangerous!)

# - ECE (Expected Calibration Error) < 0.05 is good

# - For defence: overconfidence is DANGEROUS (false sense of security)

# ===========================================================

if best_model_name and best_model_name in all_predictions:

    preds = all_predictions[best_model_name]

    calibration = confidence_analyzer.calibration_analysis(

        preds["y_true"], preds["y_probs"], n_bins=10

    )

    

    fig, axes = plt.subplots(1, cfg.NUM_CLASSES, figsize=(5 * cfg.NUM_CLASSES, 5))

    

    for i, cls in enumerate(cfg.CLASSES):

        cal = calibration[cls]

        

        # Reliability diagram

        axes[i].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Perfect')

        

        if cal["bin_confidences"] and cal["bin_accuracies"]:

            axes[i].bar(cal["bin_confidences"], cal["bin_accuracies"],

                       width=0.08, alpha=0.6, color=cfg.CLASS_COLORS[cls],

                       edgecolor='black', label=f'ECE={cal["ece"]:.4f}')

            axes[i].scatter(cal["bin_confidences"], cal["bin_accuracies"],

                          color=cfg.CLASS_COLORS[cls], s=60, zorder=5, edgecolor='black')

        

        axes[i].set_title(f"{cls.upper()}\nECE={cal['ece']:.4f}",

                         fontsize=12, fontweight='bold', color=cfg.CLASS_COLORS[cls])

        axes[i].set_xlabel("Mean Predicted Probability")

        axes[i].set_ylabel("Fraction of Positives")

        axes[i].legend(fontsize=9)

        axes[i].set_xlim(0, 1)

        axes[i].set_ylim(0, 1)

        axes[i].grid(True, alpha=0.3)

    

    plt.suptitle(f"Confidence Calibration — {best_model_name}\n"

                 f"(Bars on diagonal = well calibrated; ECE < 0.05 = good)",

                 fontsize=15, fontweight='bold', y=1.08)

    plt.tight_layout()

    save_plot(fig, "evaluation", "calibration_plot.png")

    plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 44: VIS 16 — Performance by number of classes

# ===========================================================

# SIGNIFICANCE:

# Shows how model degrades as more classes overlap.

# This is THE KEY METRIC for mixed-sound detection.

#

# INTERPRETATION:

# - 1 class: easiest (single event, clear signal)

# - 2 classes: moderate (two sounds overlapping)

# - 3-4 classes: hardest (multiple threats simultaneously)

# - Graceful degradation = good (F1 drops slowly)

# - Cliff drop = bad (model can't handle complexity)

# ===========================================================

if best_model_name and best_model_name in all_predictions:

    preds = all_predictions[best_model_name]

    y_pred_final = preds.get("y_pred_optimal", preds["y_pred"])

    

    num_classes_range = sorted(test_df["num_classes"].unique())

    

    f1_by_nc = []

    exact_by_nc = []

    counts_nc = []

    

    for nc in num_classes_range:

        mask = (test_df["num_classes"] == nc).values[:len(preds["y_true"])]

        if mask.sum() == 0:

            continue

        

        sub_true = preds["y_true"][mask]

        sub_pred = y_pred_final[mask]

        

        f1_by_nc.append(f1_score(sub_true, sub_pred, average='macro', zero_division=0))

        exact_by_nc.append(np.all(sub_pred == sub_true, axis=1).mean())

        counts_nc.append(int(mask.sum()))

    

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    

    complexity_colors = ['#66BB6A', '#42A5F5', '#FFA726', '#EF5350']

    

    # F1 by complexity

    bars = axes[0].bar(num_classes_range[:len(f1_by_nc)], f1_by_nc,

                       color=complexity_colors[:len(f1_by_nc)], edgecolor='black')

    for bar, v in zip(bars, f1_by_nc):

        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,

                    f'{v:.3f}', ha='center', fontweight='bold', fontsize=11)

    axes[0].set_title("Macro F1 by Complexity", fontsize=13, fontweight='bold')

    axes[0].set_xlabel("Number of Overlapping Classes")

    axes[0].set_ylabel("Macro F1")

    axes[0].set_ylim(0, 1)

    

    # Exact match by complexity

    bars = axes[1].bar(num_classes_range[:len(exact_by_nc)], exact_by_nc,

                       color=complexity_colors[:len(exact_by_nc)], edgecolor='black')

    for bar, v in zip(bars, exact_by_nc):

        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,

                    f'{v:.3f}', ha='center', fontweight='bold', fontsize=11)

    axes[1].set_title("Exact Match by Complexity", fontsize=13, fontweight='bold')

    axes[1].set_xlabel("Number of Overlapping Classes")

    axes[1].set_ylabel("Exact Match Ratio")

    axes[1].set_ylim(0, 1)

    

    # Sample counts

    bars = axes[2].bar(num_classes_range[:len(counts_nc)], counts_nc,

                       color=complexity_colors[:len(counts_nc)], edgecolor='black')

    for bar, v in zip(bars, counts_nc):

        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,

                    str(v), ha='center', fontweight='bold', fontsize=11)

    axes[2].set_title("Test Samples per Complexity", fontsize=13, fontweight='bold')

    axes[2].set_xlabel("Number of Overlapping Classes")

    axes[2].set_ylabel("Count")

    

    plt.suptitle(f"Performance vs Class Overlap Complexity — {best_model_name}\n"

                 f"(Key metric: can the model handle multiple simultaneous threats?)",

                 fontsize=15, fontweight='bold', y=1.05)

    plt.tight_layout()

    save_plot(fig, "evaluation", "performance_by_complexity.png")

    plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 45: VIS 17 — Hard vs Normal performance

# ===========================================================

# SIGNIFICANCE:

# Compares performance on clean vs degraded recordings.

# Hard samples have: weak signals, heavy noise, channel effects.

#

# INTERPRETATION:

# - Small gap = robust model (good for real deployment)

# - Large gap = fragile model (only works in clean conditions)

# - For defence: robustness is NON-NEGOTIABLE

# ===========================================================

if best_model_name and best_model_name in all_predictions and "hard_mode" in test_df.columns:

    preds = all_predictions[best_model_name]

    y_pred_final = preds.get("y_pred_optimal", preds["y_pred"])

    

    categories = {"Normal": 0, "Hard": 1}

    cat_results = {}

    

    for label, hm in categories.items():

        mask = (test_df["hard_mode"] == hm).values[:len(preds["y_true"])]

        if mask.sum() > 0:

            sub_true = preds["y_true"][mask]

            sub_pred = y_pred_final[mask]

            

            cat_results[label] = {

                "F1": f1_score(sub_true, sub_pred, average='macro', zero_division=0),

                "Exact": np.all(sub_pred == sub_true, axis=1).mean(),

                "Hamming": hamming_loss(sub_true, sub_pred),

                "Count": int(mask.sum())

            }

    

    if cat_results:

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        

        for idx, metric in enumerate(["F1", "Exact", "Hamming"]):

            vals = [cat_results[k][metric] for k in cat_results]

            bars = axes[idx].bar(list(cat_results.keys()), vals,

                                color=['#66BB6A', '#EF5350'], edgecolor='black', width=0.5)

            for bar, v in zip(bars, vals):

                axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,

                              f'{v:.4f}', ha='center', fontweight='bold', fontsize=12)

            axes[idx].set_title(f"{'Macro ' if metric=='F1' else ''}{metric}",

                               fontsize=13, fontweight='bold')

            if metric != "Hamming":

                axes[idx].set_ylim(0, 1)

        

        plt.suptitle(f"Robustness Test — {best_model_name}\n"

                     f"(Gap shows how model handles degraded conditions)",

                     fontsize=15, fontweight='bold', y=1.05)

        plt.tight_layout()

        save_plot(fig, "evaluation", "hard_vs_normal.png")

        plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 46: VIS 18 — Radar chart — all models

# ===========================================================

# SIGNIFICANCE:

# Overlay radar charts show each model's strengths/weaknesses.

# Larger polygon = better overall model.

#

# INTERPRETATION:

# - Most balanced (largest) polygon = best overall model

# - Irregular shape = model excels at some metrics but fails others

# - Ideal: large regular polygon touching outer edges

# ===========================================================

radar_metrics = ["macro_f1", "mAP", "exact_match", "macro_precision", "macro_recall"]

angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()

angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for name in trained_models:

    res = all_results[name]

    vals = [res.get(m, 0) for m in radar_metrics]

    vals += vals[:1]

    color = MODEL_COLORS.get(name, '#333333')

    

    ax.plot(angles, vals, linewidth=2, label=name, color=color, alpha=0.8)

    ax.fill(angles, vals, alpha=0.05, color=color)

ax.set_xticks(angles[:-1])

ax.set_xticklabels([m.replace('_', '\n').title() for m in radar_metrics], fontsize=10)

ax.set_ylim(0, 1)

ax.set_title("Model Performance Radar — All Architectures\n"

             "(Larger area = better overall)",

             fontsize=14, fontweight='bold', pad=30)

ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=9)

ax.grid(True, alpha=0.4)

plt.tight_layout()

save_plot(fig, "model_comparison", "radar_all_models.png")

plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 47: VIS 19 — Efficiency bubble chart

# ===========================================================

# SIGNIFICANCE:

# Shows the TRADEOFF between model size, speed, and accuracy.

# Ideal: small (few params), fast (low ms), accurate (high F1).

#

# INTERPRETATION:

# - Position: x=size, y=performance (top-left is ideal)

# - Bubble size: inference latency (smaller = faster)

# - Best model balances all three dimensions

# ===========================================================

fig, ax = plt.subplots(figsize=(14, 9))

for name in trained_models:

    res = all_results[name]

    color = MODEL_COLORS.get(name, '#333333')

    

    params_m = res["params"] / 1e6

    f1 = res["macro_f1"]

    speed = res["inference_ms"]

    

    size = max(speed * 15, 100)

    

    ax.scatter(params_m, f1, s=size, c=color, edgecolor='black',

              linewidth=2, alpha=0.8, zorder=5)

    

    ax.annotate(f"{name}\n{speed:.0f}ms",

               (params_m, f1),

               textcoords="offset points",

               xytext=(12, 8),

               fontsize=9, fontweight='bold',

               arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

ax.set_xlabel("Parameters (Millions)", fontsize=13)

ax.set_ylabel("Macro F1 Score", fontsize=13)

ax.set_title("Model Efficiency — Size vs Performance vs Speed\n"

             "(Bubble size = inference time; smaller = faster)",

             fontsize=15, fontweight='bold')

ax.grid(True, alpha=0.3)

plt.tight_layout()

save_plot(fig, "model_comparison", "efficiency_bubble.png")

plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 48: VIS 20 — Error analysis

# ===========================================================

# SIGNIFICANCE:

# Identifies the most common types of errors.

# "Missed X" = failed to detect class X (FALSE NEGATIVE)

# "False X" = incorrectly predicted class X (FALSE POSITIVE)

#

# INTERPRETATION:

# - Most common error type = systematic weakness

# - "Missed gunshot" = most dangerous error for defence

# - Use this to decide: add more training data for weak class

# ===========================================================

if best_model_name and best_model_name in all_predictions:

    preds = all_predictions[best_model_name]

    y_pred_final = preds.get("y_pred_optimal", preds["y_pred"])

    

    error_types = defaultdict(int)

    

    for i in range(len(preds["y_true"])):

        for j, cls in enumerate(cfg.CLASSES):

            if preds["y_true"][i, j]  1 and y_pred_final[i, j]  0:

                error_types[f"⚠ Missed {cls}"] += 1

            elif preds["y_true"][i, j]  0 and y_pred_final[i, j]  1:

                error_types[f"🔔 False {cls}"] += 1

    

    sorted_errors = sorted(error_types.items(), key=lambda x: x[1], reverse=True)

    

    if sorted_errors:

        labels, counts = zip(*sorted_errors)

        

        fig, ax = plt.subplots(figsize=(14, 6))

        

        colors_err = ['#EF5350' if 'Missed' in l else '#FF9800' for l in labels]

        bars = ax.barh(labels, counts, color=colors_err, edgecolor='black')

        

        for bar, count in zip(bars, counts):

            ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,

                   str(count), va='center', fontweight='bold', fontsize=11)

        

        ax.set_xlabel("Number of Errors", fontsize=12)

        ax.set_title(f"Error Analysis — {best_model_name}\n"

                     f"(Red = Missed Detection ⚠️ | Orange = False Alarm 🔔)",

                     fontsize=14, fontweight='bold')

        ax.invert_yaxis()

        ax.grid(True, alpha=0.3, axis='x')

        

        plt.tight_layout()

        save_plot(fig, "evaluation", "error_analysis.png")

        plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 49: VIS 21 — Per-class F1 radar for best model

# ===========================================================

# SIGNIFICANCE:

# Zoomed radar showing F1 for each class individually.

# Immediately shows which threats the model detects best/worst.

#

# INTERPRETATION:

# - Larger polygon = more consistent across classes

# - Dent in one direction = weak on that class

# - For defence: ALL classes should be reasonably high (>0.6)

# ===========================================================

if best_model_name and best_model_name in all_predictions:

    preds = all_predictions[best_model_name]

    y_pred_final = preds.get("y_pred_optimal", preds["y_pred"])

    

    per_class_f1 = []

    for i, cls in enumerate(cfg.CLASSES):

        f1 = f1_score(preds["y_true"][:, i], y_pred_final[:, i], zero_division=0)

        per_class_f1.append(f1)

    

    angles = np.linspace(0, 2 * np.pi, cfg.NUM_CLASSES, endpoint=False).tolist()

    values = per_class_f1 + [per_class_f1[0]]

    angles += angles[:1]

    

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    

    ax.fill(angles, values, color='#42A5F5', alpha=0.25)

    ax.plot(angles, values, color='#1565C0', linewidth=2.5)

    ax.scatter(angles[:-1], per_class_f1,

              color=[cfg.CLASS_COLORS[c] for c in cfg.CLASSES],

              s=120, zorder=5, edgecolor='black')

    

    ax.set_xticks(angles[:-1])

    ax.set_xticklabels([f"{cls}\nF1={f1:.3f}" for cls, f1 in zip(cfg.CLASSES, per_class_f1)],

                       fontsize=12, fontweight='bold')

    ax.set_ylim(0, 1)

    ax.set_title(f"Per-Class F1 — {best_model_name}\n(Larger = better detection per class)",

                 fontsize=14, fontweight='bold', pad=25)

    ax.grid(True, alpha=0.4)

    

    plt.tight_layout()

    save_plot(fig, "evaluation", "per_class_f1_radar.png")

    plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 50: VIS 22 — Confidence level distribution

# ===========================================================

# SIGNIFICANCE:

# Shows how predictions distribute across confidence levels.

# This is what the defence operator dashboard would show.

#

# INTERPRETATION:

# - Mostly NEGLIGIBLE for absent classes = good

# - Mostly CRITICAL/HIGH for present classes = good

# - Lots of MEDIUM = model is uncertain (needs improvement)

# ===========================================================

if best_model_name and best_model_name in all_predictions:

    preds = all_predictions[best_model_name]

    

    fig, axes = plt.subplots(1, cfg.NUM_CLASSES, figsize=(5 * cfg.NUM_CLASSES, 6))

    

    level_names = list(cfg.CONFIDENCE_LEVELS.keys())

    

    for i, cls in enumerate(cfg.CLASSES):

        # Count predictions in each confidence level

        level_counts = {level: 0 for level in level_names}

        

        for prob in preds["y_probs"][:, i]:

            level = confidence_analyzer.get_confidence_level(float(prob))

            level_counts[level] += 1

        

        colors_conf = [confidence_analyzer.get_confidence_color(l) for l in level_names]

        counts = [level_counts[l] for l in level_names]

        

        bars = axes[i].bar(level_names, counts, color=colors_conf, edgecolor='black')

        

        for bar, c in zip(bars, counts):

            if c > 0:

                axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,

                            str(c), ha='center', fontsize=9, fontweight='bold')

        

        axes[i].set_title(f"{cls.upper()}", fontsize=13, fontweight='bold',

                         color=cfg.CLASS_COLORS[cls])

        axes[i].tick_params(axis='x', rotation=45)

        axes[i].set_ylabel("Count")

    

    plt.suptitle(f"Confidence Level Distribution — {best_model_name}\n"

                 f"(CRITICAL=highest alert | NEGLIGIBLE=safe)",

                 fontsize=15, fontweight='bold', y=1.05)

    plt.tight_layout()

    save_plot(fig, "production_dashboard", "confidence_levels.png")

    plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 51: VIS 23 — Complete metrics comparison table

# ===========================================================

# SIGNIFICANCE:

# Fixed threshold (0.5) vs optimized per-class thresholds.

# Shows how much improvement threshold tuning provides.

#

# INTERPRETATION:

# - Optimal should always be >= fixed

# - Large improvement = default threshold was suboptimal

# - Small improvement = model is well-calibrated

# ===========================================================

if best_model_name and best_model_name in all_results:

    res = all_results[best_model_name]

    

    metrics_display = {

        "Macro F1 (fixed)": res.get("macro_f1", 0),

        "Macro F1 (optimal)": res.get("macro_f1_optimal", res.get("macro_f1", 0)),

        "mAP": res.get("mAP", 0),

        "Exact Match": res.get("exact_match", 0),

        "Hamming Loss": res.get("hamming_loss", 0),

        "Precision": res.get("macro_precision", 0),

        "Recall": res.get("macro_recall", 0),

    }

    

    fig, ax = plt.subplots(figsize=(12, 6))

    

    names_m = list(metrics_display.keys())

    vals_m = list(metrics_display.values())

    

    colors_m = ['#42A5F5', '#4CAF50', '#FF9800', '#9C27B0',

                '#EF5350', '#00BCD4', '#FFC107']

    

    bars = ax.bar(names_m, vals_m, color=colors_m[:len(names_m)], edgecolor='black')

    

    for bar, v in zip(bars, vals_m):

        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,

               f'{v:.4f}', ha='center', fontweight='bold', fontsize=11)

    

    ax.set_title(f"Complete Metrics — {best_model_name}\n"

                 f"(Best model with fixed and optimized thresholds)",

                 fontsize=14, fontweight='bold')

    ax.set_ylabel("Score")

    ax.tick_params(axis='x', rotation=20)

    ax.set_ylim(0, 1.1)

    

    plt.tight_layout()

    save_plot(fig, "evaluation", "best_model_all_metrics.png")

    plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 52: VIS 24 — Production dashboard summary

# ===========================================================

# SIGNIFICANCE:

# Single-page dashboard summarizing the ENTIRE project.

# This would be displayed in a defence operations center.

#

# INTERPRETATION:

# - Top left: best model identity and key metrics

# - Top right: per-class detection performance

# - Bottom left: model comparison summary

# - Bottom right: confidence level guidelines

# ===========================================================

fig = plt.figure(figsize=(22, 14))

gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)

# ── Top Left: System summary ──

ax1 = fig.add_subplot(gs[0, 0])

ax1.axis('off')

if best_model_name:

    res = all_results[best_model_name]

    summary_text = (

        f"DEFENCE SOUND CLASSIFICATION SYSTEM\n"

        f"{'─' * 45}\n\n"

        f"Best Model:     {best_model_name}\n"

        f"Attention:      {'Yes' if res.get('has_attention') else 'No'}\n"

        f"Parameters:     {res.get('params', 0):,}\n"

        f"Inference:      {res.get('inference_ms', 0):.1f} ms/sample\n\n"

        f"{'─' * 45}\n"

        f"Macro F1:       {res.get('macro_f1', 0):.4f}\n"

        f"mAP:            {res.get('mAP', 0):.4f}\n"

        f"Exact Match:    {res.get('exact_match', 0):.4f}\n"

        f"Hamming Loss:   {res.get('hamming_loss', 0):.4f}\n"

        f"Precision:      {res.get('macro_precision', 0):.4f}\n"

        f"Recall:         {res.get('macro_recall', 0):.4f}\n\n"

        f"{'─' * 45}\n"

        f"Classes:        {', '.join(cfg.CLASSES)}\n"

        f"Audio:          {cfg.SR}Hz, {cfg.DURATION}s clips\n"

        f"Features:       {cfg.N_MELS}-band log-mel spectrogram"

    )

    ax1.text(0.05, 0.95, summary_text, transform=ax1.transAxes,

            fontsize=11, fontfamily='monospace', verticalalignment='top',

            bbox=dict(boxstyle='round', facecolor='#E3F2FD', alpha=0.8))

    ax1.set_title("System Overview", fontsize=14, fontweight='bold')

# ── Top Right: Per-class F1 bars ──

ax2 = fig.add_subplot(gs[0, 1])

if best_model_name and best_model_name in all_predictions:

    preds_best = all_predictions[best_model_name]

    y_pred_use = preds_best.get("y_pred_optimal", preds_best["y_pred"])

    

    class_f1s = []

    for i, cls in enumerate(cfg.CLASSES):

        f1 = f1_score(preds_best["y_true"][:, i], y_pred_use[:, i], zero_division=0)

        class_f1s.append(f1)

    

    colors_cls = [cfg.CLASS_COLORS[c] for c in cfg.CLASSES]

    bars = ax2.bar(cfg.CLASSES, class_f1s, color=colors_cls, edgecolor='black')

    for bar, v in zip(bars, class_f1s):

        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,

                f'{v:.3f}', ha='center', fontweight='bold', fontsize=12)

    ax2.set_title("Per-Class Detection F1", fontsize=14, fontweight='bold')

    ax2.set_ylim(0, 1.1)

    ax2.set_ylabel("F1 Score")

# ── Bottom Left: Model ranking ──

ax3 = fig.add_subplot(gs[1, 0])

ax3.axis('off')

ranking_text = f"MODEL RANKING (by Macro F1)\n{'─' * 50}\n\n"

sorted_models = sorted(

    [(n, r) for n, r in all_results.items() if r.get("status") == "OK"],

    key=lambda x: -x[1].get("macro_f1", 0)

)

for rank, (name, res) in enumerate(sorted_models, 1):

    medal = "🥇" if rank  1 else ("🥈" if rank  2 else ("🥉" if rank == 3 else f"#{rank}"))

    attn = "✓" if res.get("has_attention") else "✗"

    ranking_text += (

        f"{medal} {name:<22s} F1={res.get('macro_f1',0):.4f} "

        f"mAP={res.get('mAP',0):.4f} Attn={attn}\n"

    )

ax3.text(0.05, 0.95, ranking_text, transform=ax3.transAxes,

        fontsize=10, fontfamily='monospace', verticalalignment='top',

        bbox=dict(boxstyle='round', facecolor='#FFF3E0', alpha=0.8))

ax3.set_title("Model Ranking", fontsize=14, fontweight='bold')

# ── Bottom Right: Confidence levels ──

ax4 = fig.add_subplot(gs[1, 1])

ax4.axis('off')

conf_text = f"CONFIDENCE LEVEL GUIDE\n{'─' * 50}\n\n"

for level, thresh in sorted(cfg.CONFIDENCE_LEVELS.items(),

                             key=lambda x: -x[1]):

    color = confidence_analyzer.get_confidence_color(level)

    actions = {

        "CRITICAL":   "Deploy countermeasures immediately",

        "HIGH":       "Alert operators, verify with secondary",

        "MEDIUM":     "Flag for human review",

        "LOW":        "Background monitoring only",

        "NEGLIGIBLE": "No action required"

    }

    conf_text += f"  {level:<12s} (≥{thresh:.2f}): {actions.get(level, '')}\n"

conf_text += (

    f"\n{'─' * 50}\n"

    f"Optimal Thresholds:\n"

)

if optimal_thresholds:

    for cls, vals in optimal_thresholds.items():

        conf_text += f"  {cls:<12s}: t={vals['threshold']:.2f} (F1={vals['f1']:.4f})\n"

ax4.text(0.05, 0.95, conf_text, transform=ax4.transAxes,

        fontsize=10, fontfamily='monospace', verticalalignment='top',

        bbox=dict(boxstyle='round', facecolor='#E8F5E9', alpha=0.8))

ax4.set_title("Operational Guidelines", fontsize=14, fontweight='bold')

plt.suptitle("DEFENCE SOUND CLASSIFICATION — PRODUCTION DASHBOARD",

             fontsize=18, fontweight='bold', y=1.01)

save_plot(fig, "production_dashboard", "full_dashboard.png")

plt.show()

In [ ]:


# %%

# ===========================================================

# CELL 53: VIS 25 — Final comparison: attention analysis table

# ===========================================================

# SIGNIFICANCE:

# Produces the definitive comparison for the research question:

# "Does attention improve defence sound classification?"

#

# This table goes directly into your project report.

# ===========================================================

print("\n" + "═" * 80)

print("  ATTENTION MECHANISM ANALYSIS — FINAL RESULTS")

print("═" * 80)

print(f"\n  {'Model':<22s} {'Attention':<10s} {'F1':>7s} {'mAP':>7s} {'Exact':>7s} "

      f"{'Hamming':>8s} {'Params':>10s} {'Speed':>8s}")

print(f"  {'─'*80}")

for name in trained_models:

    res = all_results[name]

    attn = "✓ Yes" if res.get("has_attention") else "✗ No"

    marker = " ★" if name == best_model_name else ""

    

    print(f"  {name:<22s} {attn:<10s} {res.get('macro_f1',0):>7.4f} "

          f"{res.get('mAP',0):>7.4f} {res.get('exact_match',0):>7.4f} "

          f"{res.get('hamming_loss',0):>8.4f} {res.get('params',0):>10,} "

          f"{res.get('inference_ms',0):>7.1f}ms{marker}")

print(f"\n  ★ = Selected best model")

# Compute average improvement from attention

no_attn_f1 = [all_results[n]["macro_f1"] for n in trained_models

              if not all_results[n].get("has_attention") and

              all_results[n].get("has_temporal") and

              all_results[n].get("category") in ["crnn"]]

with_attn_f1 = [all_results[n]["macro_f1"] for n in trained_models

                if all_results[n].get("has_attention")]

if no_attn_f1 and with_attn_f1:

    avg_no = np.mean(no_attn_f1)

    avg_with = np.mean(with_attn_f1)

    improvement = avg_with - avg_no

    

    print(f"\n  CRNN without attention — avg F1: {avg_no:.4f}")

    print(f"  CRNN with attention    — avg F1: {avg_with:.4f}")

    print(f"  Average improvement:             +{improvement:.4f} ({improvement/avg_no*100:.1f}%)")

    

    if improvement > 0:

        print(f"\n  CONCLUSION: Attention mechanisms IMPROVE defence sound classification")

        print(f"  by +{improvement:.4f} F1 on average ({improvement/avg_no*100:.1f}% relative improvement)")

    else:

        print(f"\n  CONCLUSION: Attention did NOT improve performance in this experiment.")

        print(f"  Consider: more training data, different attention placement, or tuning.")

In [ ]:

# %%

# ===========================================================

# CELL 54: Save all plots list

# ===========================================================

all_plots = list(Path(cfg.PLOTS_DIR).rglob("*.png"))

print(f"\n✓ Total plots generated: {len(all_plots)}")

print(f"\nPlots saved to: {cfg.PLOTS_DIR}/")

for p in sorted(all_plots):

    print(f"  {p.relative_to(Path(cfg.PLOTS_DIR))}")

print(f"\n{'═'*60}")

print(f"  PART 3 COMPLETE: All Visualizations Generated")

print(f"{'═'*60}")

In [ ]:
# %%

# ===========================================================

# CELL 55: Generate extreme stress test samples

# ===========================================================

# WHY STRESS TESTING:

# A defence system must work in the WORST conditions:

# - Very distant/quiet sources (gain 0.03-0.30)

# - Extremely noisy environments (SNR -8 to +2 dB)

# - Degraded recording equipment (heavy filtering, clipping)

# - Multiple overlapping threats (3-4 classes simultaneously)

# - Very short events (0.2-1.0 seconds)

#

# If the model fails here, it CANNOT be deployed.

# A stress test F1 of 0.40-0.60 is REALISTIC and ACCEPTABLE.

# A stress test F1 of 0.80+ would be suspicious.

# ===========================================================

def generate_stress_test(class_files, noise_files, output_dir, num_samples=300):

    """

    Generate extremely challenging audio test samples.

    

    These simulate the worst-case operational scenarios:

    - Target sounds are very quiet (distant source)

    - Background noise is loud and persistent

    - Recording equipment is degraded

    - Multiple threats occur simultaneously

    - Events are brief and easy to miss

    

    Args:

        class_files: Dict mapping class → list of source file paths

        noise_files: List of background noise file paths

        output_dir: Where to save generated stress test audio

        num_samples: How many stress samples to generate

    

    Returns:

        DataFrame with filepath and multi-label metadata

    """

    output_dir = Path(output_dir)

    audio_dir = output_dir / "audio"

    audio_dir.mkdir(parents=True, exist_ok=True)

    

    rows = []

    

    for i in tqdm(range(num_samples), desc="Generating stress test"):

        # Favor complex scenarios (3-4 classes)

        n_cls = random.choices([1, 2, 3, 4], weights=[0.05, 0.20, 0.40, 0.35])[0]

        chosen = random.sample(cfg.CLASSES, n_cls)

        

        mix = np.zeros(cfg.SAMPLES, dtype=np.float32)

        labels = {cls: 0 for cls in cfg.CLASSES}

        

        for cls in chosen:

            if not class_files.get(cls):

                continue

            y = audio_proc.load_audio(random.choice(class_files[cls]))

            if y is None:

                continue

            

            # Very short events (0.2 to 1.0 seconds)

            ev_len = random.randint(int(0.2  cfg.SR), int(1.0  cfg.SR))

            ev_len = min(ev_len, len(y), cfg.SAMPLES)

            y = audio_proc.pad_or_crop(y, ev_len)

            

            # Very weak signal (simulates distance)

            y,  = audioproc.random_gain(y, 0.03, 0.30)

            

            # Random placement

            placed,  = audioproc.place_randomly(y)

            mix += placed

            labels[cls] = 1

        

        # ── HEAVY NOISE (mandatory) ──

        if noise_files:

            nf = audio_proc.load_audio(str(random.choice(noise_files)))

            if nf is not None:

                nf = audio_proc.fit_noise_to_length(nf, cfg.SAMPLES)

                snr_db = random.uniform(-8, 2)  # Very low SNR

                mix = audio_proc.add_noise_at_snr(mix, nf, snr_db)

            else:

                gauss = np.random.randn(cfg.SAMPLES).astype(np.float32)

                mix = audio_proc.add_noise_at_snr(mix, gauss, random.uniform(-6, 3))

        else:

            gauss = np.random.randn(cfg.SAMPLES).astype(np.float32)

            mix = audio_proc.add_noise_at_snr(mix, gauss, random.uniform(-6, 3))

        

        # ── HEAVY CHANNEL DEGRADATION ──

        # Low-pass filter (muffled through walls/distance)

        if random.random() < 0.60:

            mix = audio_proc.lowpass_filter(mix, random.randint(1000, 3000))

        

        # High-pass filter (wind noise removal artifact)

        if random.random() < 0.50:

            mix = audio_proc.highpass_filter(mix, random.randint(300, 1000))

        

        # Reverb (enclosed/urban environment)

        if random.random() < 0.40:

            mix = audio_proc.add_reverb(mix)

        

        # Clipping (overdriven field microphone)

        if random.random() < 0.35:

            mix = audio_proc.add_clipping(mix, random.uniform(0.15, 0.40))

        

        # Low bit depth (cheap recording equipment)

        if random.random() < 0.25:

            mix = audio_proc.reduce_bitdepth(mix, random.choice([4, 6, 8]))

        

        # Bandpass (radio channel transmission)

        if random.random() < 0.20:

            mix = audio_proc.bandpass_filter(

                mix, random.randint(400, 800), random.randint(2000, 4000)

            )

        

        mix = audio_proc.normalize(mix)

        

        # Save

        fname = f"stress_{i:06d}.wav"

        fpath = str(audio_dir / fname)

        sf.write(fpath, mix, cfg.SR)

        

        row = {"filepath": fpath, "hard_mode": 2}

        for cls in cfg.CLASSES:

            row[cls] = labels[cls]

        present = [c for c in cfg.CLASSES if labels[c] == 1]

        row["classes_present"] = ",".join(present)

        row["num_classes"] = len(present)

        rows.append(row)

    

    df = pd.DataFrame(rows)

    df.to_csv(output_dir / "metadata.csv", index=False)

    logger.info(f"Stress test: {len(df)} samples saved to {output_dir}")

    return df

# ── Generate stress test samples ──

stress_class_files = build_split_class_dict(split_df, "test")

stress_df = generate_stress_test(

    stress_class_files, noise_files,

    cfg.STRESS_TEST_DIR, num_samples=cfg.NUM_STRESS_TEST

)

print(f"\n✓ Stress test generated: {len(stress_df)} samples")

print(f"  Complexity distribution:")

for nc in sorted(stress_df["num_classes"].unique()):

    count = (stress_df["num_classes"] == nc).sum()

    print(f"    {nc} classes: {count} samples ({count/len(stress_df)*100:.0f}%)")

In [ ]:
# %%

# ===========================================================

# CELL 56: Evaluate best model on stress test

# ===========================================================

if best_model_name:

    # Load the best model

    best_model_path = Path(cfg.BEST_MODEL_DIR) / "model.keras"

    

    if best_model_path.exists():

        logger.info(f"Loading best model from: {best_model_path}")

        best_model = tf.keras.models.load_model(

            str(best_model_path),

            custom_objects={"loss_fn": focal_loss(2.0, 0.25)}

        )

    else:

        # Model might still be in memory from training

        logger.info("Using model from training session")

        # Rebuild from registry

        best_model = MODEL_REGISTRY[best_model_name]["builder"](INPUT_SHAPE, cfg.NUM_CLASSES)

        best_model = compile_model(best_model)

        # Load weights from saved checkpoint

        checkpoint_path = Path(cfg.MODELS_DIR) / best_model_name / "model.keras"

        if checkpoint_path.exists():

            best_model.load_weights(str(checkpoint_path))

    

    # Create stress test generator

    stress_gen = ProductionDataGenerator(

        stress_df, batch_size=cfg.BATCH_SIZE, augment=False,

        shuffle=False, mean=MEAN, std=STD

    )

    

    # Collect predictions

    y_stress_true_list = []

    y_stress_pred_list = []

    

    for i in tqdm(range(len(stress_gen)), desc="Stress test evaluation"):

        X_batch, y_batch = stress_gen[i]

        preds = best_model.predict(X_batch, verbose=0)

        y_stress_true_list.append(y_batch)

        y_stress_pred_list.append(preds)

    

    y_stress_true = (np.vstack(y_stress_true_list) >= 0.5).astype(int)

    y_stress_probs = np.vstack(y_stress_pred_list)

    y_stress_pred = (y_stress_probs >= cfg.THRESHOLD).astype(int)

    

    # Apply optimal thresholds if available

    if optimal_thresholds:

        y_stress_pred_opt = np.zeros_like(y_stress_pred)

        for i, cls in enumerate(cfg.CLASSES):

            t = optimal_thresholds[cls]["threshold"]

            y_stress_pred_opt[:, i] = (y_stress_probs[:, i] >= t).astype(int)

    else:

        y_stress_pred_opt = y_stress_pred

    

    # Compute metrics

    stress_macro_f1 = f1_score(y_stress_true, y_stress_pred_opt, average='macro', zero_division=0)

    stress_micro_f1 = f1_score(y_stress_true, y_stress_pred_opt, average='micro', zero_division=0)

    stress_exact = np.all(y_stress_pred_opt == y_stress_true, axis=1).mean()

    stress_hamming = hamming_loss(y_stress_true, y_stress_pred_opt)

    stress_precision = precision_score(y_stress_true, y_stress_pred_opt, average='macro', zero_division=0)

    stress_recall = recall_score(y_stress_true, y_stress_pred_opt, average='macro', zero_division=0)

    

    stress_aps = []

    for j in range(cfg.NUM_CLASSES):

        try:

            ap = average_precision_score(y_stress_true[:, j], y_stress_probs[:, j])

        except:

            ap = 0.0

        stress_aps.append(ap)

    stress_mAP = np.mean(stress_aps)

    

    stress_results = {

        "macro_f1": round(stress_macro_f1, 4),

        "micro_f1": round(stress_micro_f1, 4),

        "exact_match": round(stress_exact, 4),

        "hamming_loss": round(stress_hamming, 4),

        "macro_precision": round(stress_precision, 4),

        "macro_recall": round(stress_recall, 4),

        "mAP": round(stress_mAP, 4)

    }

    

    # Validate stress results

    stress_validation = validator.validate_test_metrics(

        y_stress_true, y_stress_pred_opt, y_stress_probs,

        f"{best_model_name}_STRESS"

    )

    validator.print_report(stress_validation)

    

    print(f"\n{'═'*60}")

    print(f"  STRESS TEST RESULTS — {best_model_name}")

    print(f"{'═'*60}")

    for k, v in stress_results.items():

        print(f"  {k:<20s}: {v:.4f}")

    

    # Save

    with open(f"{cfg.STRESS_TEST_DIR}/stress_results.json", 'w') as f:

        json.dump(stress_results, f, indent=2)

else:

    stress_results = {}

    print("⚠ No best model available for stress testing")

In [ ]:
# %%

# ===========================================================

# CELL 57: VIS 26 — Stress test vs normal test comparison

# ===========================================================

# SIGNIFICANCE:

# THE ultimate robustness chart. Shows how much performance

# drops under extreme conditions.

#

# INTERPRETATION:

# - Normal test: standard mixed clips

# - Stress test: worst-case scenario clips

# - Small gap = robust (good for deployment)

# - Large gap = fragile (needs more training on hard data)

# - Stress F1 of 0.40-0.60 is realistic and acceptable

# ===========================================================

if best_model_name and stress_results:

    normal_res = all_results[best_model_name]

    

    fig, axes = plt.subplots(1, 4, figsize=(22, 6))

    

    compare_metrics = [

        ("Macro F1", "macro_f1"),

        ("mAP", "mAP"),

        ("Exact Match", "exact_match"),

        ("Hamming Loss", "hamming_loss")

    ]

    

    for idx, (display_name, key) in enumerate(compare_metrics):

        normal_val = normal_res.get(key, 0)

        stress_val = stress_results.get(key, 0)

        

        bars = axes[idx].bar(

            ["Normal\nTest", "Stress\nTest"],

            [normal_val, stress_val],

            color=["#42A5F5", "#EF5350"],

            edgecolor='black', width=0.5

        )

        

        for bar, v in zip(bars, [normal_val, stress_val]):

            axes[idx].text(bar.get_x() + bar.get_width()/2,

                          bar.get_height() + 0.01,

                          f'{v:.4f}', ha='center', fontweight='bold', fontsize=12)

        

        # Show degradation

        if key == "hamming_loss":

            change = stress_val - normal_val

            direction = "↑" if change > 0 else "↓"

        else:

            change = normal_val - stress_val

            direction = "↓" if change > 0 else "↑"

        

        axes[idx].set_title(f"{display_name}\n({direction} {abs(change):.4f} degradation)",

                           fontsize=12, fontweight='bold')

        axes[idx].set_ylabel("Score")

        

        if key != "hamming_loss":

            axes[idx].set_ylim(0, 1.1)

    

    plt.suptitle(f"Robustness Assessment — {best_model_name}\n"

                 f"Normal Test vs Extreme Stress Test",

                 fontsize=16, fontweight='bold', y=1.05)

    plt.tight_layout()

    save_plot(fig, "evaluation", "stress_vs_normal.png")

    plt.show()

In [ ]:
# %%

# ===========================================================

# CELL 58: VIS 27 — Stress test per-class analysis

# ===========================================================

if best_model_name and stress_results:

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    

    # Per-class F1 comparison

    normal_f1s = []

    stress_f1s = []

    

    for i, cls in enumerate(cfg.CLASSES):

        # Normal test

        normal_preds = all_predictions[best_model_name]

        y_pred_n = normal_preds.get("y_pred_optimal", normal_preds["y_pred"])

        nf1 = f1_score(normal_preds["y_true"][:, i], y_pred_n[:, i], zero_division=0)

        normal_f1s.append(nf1)

        

        # Stress test

        sf1 = f1_score(y_stress_true[:, i], y_stress_pred_opt[:, i], zero_division=0)

        stress_f1s.append(sf1)

    

    x = np.arange(cfg.NUM_CLASSES)

    width = 0.35

    

    bars1 = axes[0].bar(x - width/2, normal_f1s, width, label='Normal Test',

                        color='#42A5F5', edgecolor='black')

    bars2 = axes[0].bar(x + width/2, stress_f1s, width, label='Stress Test',

                        color='#EF5350', edgecolor='black')

    

    for bar, v in zip(bars1, normal_f1s):

        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,

                    f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

    for bar, v in zip(bars2, stress_f1s):

        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,

                    f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

    

    axes[0].set_xticks(x)

    axes[0].set_xticklabels(cfg.CLASSES)

    axes[0].set_title("Per-Class F1: Normal vs Stress", fontsize=13, fontweight='bold')

    axes[0].set_ylabel("F1 Score")

    axes[0].set_ylim(0, 1.1)

    axes[0].legend()

    

    # Degradation percentage

    degradation = [(n - s) / max(n, 1e-8) * 100 for n, s in zip(normal_f1s, stress_f1s)]

    colors_deg = ['#66BB6A' if d < 30 else '#FF9800' if d < 50 else '#EF5350' for d in degradation]

    

    bars = axes[1].bar(cfg.CLASSES, degradation, color=colors_deg, edgecolor='black')

    for bar, v in zip(bars, degradation):

        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,

                    f'{v:.1f}%', ha='center', fontweight='bold', fontsize=11)

    

    axes[1].set_title("F1 Degradation Under Stress (%)\n"

                      "(Green <30% | Orange <50% | Red ≥50%)",

                      fontsize=12, fontweight='bold')

    axes[1].set_ylabel("Degradation (%)")

    

    plt.suptitle(f"Per-Class Stress Analysis — {best_model_name}",

                 fontsize=15, fontweight='bold', y=1.03)

    plt.tight_layout()

    save_plot(fig, "evaluation", "stress_per_class.png")

    plt.show()

In [ ]:
# %%

# ===========================================================

# CELL 59: Production inference class

# ===========================================================

# This is the FINAL inference module that will be used by

# the React frontend via the API.

#

# It loads the saved model, normalization stats, thresholds,

# and provides a clean predict() method.

#

# This class is also saved as a standalone Python file

# in the api/ folder for deployment.

# ===========================================================

class DefenceSoundClassifier:

    """

    Production-ready sound classification system.

    

    This class encapsulates the complete inference pipeline:

    1. Load audio file

    2. Extract mel spectrogram features

    3. Normalize using training statistics

    4. Run model prediction

    5. Apply per-class optimal thresholds

    6. Assign confidence levels

    7. Generate structured analysis report

    

    USAGE:

        classifier = DefenceSoundClassifier.from_saved_model("models/best_model")

        result = classifier.predict("path/to/audio.wav")

        print(result["summary"])

    """

    

    def init(self, model, config, normalization, thresholds, class_labels):

        """

        Initialize classifier with all required components.

        

        Args:

            model: Trained Keras model

            config: Dict with model configuration

            normalization: Dict with {"mean": float, "std": float}

            thresholds: Dict with per-class optimal thresholds

            class_labels: List of class names

        """

        self.model = model

        self.config = config

        self.norm = normalization

        self.thresholds = thresholds

        self.classes = class_labels

        self.num_classes = len(class_labels)

        self.audio_proc = AudioProcessor(cfg)

        self.confidence = ConfidenceAnalyzer(cfg)

        

        logger.info(f"Classifier initialized: {config.get('model_name', 'unknown')}")

        logger.info(f"Classes: {class_labels}")

    

    @classmethod

    def from_saved_model(cls, model_dir: str):

        """

        Load classifier from saved model directory.

        

        Expected files in model_dir:

        - model.keras

        - config.json

        - normalization_stats.json

        - optimal_thresholds.json

        - class_labels.json

        

        Args:

            model_dir: Path to saved model directory

        

        Returns:

            Initialized DefenceSoundClassifier instance

        """

        model_dir = Path(model_dir)

        

        # Load model

        model_path = model_dir / "model.keras"

        if not model_path.exists():

            raise FileNotFoundError(f"Model not found: {model_path}")

        

        model = tf.keras.models.load_model(

            str(model_path),

            custom_objects={"loss_fn": focal_loss(2.0, 0.25)}

        )

        logger.info(f"Model loaded from: {model_path}")

        

        # Load config

        with open(model_dir / "config.json", 'r') as f:

            config = json.load(f)

        

        # Load normalization stats

        with open(model_dir / "normalization_stats.json", 'r') as f:

            normalization = json.load(f)

        

        # Load thresholds

        thresh_path = model_dir / "optimal_thresholds.json"

        if thresh_path.exists():

            with open(thresh_path, 'r') as f:

                thresholds = json.load(f)

        else:

            thresholds = None

        

        # Load class labels

        labels_path = model_dir / "class_labels.json"

        if labels_path.exists():

            with open(labels_path, 'r') as f:

                labels_data = json.load(f)

                class_labels = labels_data["classes"]

        else:

            class_labels = cfg.CLASSES

        

        return cls(model, config, normalization, thresholds, class_labels)

    

    def predict(self, audio_path: str, return_raw: bool = False) -> dict:

        """

        Predict threat classes in an audio file.

        

        This is the main method called by the API/frontend.

        

        Args:

            audio_path: Path to audio file (.wav, .mp3, etc.)

            return_raw: If True, include raw probability array

        

        Returns:

            Dict with:

            - timestamp: ISO format timestamp

            - file: Input file path

            - overall_threat_level: CRITICAL/HIGH/MEDIUM/LOW/NEGLIGIBLE

            - detections: List of per-class analysis dicts

            - summary: Human-readable summary string

            - processing_time_ms: Inference latency

        """

        start_time = time.time()

        

        # Step 1: Extract features

        mel = self.audio_proc.extract_mel_spectrogram(audio_path, augment=False)

        

        # Step 2: Normalize

        mel = (mel - self.norm["mean"]) / (self.norm["std"] + 1e-8)

        

        # Step 3: Prepare for model

        mel = mel[..., np.newaxis]  # Add channel dimension

        mel = np.expand_dims(mel, axis=0)  # Add batch dimension

        

        # Step 4: Predict

        probs = self.model.predict(mel, verbose=0)[0]

        

        # Step 5: Apply thresholds and analyze

        analysis = self.confidence.analyze_prediction(

            probs, self.classes,

            per_class_thresholds=self.thresholds

        )

        

        # Step 6: Add metadata

        processing_time = (time.time() - start_time) * 1000

        

        result = {

            "timestamp": datetime.now().isoformat(),

            "file": str(audio_path),

            "file_name": Path(audio_path).name,

            "overall_threat_level": analysis["overall_threat_level"],

            "overall_color": self.confidence.get_confidence_color(

                analysis["overall_threat_level"]

            ),

            "detections": analysis["detections"],

            "summary": analysis["summary"],

            "processing_time_ms": round(processing_time, 2),

            "model_name": self.config.get("model_name", "unknown"),

        }

        

        if return_raw:

            result["raw_probabilities"] = probs.tolist()

        

        return result

    

    def predict_batch(self, audio_paths: List[str]) -> List[dict]:

        """

        Predict on multiple audio files.

        

        Args:

            audio_paths: List of file paths

        

        Returns:

            List of prediction dicts (same format as predict())

        """

        results = []

        for path in tqdm(audio_paths, desc="Batch prediction"):

            result = self.predict(path)

            results.append(result)

        return results

    

    def print_prediction(self, result: dict):

        """

        Pretty-print a prediction result to console.

        

        This format would be displayed on a defence terminal.

        """

        threat_emoji = {

            "CRITICAL": "🔴", "HIGH": "🟠", "MEDIUM": "🟡",

            "LOW": "🟢", "NEGLIGIBLE": "⚪"

        }

        

        level = result["overall_threat_level"]

        emoji = threat_emoji.get(level, "❓")

        

        print(f"\n{'━'*60}")

        print(f"  {emoji} THREAT ASSESSMENT — {result['file_name']}")

        print(f"{'━'*60}")

        print(f"  Time:    {result['timestamp']}")

        print(f"  Model:   {result['model_name']}")

        print(f"  Latency: {result['processing_time_ms']:.1f} ms")

        print(f"{'━'*60}")

        

        for det in result["detections"]:

            prob = det["probability"]

            bar_len = int(prob * 30)

            bar = "█"  bar_len + "░"  (30 - bar_len)

            

            status = "✓ DETECTED" if det["detected"] else "  ─"

            level_tag = f"[{det['confidence_level']}]" if det["detected"] else ""

            

            print(f"  {det['class']:>12s}: {prob:.4f} [{bar}] {status} {level_tag}")

        

        print(f"{'━'*60}")

        print(f"  {result['summary']}")

        

        for det in result["detections"]:

            if det["detected"]:

                print(f"  → {det['action']}")

        

        print(f"{'━'*60}")

# ── Initialize production classifier ──

if best_model_name:

    try:

        classifier = DefenceSoundClassifier.from_saved_model(cfg.BEST_MODEL_DIR)

        print("✓ Production classifier loaded successfully")

    except Exception as e:

        logger.warning(f"Could not load saved model: {e}")

        logger.info("Creating classifier from training session model...")

        

        # Fallback: use model from memory

        classifier = DefenceSoundClassifier(

            model=best_model,

            config={"model_name": best_model_name},

            normalization={"mean": MEAN, "std": STD},

            thresholds=optimal_thresholds if optimal_thresholds else None,

            class_labels=cfg.CLASSES

        )

        print("✓ Production classifier created from training session")

else:

    classifier = None

    print("⚠ No best model available for classifier initialization")

In [ ]:
# %%

# ===========================================================

# CELL 60: Demo predictions on stress test samples

# ===========================================================

# Shows the classifier in action on challenging samples.

# This is what the React frontend would display.

# ===========================================================

if classifier and len(stress_df) > 0:

    print("=" * 60)

    print("  DEMO: Production Predictions on Stress Test Samples")

    print("=" * 60)

    

    # Pick diverse samples

    demo_indices = []

    for nc in sorted(stress_df["num_classes"].unique()):

        subset = stress_df[stress_df["num_classes"] == nc]

        if len(subset) > 0:

            demo_indices.append(subset.sample(1, random_state=cfg.SEED).index[0])

    

    # Add a few random ones

    extra = stress_df.sample(min(3, len(stress_df)), random_state=cfg.SEED + 1).index.tolist()

    demo_indices.extend([i for i in extra if i not in demo_indices])

    

    for idx in demo_indices[:8]:

        row = stress_df.iloc[idx]

        print(f"\n  Ground Truth: {row['classes_present']} ({row['num_classes']} classes)")

        

        result = classifier.predict(row["filepath"], return_raw=True)

        classifier.print_prediction(result)

In [ ]:
# %%

# ===========================================================

# CELL 61: Save standalone inference module

# ===========================================================

# Creates a standalone Python file that can be imported

# by the Flask/FastAPI backend for the React frontend.

#

# This file contains everything needed for inference

# WITHOUT needing the training notebook.

# ===========================================================

inference_module_code = '''"""

Defence Sound Classification — Production Inference Module

This module provides the DefenceSoundClassifier class for

production deployment. Import and use as:

    from inference import DefenceSoundClassifier

    classifier = DefenceSoundClassifier.from_saved_model("models/best_model")

    result = classifier.predict("audio.wav")

Requirements:

    pip install tensorflow librosa numpy soundfile

"""

import os

import json

import time

import numpy as np

import librosa

import soundfile as sf

import tensorflow as tf

from pathlib import Path

from datetime import datetime

# Suppress TF warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

CONFIDENCE_LEVELS = {

    "CRITICAL": 0.90,

    "HIGH": 0.75,

    "MEDIUM": 0.50,

    "LOW": 0.30,

    "NEGLIGIBLE": 0.0

}

CONFIDENCE_COLORS = {

    "CRITICAL": "#D32F2F",

    "HIGH": "#F44336",

    "MEDIUM": "#FF9800",

    "LOW": "#FFC107",

    "NEGLIGIBLE": "#4CAF50"

}

ACTIONS = {

    "CRITICAL": "IMMEDIATE: Deploy countermeasures",

    "HIGH": "ALERT: Verify with secondary sensor",

    "MEDIUM": "FLAG: Assign operator for review",

    "LOW": "WATCH: Continue monitoring",

    "NEGLIGIBLE": "CLEAR: No action needed"

}

SR = 22050

DURATION = 6

SAMPLES = SR * DURATION

N_MELS = 128

N_FFT = 2048

HOP_LENGTH = 512

def get_confidence_level(prob):

    for level, thresh in sorted(CONFIDENCE_LEVELS.items(), key=lambda x: -x[1]):

        if prob >= thresh:

            return level

    return "NEGLIGIBLE"

class DefenceSoundClassifier:

    def init(self, model, config, normalization, thresholds, classes):

        self.model = model

        self.config = config

        self.norm = normalization

        self.thresholds = thresholds

        self.classes = classes

    @classmethod

    def from_saved_model(cls, model_dir):

        model_dir = Path(model_dir)

        

        model = tf.keras.models.load_model(str(model_dir / "model.keras"))

        

        with open(model_dir / "config.json") as f:

            config = json.load(f)

        with open(model_dir / "normalization_stats.json") as f:

            normalization = json.load(f)

        

        thresholds = None

        if (model_dir / "optimal_thresholds.json").exists():

            with open(model_dir / "optimal_thresholds.json") as f:

                thresholds = json.load(f)

        

        classes = config.get("classes", ["drone", "gunshot", "jetplane", "vehicle"])

        

        return cls(model, config, normalization, thresholds, classes)

    def predict(self, audio_path, return_raw=False):

        start = time.time()

        

        # Load and process audio

        y,  = librosa.load(audiopath, sr=SR, mono=True)

        

        # Pad or crop

        if len(y) < SAMPLES:

            y = np.pad(y, (0, SAMPLES - len(y)))

        else:

            y = y[:SAMPLES]

        

        # Extract features

        mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS,

                                              n_fft=N_FFT, hop_length=HOP_LENGTH)

        mel_db = librosa.power_to_db(mel, ref=np.max).astype(np.float32)

        

        # Normalize

        mel_db = (mel_db - self.norm["mean"]) / (self.norm["std"] + 1e-8)

        

        # Predict

        inp = mel_db[np.newaxis, ..., np.newaxis]

        probs = self.model.predict(inp, verbose=0)[0]

        

        # Analyze

        detections = []

        max_level = "NEGLIGIBLE"

        level_priority = {"NEGLIGIBLE": 0, "LOW": 1, "MEDIUM": 2, "HIGH": 3, "CRITICAL": 4}

        

        for i, cls in enumerate(self.classes):

            prob = float(probs[i])

            t = self.thresholds[cls]["threshold"] if self.thresholds else 0.5

            detected = prob >= t

            level = get_confidence_level(prob)

            

            det = {

                "class": cls,

                "probability": round(prob, 4),

                "threshold": t,

                "detected": detected,

                "confidence_level": level,

                "color": CONFIDENCE_COLORS.get(level, "#9E9E9E"),

                "action": ACTIONS.get(level, "")

            }

            detections.append(det)

            

            if detected and level_priority.get(level, 0) > level_priority.get(max_level, 0):

                max_level = level

        

        detected_classes = [d["class"] for d in detections if d["detected"]]

        summary = f"THREATS: {', '.join(detected_classes)} | Level: {max_level}" if detected_classes else "ALL CLEAR"

        

        result = {

            "timestamp": datetime.now().isoformat(),

            "file": str(audio_path),

            "overall_threat_level": max_level,

            "overall_color": CONFIDENCE_COLORS.get(max_level, "#9E9E9E"),

            "detections": detections,

            "summary": summary,

            "processing_time_ms": round((time.time() - start) * 1000, 2)

        }

        

        if return_raw:

            result["raw_probabilities"] = probs.tolist()

        

        return result

'''

# Save inference module

inference_path = Path(cfg.PROJECT_ROOT) / "api" / "inference.py"

inference_path.parent.mkdir(parents=True, exist_ok=True)

with open(inference_path, 'w') as f:

    f.write(inference_module_code)

print(f"✓ Inference module saved: {inference_path}")

In [ ]:
# %%

# ===========================================================

# CELL 62: Save Flask API server

# ===========================================================

# Creates a minimal Flask API that the React frontend can call.

# Endpoints:

#   POST /predict — Upload audio file, get prediction

#   GET /health — Health check

#   GET /config — Get model configuration

# ===========================================================

api_code = '''"""

Defence Sound Classification — REST API Server

Run:

    cd defence_sound_classification/api

    python app.py

Endpoints:

    POST /predict     — Upload audio file, get prediction JSON

    GET  /health      — Health check

    GET  /config      — Model configuration

    GET  /classes     — List of detectable classes

Requires:

    pip install flask flask-cors

"""

import os

import sys

import json

import tempfile

from pathlib import Path

from flask import Flask, request, jsonify

from flask_cors import CORS

# Add parent directory to path so we can import inference module

sys.path.insert(0, str(Path(__file__).parent))

from inference import DefenceSoundClassifier

app = Flask(__name__)

CORS(app)  # Allow React frontend to call this API

# Load classifier once at startup

MODEL_DIR = str(Path(__file__).parent.parent / "models" / "best_model")

classifier = None

try:

    classifier = DefenceSoundClassifier.from_saved_model(MODEL_DIR)

    print(f"Model loaded from: {MODEL_DIR}")

except Exception as e:

    print(f"ERROR loading model: {e}")

    print(f"Looked in: {MODEL_DIR}")

@app.route("/health", methods=["GET"])

def health():

    """Health check endpoint."""

    return jsonify({

        "status": "ok" if classifier else "no_model",

        "model_loaded": classifier is not None

    })

@app.route("/config", methods=["GET"])

def get_config():

    """Return model configuration."""

    if not classifier:

        return jsonify({"error": "Model not loaded"}), 503

    return jsonify(classifier.config)

@app.route("/classes", methods=["GET"])

def get_classes():

    """Return list of detectable classes."""

    if not classifier:

        return jsonify({"error": "Model not loaded"}), 503

    return jsonify({

        "classes": classifier.classes,

        "num_classes": len(classifier.classes)

    })

@app.route("/predict", methods=["POST"])

def predict():

    """

    Predict threat classes in uploaded audio file.

    

    Accepts: multipart/form-data with 'file' field

    Returns: JSON prediction result

    """

    if not classifier:

        return jsonify({"error": "Model not loaded"}), 503

    

    if "file" not in request.files:

        return jsonify({"error": "No file uploaded. Use 'file' field."}), 400

    

    file = request.files["file"]

    if file.filename == "":

        return jsonify({"error": "Empty filename"}), 400

    

    # Save to temp file

    suffix = Path(file.filename).suffix or ".wav"

    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:

        file.save(tmp.name)

        tmp_path = tmp.name

    

    try:

        result = classifier.predict(tmp_path, return_raw=True)

        return jsonify(result)

    except Exception as e:

        return jsonify({"error": str(e)}), 500

    finally:

        os.unlink(tmp_path)

if name == "__main__":

    print("Starting Defence Sound Classification API...")

    print("Endpoints:")

    print("  GET  /health")

    print("  GET  /config")

    print("  GET  /classes")

    print("  POST /predict")

    app.run(host="0.0.0.0", port=5000, debug=False)

'''

api_path = Path(cfg.PROJECT_ROOT) / "api" / "app.py"

api_path.parent.mkdir(parents=True, exist_ok=True)

with open(api_path, 'w') as f:

    f.write(api_code)

# Save API requirements

api_reqs = "flask\nflask-cors\ntensorflow\nlibrosa\nnumpy\nsoundfile\n"

with open(Path(cfg.PROJECT_ROOT) / "api" / "requirements_api.txt", 'w') as f:

    f.write(api_reqs)

print(f"✓ API server saved: {api_path}")

print(f"✓ API requirements saved: {api_path.parent / 'requirements_api.txt'}")

# %%

# ===========================================================

# CELL 63: Save project requirements.txt

# ===========================================================

requirements = """# Defence Sound Classification System — Requirements

# Install: pip install -r requirements.txt

# Core ML

tensorflow>=2.12.0

numpy>=1.23.0

pandas>=1.5.0

scikit-learn>=1.2.0

# Audio

librosa>=0.10.0

soundfile>=0.12.0

pydub>=0.25.0

scipy>=1.10.0

# Visualization

matplotlib>=3.7.0

seaborn>=0.12.0

# Progress

tqdm>=4.65.0

# API (optional — for React frontend)

flask>=2.3.0

flask-cors>=4.0.0

"""

req_path = Path(cfg.PROJECT_ROOT) / "requirements.txt"

with open(req_path, 'w') as f:

    f.write(requirements)

print(f"✓ Requirements saved: {req_path}")

In [ ]:
# %%

# ===========================================================

# CELL 64: Generate README.md

# ===========================================================

readme_content = f"""# Defence Sound Classification System

## Overview

Multi-label audio classification system for detecting defence-related sounds:

- Drone — UAV/quadcopter rotor sounds

- Gunshot — Firearm discharge sounds

- Jetplane — Jet aircraft engine sounds

- Vehicle — Ground vehicle engine sounds

## Architecture

Best model: {best_model_name if best_model_name else 'TBD'}

- CNN feature extraction with attention mechanisms

- Bidirectional GRU for temporal modeling

- Multi-label sigmoid output

- Per-class optimized thresholds

- Confidence level system (CRITICAL/HIGH/MEDIUM/LOW)

## Results

| Metric | Score |

|--------|-------|

| Macro F1 | {all_results.get(best_model_name, {}).get('macro_f1', 'N/A')} |

| mAP | {all_results.get(best_model_name, {}).get('mAP', 'N/A')} |

| Exact Match | {all_results.get(best_model_name, {}).get('exact_match', 'N/A')} |

| Hamming Loss | {all_results.get(best_model_name, {}).get('hamming_loss', 'N/A')} |

| Stress Test F1 | {stress_results.get('macro_f1', 'N/A')} |

## Quick Start

```bash

# Install dependencies

pip install -r requirements.txt

# Run API server

cd api

python app.py

# API will be available at http://localhost:5000

```

## API Usage

```bash

# Health check

curl http://localhost:5000/health

# Predict

curl -X POST -F "file=@audio.wav" http://localhost:5000/predict

```

## Project Structure

```

defence_sound_classification/

├── data/           # Raw, processed, generated audio

├── models/         # Trained models

├── reports/        # Plots and evaluation reports

├── notebooks/      # Training notebook

├── api/            # Flask API for React frontend

└── requirements.txt

```

## Models Compared

{chr(10).join(f'- {name}: {info["description"]}' for name, info in MODEL_REGISTRY.items())}

## Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

"""

readme_path = Path(cfg.PROJECT_ROOT) / "README.md"

with open(readme_path, 'w') as f:

    f.write(readme_content)

print(f"✓ README saved: {readme_path}")

In [ ]:
# %%

# ===========================================================

# CELL 65: FINAL COMPREHENSIVE REPORT

# ===========================================================

report_text = []

report_text.append("=" * 80)

report_text.append("  DEFENCE SOUND CLASSIFICATION SYSTEM — FINAL REPORT")

report_text.append(f"  Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

report_text.append("=" * 80)

report_text.append(f"\n1. SYSTEM CONFIGURATION")

report_text.append(f"   Classes:         {', '.join(cfg.CLASSES)}")

report_text.append(f"   Audio:           {cfg.SR}Hz, {cfg.DURATION}s clips")

report_text.append(f"   Features:        {cfg.N_MELS}-band log-mel spectrogram")

report_text.append(f"   Loss:            Focal Loss (γ=2.0, α=0.25)")

report_text.append(f"   Regularization:  L2={cfg.L2_REG}, Dropout, Label Smoothing={cfg.LABEL_SMOOTHING}, Mixup={cfg.MIXUP_ALPHA}")

report_text.append(f"\n2. DATASET")

report_text.append(f"   Train: {len(train_df):,} samples")

report_text.append(f"   Val:   {len(val_df):,} samples")

report_text.append(f"   Test:  {len(test_df):,} samples")

report_text.append(f"   Stress:{len(stress_df):,} samples")

report_text.append(f"   Difficulty: gain=[{cfg.MIN_GAIN},{cfg.MAX_GAIN}], SNR=[{cfg.MIN_SNR},{cfg.MAX_SNR}]dB")

report_text.append(f"\n3. MODELS TRAINED: {len(trained_models)}")

report_text.append(f"   {'Model':<22s} {'Attention':<10s} {'F1':>7s} {'mAP':>7s} {'Exact':>7s} {'Hamming':>8s} {'Params':>10s} {'Speed':>8s}")

report_text.append(f"   {'─'*82}")

for name in trained_models:

    res = all_results[name]

    attn = "Yes" if res.get("has_attention") else "No"

    marker = " ★" if name == best_model_name else ""

    report_text.append(

        f"   {name:<22s} {attn:<10s} {res.get('macro_f1',0):>7.4f} "

        f"{res.get('mAP',0):>7.4f} {res.get('exact_match',0):>7.4f} "

        f"{res.get('hamming_loss',0):>8.4f} {res.get('params',0):>10,} "

        f"{res.get('inference_ms',0):>7.1f}ms{marker}"

    )

if best_model_name:

    report_text.append(f"\n4. BEST MODEL: {best_model_name}")

    res = all_results[best_model_name]

    report_text.append(f"   Macro F1:       {res.get('macro_f1', 0):.4f}")

    report_text.append(f"   Optimized F1:   {res.get('macro_f1_optimal', res.get('macro_f1', 0)):.4f}")

    report_text.append(f"   mAP:            {res.get('mAP', 0):.4f}")

    report_text.append(f"   Exact Match:    {res.get('exact_match', 0):.4f}")

    report_text.append(f"   Hamming Loss:   {res.get('hamming_loss', 0):.4f}")

    report_text.append(f"   Precision:      {res.get('macro_precision', 0):.4f}")

    report_text.append(f"   Recall:         {res.get('macro_recall', 0):.4f}")

    report_text.append(f"   Inference:      {res.get('inference_ms', 0):.1f} ms/sample")

    report_text.append(f"   Attention:      {'Yes' if res.get('has_attention') else 'No'}")

    if optimal_thresholds:

        report_text.append(f"\n5. OPTIMAL THRESHOLDS")

        for cls, vals in optimal_thresholds.items():

            report_text.append(f"   {cls:<12s}: t={vals['threshold']:.2f} (F1={vals['f1']:.4f})")

if stress_results:

    report_text.append(f"\n6. STRESS TEST RESULTS")

    for k, v in stress_results.items():

        report_text.append(f"   {k:<20s}: {v:.4f}")

# Attention analysis

no_attn = [all_results[n]["macro_f1"] for n in trained_models

           if not all_results[n].get("has_attention") and all_results[n].get("category") in ["crnn"]]

with_attn = [all_results[n]["macro_f1"] for n in trained_models

             if all_results[n].get("has_attention")]

if no_attn and with_attn:

    report_text.append(f"\n7. ATTENTION ANALYSIS")

    report_text.append(f"   CRNN without attention — avg F1: {np.mean(no_attn):.4f}")

    report_text.append(f"   CRNN with attention    — avg F1: {np.mean(with_attn):.4f}")

    improvement = np.mean(with_attn) - np.mean(no_attn)

    report_text.append(f"   Improvement:                     +{improvement:.4f} ({improvement/np.mean(no_attn)*100:.1f}%)")

# Validation summary

pass_count = sum(1 for r in all_validation_reports if r["status"] == "PASS")

warn_count = sum(1 for r in all_validation_reports if r["status"] == "WARNING")

fail_count = sum(1 for r in all_validation_reports if r["status"] == "FAIL")

report_text.append(f"\n8. VALIDATION SUMMARY")

report_text.append(f"   Total checks:  {len(all_validation_reports)}")

report_text.append(f"   Passed:        {pass_count}")

report_text.append(f"   Warnings:      {warn_count}")

report_text.append(f"   Failed:        {fail_count}")

# Saved artifacts

all_plots = list(Path(cfg.PLOTS_DIR).rglob("*.png"))

report_text.append(f"\n9. SAVED ARTIFACTS")

report_text.append(f"   Model:          {cfg.BEST_MODEL_DIR}/model.keras")

report_text.append(f"   Config:         {cfg.BEST_MODEL_DIR}/config.json")

report_text.append(f"   Normalization:  {cfg.BEST_MODEL_DIR}/normalization_stats.json")

report_text.append(f"   Thresholds:     {cfg.BEST_MODEL_DIR}/optimal_thresholds.json")

report_text.append(f"   API Server:     {cfg.PROJECT_ROOT}/api/app.py")

report_text.append(f"   Inference:      {cfg.PROJECT_ROOT}/api/inference.py")

report_text.append(f"   Plots:          {len(all_plots)} visualizations")

report_text.append(f"   Report:         {cfg.REPORTS_DIR}/final_summary.txt")

report_text.append(f"\n10. DEPLOYMENT INSTRUCTIONS")

report_text.append(f"   1. Install: pip install -r requirements.txt")

report_text.append(f"   2. Start API: cd api && python app.py")

report_text.append(f"   3. Test: curl -X POST -F 'file=@test.wav' http://localhost:5000/predict")

report_text.append(f"   4. Connect React frontend to http://localhost:5000")

report_text.append(f"\n{'═'*80}")

report_text.append(f"  REPORT COMPLETE")

report_text.append(f"{'═'*80}")

# Save report

report_full = "\n".join(report_text)

report_path = Path(cfg.REPORTS_DIR) / "final_summary.txt"

with open(report_path, 'w') as f:

    f.write(report_full)

# Print report

print(report_full)

In [ ]:
# %%

# ===========================================================

# CELL 66: Final file listing

# ===========================================================

# Show everything that was created during this notebook run.

# This serves as the inventory for the project deliverable.

# ===========================================================

print("\n" + "═" * 70)

print("  PROJECT FILE INVENTORY")

print("═" * 70)

def count_files(directory, pattern="*"):

    d = Path(directory)

    if d.exists():

        return len(list(d.rglob(pattern)))

    return 0

sections = {

    "Data — Raw": (cfg.RAW_DATASET, "*.wav"),

    "Data — Processed": (cfg.PROCESSED_DATASET, "*.wav"),

    "Data — Generated Train": (f"{cfg.GENERATED_ROOT}/train", "*.*"),

    "Data — Generated Val": (f"{cfg.GENERATED_ROOT}/val", "*.*"),

    "Data — Generated Test": (f"{cfg.GENERATED_ROOT}/test", "*.*"),

    "Data — Stress Test": (cfg.STRESS_TEST_DIR, "*.*"),

    "Models — All": (cfg.MODELS_DIR, "*.keras"),

    "Models — Best": (cfg.BEST_MODEL_DIR, "*.*"),

    "Reports — Plots": (cfg.PLOTS_DIR, "*.png"),

    "Reports — CSV/TXT": (cfg.REPORTS_DIR, "*.csv"),

    "API": (f"{cfg.PROJECT_ROOT}/api", "*.py"),

}

total_files = 0

for section_name, (directory, pattern) in sections.items():

    count = count_files(directory, pattern)

    total_files += count

    status = "✓" if count > 0 else "⚠"

    print(f"  {status} {section_name:<30s}: {count:>5} files  ({directory})")

print(f"\n  Total project files: {total_files:,}")

# Key files

print(f"\n  KEY FILES:")

key_files = [

    f"{cfg.BEST_MODEL_DIR}/model.keras",

    f"{cfg.BEST_MODEL_DIR}/config.json",

    f"{cfg.BEST_MODEL_DIR}/normalization_stats.json",

    f"{cfg.BEST_MODEL_DIR}/optimal_thresholds.json",

    f"{cfg.BEST_MODEL_DIR}/class_labels.json",

    f"{cfg.PROJECT_ROOT}/api/app.py",

    f"{cfg.PROJECT_ROOT}/api/inference.py",

    f"{cfg.PROJECT_ROOT}/config.json",

    f"{cfg.PROJECT_ROOT}/requirements.txt",

    f"{cfg.PROJECT_ROOT}/README.md",

    f"{cfg.REPORTS_DIR}/final_summary.txt",

    f"{cfg.REPORTS_DIR}/model_comparison_report.csv",

    f"{cfg.REPORTS_DIR}/validation_report.txt",

]

for kf in key_files:

    exists = "✓" if Path(kf).exists() else "✗"

    print(f"    {exists} {kf}")

In [ ]:
# %%

# ===========================================================

# CELL 67: Final system verification

# ===========================================================

# Run a complete end-to-end verification that everything works.

# This is the "smoke test" before deployment.

# ===========================================================

print("\n" + "═" * 70)

print("  SYSTEM VERIFICATION")

print("═" * 70)

checks_passed = 0

checks_total = 0

def verify(description, condition):

    global checks_passed, checks_total

    checks_total += 1

    status = "✅ PASS" if condition else "❌ FAIL"

    if condition:

        checks_passed += 1

    print(f"  {status}: {description}")

# Data checks

verify("Raw dataset exists", Path(cfg.RAW_DATASET).exists())

verify("Processed dataset exists", Path(cfg.PROCESSED_DATASET).exists())

verify("Split CSV exists", Path(cfg.SPLIT_CSV).exists())

verify("Train metadata exists", Path(f"{cfg.GENERATED_ROOT}/train/metadata.csv").exists())

verify("Val metadata exists", Path(f"{cfg.GENERATED_ROOT}/val/metadata.csv").exists())

verify("Test metadata exists", Path(f"{cfg.GENERATED_ROOT}/test/metadata.csv").exists())

verify("Stress test exists", Path(f"{cfg.STRESS_TEST_DIR}/metadata.csv").exists())

# Model checks

verify("Best model saved", Path(f"{cfg.BEST_MODEL_DIR}/model.keras").exists())

verify("Config saved", Path(f"{cfg.BEST_MODEL_DIR}/config.json").exists())

verify("Normalization saved", Path(f"{cfg.BEST_MODEL_DIR}/normalization_stats.json").exists())

verify("Thresholds saved", Path(f"{cfg.BEST_MODEL_DIR}/optimal_thresholds.json").exists())

verify("Class labels saved", Path(f"{cfg.BEST_MODEL_DIR}/class_labels.json").exists())

# Results checks

verify("Comparison report saved", Path(f"{cfg.REPORTS_DIR}/model_comparison_report.csv").exists())

verify("Validation report saved", Path(f"{cfg.REPORTS_DIR}/validation_report.txt").exists())

verify("Final summary saved", Path(f"{cfg.REPORTS_DIR}/final_summary.txt").exists())

verify(f"Plots generated (>20)", len(list(Path(cfg.PLOTS_DIR).rglob("*.png"))) >= 20)

# API checks

verify("API server saved", Path(f"{cfg.PROJECT_ROOT}/api/app.py").exists())

verify("Inference module saved", Path(f"{cfg.PROJECT_ROOT}/api/inference.py").exists())

verify("Requirements saved", Path(f"{cfg.PROJECT_ROOT}/requirements.txt").exists())

verify("README saved", Path(f"{cfg.PROJECT_ROOT}/README.md").exists())

# Metric checks

if best_model_name:

    res = all_results[best_model_name]

    verify(f"Best model F1 > 0.50", res.get("macro_f1", 0) > 0.50)

    verify(f"Best model F1 < 0.98 (not suspicious)", res.get("macro_f1", 0) < 0.98)

    verify(f"Best model mAP > 0.50", res.get("mAP", 0) > 0.50)

    verify(f"Hamming loss < 0.30", res.get("hamming_loss", 1) < 0.30)

if stress_results:

    verify(f"Stress test F1 > 0.30", stress_results.get("macro_f1", 0) > 0.30)

    verify(f"Stress test F1 < best test F1", 

           stress_results.get("macro_f1", 1) <= all_results.get(best_model_name, {}).get("macro_f1", 0) + 0.01)

# Inference check

if classifier:

    test_files = list(Path(cfg.STRESS_TEST_DIR).rglob("*.wav"))

    if test_files:

        try:

            test_result = classifier.predict(str(test_files[0]))

            verify("Inference works end-to-end", "detections" in test_result)

            verify(f"Inference < 1000ms", test_result.get("processing_time_ms", 9999) < 1000)

        except Exception as e:

            verify(f"Inference works: {e}", False)

print(f"\n  {'─'*50}")

print(f"  VERIFICATION: {checks_passed}/{checks_total} checks passed")

if checks_passed == checks_total:

    print(f"  🎉 ALL CHECKS PASSED — System ready for deployment!")

elif checks_passed >= checks_total * 0.8:

    print(f"  ⚠️  Most checks passed. Review failures before deployment.")

else:

    print(f"  ❌ Multiple failures. DO NOT deploy until resolved.")

In [ ]:
# %%

# ===========================================================

# CELL 68: FINAL MESSAGE

# ===========================================================

print(f"""

╔══════════════════════════════════════════════════════════════════════╗

║                                                                      ║

║   🛡️  DEFENCE SOUND CLASSIFICATION SYSTEM — COMPLETE  🛡️            ║

║                                                                      ║

║   Best Model:    {best_model_name if best_model_name else 'N/A':<48s} ║

║   Macro F1:      {all_results.get(best_model_name, {}).get('macro_f1', 'N/A')!s:<48s} ║

║   Stress F1:     {stress_results.get('macro_f1', 'N/A')!s:<48s} ║

║   Models Tested: {len(trained_models):<48d} ║

║   Plots:         {len(list(Path(cfg.PLOTS_DIR).rglob('*.png'))):<48d} ║

║   Verification:  {checks_passed}/{checks_total} passed{' ' * 39} ║

║                                                                      ║

║   TO START THE API:                                                  ║

║   cd {str(Path(cfg.PROJECT_ROOT) / 'api'):<60s}  ║

║   python app.py                                                      ║

║                                                                      ║

║   Then connect your React frontend to http://localhost:5000          ║

║                                                                      ║

╚══════════════════════════════════════════════════════════════════════╝

""")

In [ ]:


# To connect React frontend

# Your API runs at http://localhost:5000. The React frontend should call:

# ```javascript

# // React component example

# const predictAudio = async (audioFile) => {

#   const formData = new FormData();

#   formData.append('file', audioFile);

  

#   const response = await fetch('http://localhost:5000/predict', {

#     method: 'POST',

#     body: formData

#   });

  

#   const result = await response.json();

#   // result contains: overall_threat_level, detections[], summary, etc.

#   return result;

# };

# ```

# The API returns JSON with:

# - overall_threat_level: CRITICAL / HIGH / MEDIUM / LOW / NEGLIGIBLE

# - detections[]: Per-class probability, threshold, confidence level, action

# - summary: Human-readable text

# - processing_time_ms: Inference latency